# Part 1 Risk Scoring

### Pipeline to parse all drugs under ATC group N05A (Antipsychotics)

In [1]:
"""
Pipeline to parse all drugs under ATC group N05A (Antipsychotics)
from the WHO ATC/DDD Index (https://atcddd.fhi.no/atc_ddd_index/).

Strategy:
    1. Fetch the N05A index page.
    2. Extract the child subgroup codes
       (N05AA, N05AB, N05AC, N05AD, N05AE, N05AF, N05AG,
        N05AH, N05AL, N05AN, N05AX).
    3. Visit each subgroup page and parse the drug-level rows
       (ATC code + name, and where available DDD / unit / route / note).
    4. Return a flat list of drug dicts.
    5. Save the results to a CSV file.

Notes:
    - N05AN covers lithium (used in bipolar disorder).
    - Reserpine is NOT in N05A; it is classified under C02
      (antihypertensives), so it will not appear in these results.
    - Some atypicals (e.g. aripiprazole, risperidone, lurasidone,
      ziprasidone) live in N05AX ("Other antipsychotics").
"""

import csv
import time
import requests
from bs4 import BeautifulSoup

BASE_URL = "https://atcddd.fhi.no/atc_ddd_index/"
HEADERS = {"User-Agent": "Mozilla/5.0 (ATC-DDD-parser; research use)"}

# Column order used for the CSV output
CSV_FIELDS = ["atc_code", "name", "ddd", "unit", "route", "note", "subgroup"]


# ----------------------------------------------------------------------
# Low-level fetch
# ----------------------------------------------------------------------
def fetch(code: str, session: requests.Session) -> BeautifulSoup:
    """Fetch a single ATC code page and return parsed soup."""
    resp = session.get(BASE_URL, params={"code": code}, headers=HEADERS, timeout=30)
    resp.raise_for_status()
    return BeautifulSoup(resp.text, "html.parser")


# ----------------------------------------------------------------------
# Discover subgroup codes under a parent code
# ----------------------------------------------------------------------
def get_child_codes(soup: BeautifulSoup, parent_code: str) -> list[str]:
    """
    Extract immediate child ATC codes from a page.
    Children appear as links of the form '?code=N05AA'.
    """
    codes = []
    for a in soup.select("a[href*='code=']"):
        href = a["href"]
        code = href.split("code=")[-1].split("&")[0].strip()
        # keep codes that extend the parent (e.g. N05AA under N05A)
        if (
            code.startswith(parent_code)
            and code != parent_code
            and len(code) == len(parent_code) + 1   # one level deeper
            and code not in codes
        ):
            codes.append(code)
    return codes


# ----------------------------------------------------------------------
# Parse drug-level rows from a subgroup page
# ----------------------------------------------------------------------
def parse_drugs(soup: BeautifulSoup, subgroup_code: str) -> list[dict]:
    """
    On a subgroup (chemical subgroup) page the substances are listed in a
    table whose rows look like:
        ATC code | Name | DDD | Unit | Adm.R | Note
    The 7-character codes (e.g. N05AB03) identify chemical substances.
    """
    drugs = []
    for row in soup.select("table tr"):
        cells = [c.get_text(strip=True) for c in row.find_all("td")]
        if not cells:
            continue

        atc_code = cells[0]
        # A substance-level code is the subgroup code + 2 digits, e.g. N05AB03
        if not (atc_code.startswith(subgroup_code) and len(atc_code) == len(subgroup_code) + 2):
            continue

        drug = {
            "atc_code": atc_code,
            "name":     cells[1] if len(cells) > 1 else None,
            "ddd":      cells[2] if len(cells) > 2 and cells[2] else None,
            "unit":     cells[3] if len(cells) > 3 and cells[3] else None,
            "route":    cells[4] if len(cells) > 4 and cells[4] else None,
            "note":     cells[5] if len(cells) > 5 and cells[5] else None,
            "subgroup": subgroup_code,
        }
        # Some substances span multiple DDD/route rows; the name cell is empty
        # on continuation rows, so carry the previous name forward.
        if not drug["name"] and drugs:
            drug["name"] = drugs[-1]["name"]
        drugs.append(drug)
    return drugs


# ----------------------------------------------------------------------
# Orchestration
# ----------------------------------------------------------------------
def scrape_atc(parent_code: str = "N05A", polite_delay: float = 0.5) -> list[dict]:
    with requests.Session() as session:
        root = fetch(parent_code, session)
        subgroups = get_child_codes(root, parent_code)

        all_drugs = []
        for sub in subgroups:
            sub_soup = fetch(sub, session)
            all_drugs.extend(parse_drugs(sub_soup, sub))
            time.sleep(polite_delay)   # be courteous to the server
        return all_drugs


# ----------------------------------------------------------------------
# CSV export
# ----------------------------------------------------------------------
def save_to_csv(drugs: list[dict], filename: str = "atc_n05a_drugs.csv") -> None:
    """
    Write the parsed drug entries to a CSV file.

    Each dict in `drugs` is written as one row using the column order in
    CSV_FIELDS. Missing values (None) are written as empty cells.
    """
    with open(filename, "w", newline="", encoding="utf-8") as f:
        writer = csv.DictWriter(f, fieldnames=CSV_FIELDS, extrasaction="ignore")
        writer.writeheader()
        for d in drugs:
            # Replace None with "" so empty cells are clean in the CSV
            writer.writerow({k: (d.get(k) if d.get(k) is not None else "") for k in CSV_FIELDS})

    print(f"Saved {len(drugs)} rows to '{filename}'")


# ----------------------------------------------------------------------
if __name__ == "__main__":
    drugs = scrape_atc("N05A")
    print(f"Found {len(drugs)} drug entries under N05A (Antipsychotics)\n")
    for d in drugs:
        ddd = f"{d['ddd']} {d['unit']} ({d['route']})" if d["ddd"] else "no DDD"
        print(f"{d['atc_code']:<9} {d['name']:<30} {ddd}")

    # If you only want the plain list of drug names:
    names = sorted({d["name"] for d in drugs if d["name"]})
    print("\nUnique drug names:")
    print(names)

    # Save the full result set to CSV
    save_to_csv(drugs, "atc_n05a_drugs.csv")

Found 70 drug entries under N05A (Antipsychotics)

N05AA01   chlorpromazine                 0.3 g (O)
N05AA02   levomepromazine                0.3 g (O)
N05AA03   promazine                      0.3 g (O)
N05AA04   acepromazine                   0.1 g (O)
N05AA05   triflupromazine                0.1 g (O)
N05AA06   cyamemazine                    no DDD
N05AA07   chlorproethazine               no DDD
N05AB01   dixyrazine                     50 mg (O)
N05AB02   fluphenazine                   10 mg (O)
N05AB03   perphenazine                   30 mg (O)
N05AB04   prochlorperazine               0.1 g (O)
N05AB05   thiopropazate                  60 mg (O)
N05AB06   trifluoperazine                20 mg (O)
N05AB07   acetophenazine                 50 mg (O)
N05AB08   thioproperazine                75 mg (O)
N05AB09   butaperazine                   10 mg (O)
N05AB10   perazine                       0.1 g (O)
N05AC01   periciazine                    50 mg (O)
N05AC02   thioridazine               

In [ ]:
!cp "/content/atc_n05a_drugs.csv" "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_database/atc_n05a_drugs.csv"

### Ki Db

In [2]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_database" /content/ -r

### Loading twas

In [ ]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity_cvs/dm" /content/ -r

In [3]:
!cp "/content/drive/MyDrive/Dr Uccello/00_Studies/Z_proximity/proximity_cvs/lipids" /content/ -r

In [5]:
!pip install rapidfuzz tqdm openpyxl

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 46.0 MB/s eta 0:00:00


# Version 1 (Original, no scaling for TWAS)

In [7]:
"""
N05A (Antipsychotics) → Ki → DDD → TWAS enrichment pipeline  (multi-TWAS)
========================================================================

For every TWAS directory in `TWAS_DIRS`, and for every drug in
`atc_n05a_drugs.csv`:
    1. Fuzzy-match the drug name to the "Test Ligands" column of the Ki database.
    2. Extract Ki (nM) for every receptor the drug binds.
    3. Aggregate multiple Ki measurements per drug-receptor pair
       (default = "min" => strongest binding).
    4. Compute:
         inv_Ki              = 1 / Ki_nM
         DDD_mg              = DDD normalised to mg
         affinity_dose_score = inv_Ki * log(DDD_mg + 1)
    5. Map each receptor -> official gene symbol (HTR2A, DRD2, ...).
    6. Look up TWAS enrichment (z / p) for that gene via a per-dir lookup.
    7. Optionally impute missing Ki and/or TWAS values (mean / median).
    8. Drop all entries whose affinity_dose_score == 0.
    9. Write long-format CSV, multi-sheet XLSX, and a per-drug summary
       (into a per-TWAS sub-folder).

Finally a single detailed `pipeline_summary.txt` is written aggregating
every TWAS directory that was processed.

Dependencies:
    pip install pandas numpy rapidfuzz tqdm openpyxl
"""

import os
import re
import glob
import datetime
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

# ======================================================================
# CONFIG  --  EDIT THESE PATHS
# ======================================================================
KI_DB_PATH     = "/content/Z_database/KiDatabase_2026-06-22.csv"
DRUGS_CSV_PATH = "/content/Z_database/atc_n05a_drugs.csv"

# ---- NOW A LIST: the pipeline repeats over each TWAS directory --------
TWAS_DIRS = [
    "/content/lipids/ldl",
    "/content/lipids/hdl",
    "/content/lipids/logtg",
    "/content/lipids/nonhdl",
    "/content/lipids/tc",
]

# Root output dir. Each TWAS dir gets its own sub-folder underneath.
OUTPUT_DIR = "/content/pipeline_output/n05a_ki_twas_log"

# Fuzzy matching: minimum score (0-100) to accept a drug<->ligand match
FUZZY_THRESHOLD = 80

# How to collapse multiple Ki measurements for the same drug+receptor:
#   "min"    -> strongest binding (lowest Ki)        [default]
#   "median" -> central tendency
#   "mean"   -> arithmetic mean
KI_AGGREGATION = "min"

# Threshold (nM) below which a binder is flagged as "strong"
STRONG_BINDER_KI_NM = 10.0

# TWAS significance threshold (p-value)
TWAS_P_SIGNIFICANT = 0.05

# ---- Imputation of MISSING receptor Ki -------------------------------
#   "none"   -> leave Ki missing (row excluded from scoring)
#   "mean"   -> per-receptor mean Ki (nM)
#   "median" -> per-receptor median Ki (nM)
KI_IMPUTE_STRATEGY = "mean"

# ---- Imputation of MISSING TWAS values -------------------------------
#   "none"   -> leave TWAS missing
#   "mean"   -> global mean twas_z / twas_p
#   "median" -> global median twas_z / twas_p
TWAS_IMPUTE_STRATEGY = "mean"

# Encodings to try, in order, when reading any CSV/TSV.
CSV_ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

# ---- TWAS column auto-detection --------------------------------------
TWAS_SYMBOL_COL_CANDIDATES = [
    "gene_name", "genename", "gene_symbol", "symbol",
    "hgnc_symbol",
]
TWAS_ENSG_COL_CANDIDATES = [
    "gene", "ensembl_gene_id", "gene_id", "geneid", "ensembl", "id",
]
TWAS_Z_COL_CANDIDATES = [
    "twas.z", "twas_z", "zscore", "z_score", "zstat", "z", "zscore",
]
TWAS_P_COL_CANDIDATES = [
    "twas.p", "twas_p", "pvalue", "p_value", "pval", "p",
]


# ======================================================================
# Receptor (Ki DB label) -> official HGNC gene symbol
# ======================================================================
RECEPTOR_TO_GENE = {
    # --- Serotonin (5-HT) ---
    "5-HT1A": "HTR1A", "5HT1A": "HTR1A",
    "5-HT1B": "HTR1B", "5HT1B": "HTR1B",
    "5-HT1D": "HTR1D",
    "5-HT1E": "HTR1E",
    "5-HT1F": "HTR1F",
    "5-HT2A": "HTR2A", "5HT2A": "HTR2A",
    "5-HT2B": "HTR2B",
    "5-HT2C": "HTR2C", "5HT2C": "HTR2C",
    "5-HT3":  "HTR3A",
    "5-HT5A": "HTR5A",
    "5-HT6":  "HTR6",
    "5-HT7":  "HTR7",
    # --- Dopamine ---
    "D1": "DRD1", "D2": "DRD2", "D3": "DRD3", "D4": "DRD4", "D5": "DRD5",
    "DRD1": "DRD1", "DRD2": "DRD2", "DRD3": "DRD3", "DRD4": "DRD4", "DRD5": "DRD5",
    # --- Adrenergic ---
    "alpha1A": "ADRA1A", "alpha1B": "ADRA1B", "alpha1D": "ADRA1D",
    "alpha2A": "ADRA2A", "alpha2B": "ADRA2B", "alpha2C": "ADRA2C",
    "alpha1":  "ADRA1A", "alpha2": "ADRA2A",
    "beta1": "ADRB1", "beta2": "ADRB2", "beta3": "ADRB3",
    # --- Histamine ---
    "H1": "HRH1", "H2": "HRH2", "H3": "HRH3", "H4": "HRH4",
    # --- Muscarinic acetylcholine ---
    "M1": "CHRM1", "M2": "CHRM2", "M3": "CHRM3", "M4": "CHRM4", "M5": "CHRM5",
    "Muscarinic": "CHRM1",
    # --- Sigma / others sometimes present ---
    "Sigma1": "SIGMAR1", "Sigma 1": "SIGMAR1",
    "SERT": "SLC6A4", "DAT": "SLC6A3", "NET": "SLC6A2",
}
# normalise keys
RECEPTOR_TO_GENE = {k.strip().lower(): v for k, v in RECEPTOR_TO_GENE.items()}


# ======================================================================
# METABOLIC / DIABETES RISK WEIGHTS FOR ANTIPSYCHOTICS
# ======================================================================
METABOLIC_WEIGHTS = {
    # === Highest evidence (core metabolic risk receptors) ===
    "HRH1":    1.00,   # H1 histamine - strongest predictor of weight gain
    "HTR2C":   0.92,   # 5-HT2C - appetite dysregulation & weight gain
    "CHRM3":   0.85,   # M3 muscarinic - impairs insulin secretion

    # === Strong / moderate evidence ===
    "HTR2A":   0.55,
    "ADRA1A":  0.48,
    "ADRA1B":  0.48,
    "HTR6":    0.42,

    # === Moderate / weaker evidence ===
    "ADRA2A":  0.32,
    "ADRA2B":  0.30,
    "ADRA2C":  0.30,
    "CHRM1":   0.28,
    "CHRM4":   0.22,
    "CHRM5":   0.20,
    "HTR7":    0.25,
    "DRD3":    0.18,

    # === Low evidence for metabolic risk ===
    "DRD2":    0.12,
    "DRD4":    0.10,
    "HTR1A":   0.08,
    "ADRB1":   0.08,
    "ADRB2":   0.07,
    "SIGMAR1": 0.05,

    # === Transporters (generally low for metabolic risk) ===
    "SLC6A4":  0.06,   # SERT
    "SLC6A2":  0.05,   # NET
    "SLC6A3":  0.04,   # DAT
}


# ======================================================================
# Helpers
# ======================================================================
def _is_nan(x) -> bool:
    """Safe NaN check that tolerates None / non-floats."""
    if x is None:
        return True
    try:
        return bool(np.isnan(x))
    except (TypeError, ValueError):
        return False


def read_csv_robust(path: str, **kwargs) -> pd.DataFrame:
    """
    Read a CSV/TSV trying several encodings in order. Defaults to
    low_memory=False to avoid mixed-dtype chunk warnings on big files.
    """
    kwargs.setdefault("low_memory", False)
    last_err = None
    for enc in CSV_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    # final attempt: replace undecodable bytes so we never hard-crash
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", **kwargs)
    except Exception:
        raise last_err if last_err else RuntimeError(f"Could not read {path}")


def normalise_name(s: str) -> str:
    """Lower-case, strip, collapse whitespace for matching."""
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip().lower())


def receptor_to_gene(receptor: str) -> str | None:
    """Map a Ki-DB receptor label to a gene symbol (or None if unknown)."""
    if not isinstance(receptor, str):
        return None
    key = receptor.strip().lower()
    if key in RECEPTOR_TO_GENE:
        return RECEPTOR_TO_GENE[key]
    # try a looser variant: remove spaces / hyphens
    key2 = re.sub(r"[\s\-]", "", key)
    for k, v in RECEPTOR_TO_GENE.items():
        if re.sub(r"[\s\-]", "", k) == key2:
            return v
    return None


def parse_ki(value) -> float:
    """
    Convert a Ki cell to a float (nM). Handles strings like '>10000',
    '~5.3', '1,200', blanks, etc. Returns np.nan when unparseable.
    """
    if value is None:
        return np.nan
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"na", "nan", "nd", "n/a", "-"}:
        return np.nan
    s = s.replace(",", "")                 # thousands separators
    s = re.sub(r"^[><=~]+", "", s)         # leading >, <, =, ~
    m = re.search(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", s)
    return float(m.group()) if m else np.nan


def ddd_to_mg(ddd, unit) -> float:
    """Normalise a DDD value to milligrams. Returns np.nan if missing."""
    ddd_val = parse_ki(ddd)          # reuse the numeric parser
    if _is_nan(ddd_val):
        return np.nan
    u = (str(unit).strip().lower() if unit is not None else "")
    factors = {"g": 1000.0, "mg": 1.0, "mcg": 0.001, "µg": 0.001, "ug": 0.001}
    factor = factors.get(u, 1.0)     # assume mg if unit unrecognised
    return ddd_val * factor


def aggregate_ki(values: pd.Series, how: str) -> float:
    vals = values.dropna()
    if vals.empty:
        return np.nan
    if how == "min":
        return float(vals.min())
    if how == "median":
        return float(vals.median())
    if how == "mean":
        return float(vals.mean())
    return float(vals.min())


def find_col(df: pd.DataFrame, candidates: list[str]) -> str | None:
    """Return the first column in df whose (lower) name matches a candidate."""
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _strip_ensg_version(s) -> str:
    """ENSG00000168763.15 -> ENSG00000168763 (and upper/strip)."""
    return re.sub(r"\.\d+$", "", str(s).strip().upper())


def _safe_label(path: str) -> str:
    """Make a filesystem-safe label from a TWAS directory path."""
    base = os.path.basename(os.path.normpath(path))
    if not base:
        base = re.sub(r"[^A-Za-z0-9_.-]+", "_", path).strip("_")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", base).strip("_") or "twas"


# ======================================================================
# Recompute affinity-derived columns (used after Ki imputation)
# ======================================================================
def _recompute_affinity_columns(results: pd.DataFrame) -> pd.DataFrame:
    """
    Recompute inv_Ki, affinity_dose_score and strong_binder from the
    (possibly imputed) Ki_nM and DDD_mg columns. Keeps the table
    internally consistent after any Ki imputation step.

    Score formula:  affinity_dose_score = inv_Ki * log(DDD_mg + 1)
    (falls back to inv_Ki alone when DDD is missing).
    """
    ki_nM  = results["Ki_nM"]
    ddd_mg = results["DDD_mg"]

    has_ki = ki_nM.notna() & (ki_nM > 0)
    results["inv_Ki"] = np.where(has_ki, 1.0 / ki_nM, np.nan)

    has_inv_ki = results["inv_Ki"].notna()
    has_ddd    = ddd_mg.notna() & (ddd_mg > 0)
    log_ddd    = np.log(ddd_mg.where(has_ddd) + 1.0)
    results["affinity_dose_score"] = np.where(
        has_inv_ki & has_ddd, results["inv_Ki"] * log_ddd,
        np.where(has_inv_ki, results["inv_Ki"], np.nan),
    )

    results["strong_binder"] = ki_nM.notna() & (ki_nM < STRONG_BINDER_KI_NM)
    return results


# ======================================================================
# Drop rows whose affinity_dose_score == 0
# ======================================================================
def drop_zero_score_rows(results: pd.DataFrame) -> tuple[pd.DataFrame, int]:
    """
    Delete every entry whose affinity_dose_score is exactly 0.
    Returns the filtered frame and the number of rows removed.
    NaN scores (e.g. unmatched drugs) are left untouched.
    """
    if results.empty or "affinity_dose_score" not in results.columns:
        return results, 0
    zero_mask = results["affinity_dose_score"] == 0
    n_dropped = int(zero_mask.sum())
    if n_dropped:
        results = results.loc[~zero_mask].reset_index(drop=True)
        print(f"  [drop-zero] removed {n_dropped} row(s) with affinity_dose_score == 0.")
    else:
        print("  [drop-zero] no rows with affinity_dose_score == 0.")
    return results, n_dropped


# ======================================================================
# Imputation of missing receptor Ki
# ======================================================================
def impute_missing_ki(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    """
    Fill missing Ki values with a per-receptor reference (mean/median),
    falling back to a global reference. Adds a `ki_imputed` column and
    recomputes affinity-derived columns for the filled rows.
    """
    if results.empty:
        return results
    if "ki_imputed" not in results.columns:
        results["ki_imputed"] = False

    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown KI_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )

    valid = results.loc[results["Ki_nM"].notna(), ["receptor", "Ki_nM"]]
    if valid.empty:
        print("  [ki-impute] no Ki values available to build a reference; skipped.")
        return results

    if strategy == "mean":
        per_receptor = valid.groupby("receptor")["Ki_nM"].mean()
        global_ref = float(valid["Ki_nM"].mean())
    else:  # median
        per_receptor = valid.groupby("receptor")["Ki_nM"].median()
        global_ref = float(valid["Ki_nM"].median())

    # rows needing imputation: a receptor is present but Ki is missing
    need = results["Ki_nM"].isna() & results["receptor"].notna()
    n_need = int(need.sum())
    if n_need == 0:
        print(f"  [ki-impute] strategy='{strategy}': no missing Ki values to fill.")
        return results

    filled_ref = results.loc[need, "receptor"].map(per_receptor)
    filled_ref = filled_ref.fillna(global_ref)   # global fallback

    results.loc[need, "Ki_nM"] = filled_ref.values
    results.loc[need, "ki_imputed"] = True

    results = _recompute_affinity_columns(results)

    print(f"  [ki-impute] strategy='{strategy}': filled {n_need} missing Ki "
          f"value(s) (global fallback Ki = {global_ref:.3g} nM).")
    return results


# ======================================================================
# Imputation of missing TWAS values
# ======================================================================
def impute_missing_twas(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    """
    Fill missing TWAS values (recognised genes with no twas_z / twas_p)
    with the GLOBAL mean or median of the values that were found.
    Imputed rows are deliberately NOT marked TWAS-significant.
    """
    if results.empty:
        return results
    if "twas_imputed" not in results.columns:
        results["twas_imputed"] = False

    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown TWAS_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )

    valid_z = results.loc[results["twas_z"].notna(), "twas_z"]
    valid_p = results.loc[results["twas_p"].notna(), "twas_p"]
    if valid_z.empty and valid_p.empty:
        print("  [twas-impute] no TWAS values available to build a reference; skipped.")
        return results

    if strategy == "mean":
        fill_z = float(valid_z.mean()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.mean()) if not valid_p.empty else np.nan
    else:  # median
        fill_z = float(valid_z.median()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.median()) if not valid_p.empty else np.nan

    # only impute rows that map to a recognised gene
    need_z = results["gene"].notna() & results["twas_z"].isna()
    need_p = results["gene"].notna() & results["twas_p"].isna()
    n_need = int((need_z | need_p).sum())
    if n_need == 0:
        print(f"  [twas-impute] strategy='{strategy}': no missing TWAS values to fill.")
        return results

    if not _is_nan(fill_z):
        results.loc[need_z, "twas_z"] = fill_z
    if not _is_nan(fill_p):
        results.loc[need_p, "twas_p"] = fill_p

    results.loc[need_z | need_p, "twas_imputed"] = True
    # NOTE: do not flip twas_significant on for imputed rows.

    print(f"  [twas-impute] strategy='{strategy}': filled {n_need} gene row(s) "
          f"(twas_z={fill_z:.3f}, twas_p={fill_p:.3g}); "
          f"imputed rows kept NON-significant.")
    return results


# ======================================================================
# TWAS lookup  --  built fresh per TWAS directory
# ======================================================================
def _read_twas_file(fpath: str) -> pd.DataFrame | None:
    """Very defensive file reader. Returns a DataFrame or None."""
    # Strategy 1: Try common combinations
    for sep in [",", "\t", None]:
        for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
            try:
                df = pd.read_csv(
                    fpath,
                    sep=sep,
                    encoding=enc,
                    engine="python",
                    low_memory=False,
                    on_bad_lines="skip"   # skip bad lines instead of failing
                )
                if df is not None and not df.empty and df.shape[1] >= 2:
                    return df
            except Exception:
                continue

    # Strategy 2: Let pandas auto-detect everything (last resort)
    try:
        df = pd.read_csv(fpath, engine="python", on_bad_lines="skip")
        if df is not None and not df.empty:
            return df
    except Exception:
        pass

    return None


def build_twas_lookup(twas_dir: str) -> dict[str, dict]:
    """
    Build a fast gene -> {'twas_z':, 'twas_p':} lookup from every file in
    `twas_dir`. Keeps the most-significant (smallest p) record per gene.
    """
    lookup: dict[str, dict] = {}
    files = glob.glob(os.path.join(twas_dir, "*.*"))

    if not files:
        print(f"  [warn] No files found in {twas_dir}")
        return lookup

    successful_files = 0
    skipped_files = []

    for fpath in files:
        try:
            df = _read_twas_file(fpath)
            if df is None or df.empty:
                skipped_files.append((os.path.basename(fpath), "unreadable/empty"))
                continue

            sym_col  = find_col(df, TWAS_SYMBOL_COL_CANDIDATES)
            ensg_col = find_col(df, TWAS_ENSG_COL_CANDIDATES)
            z_col    = find_col(df, TWAS_Z_COL_CANDIDATES)
            p_col    = find_col(df, TWAS_P_COL_CANDIDATES)

            # don't let the ENSG fallback collide with the symbol column
            if ensg_col is not None and ensg_col == sym_col:
                ensg_col = None

            if z_col is None or (sym_col is None and ensg_col is None):
                skipped_files.append((
                    os.path.basename(fpath),
                    f"missing cols (z={z_col}, sym={sym_col}, ensg={ensg_col}); "
                    f"had {list(df.columns)[:8]}"
                ))
                continue

            # ---- vectorised extraction for speed ----
            n = len(df)
            sym_vals  = (df[sym_col].astype(str).str.strip().str.upper()
                         if sym_col else pd.Series([""] * n, index=df.index))
            ensg_vals = (df[ensg_col].map(_strip_ensg_version)
                         if ensg_col else pd.Series([""] * n, index=df.index))
            z_vals = df[z_col].map(parse_ki)
            p_vals = (df[p_col].map(parse_ki)
                      if p_col else pd.Series([np.nan] * n, index=df.index))

            bad = {"NAN", "NA", "NONE", ""}
            for i in df.index:
                gene = sym_vals[i]
                if gene in bad:
                    gene = ensg_vals[i]
                if gene in bad:
                    continue

                z = z_vals[i]
                p = p_vals[i]
                if _is_nan(z) and _is_nan(p):
                    continue

                prev = lookup.get(gene)
                if prev is None:
                    lookup[gene] = {"twas_z": z, "twas_p": p}
                else:
                    prev_p = prev.get("twas_p", np.nan)
                    if not _is_nan(p) and (_is_nan(prev_p) or p < prev_p):
                        lookup[gene] = {"twas_z": z, "twas_p": p}

            successful_files += 1

        except Exception as e:
            skipped_files.append((os.path.basename(fpath), f"error: {e}"))
            continue

    print(f"  Pre-loaded TWAS data for {len(lookup):,} genes "
          f"from {successful_files}/{len(files)} file(s)")
    if skipped_files:
        print(f"  [warn] {len(skipped_files)} file(s) skipped:")
        for name, reason in skipped_files[:8]:
            print(f"         - {name}: {reason}")

    return lookup


def get_twas_for_gene(gene: str, lookup: dict[str, dict]) -> dict | None:
    """Return {'twas_z':.., 'twas_p':..} for a gene symbol, or None."""
    if gene is None or lookup is None:
        return None
    return lookup.get(str(gene).strip().upper())


# ======================================================================
# Metabolic risk score
# ======================================================================
def calculate_metabolic_risk_score(
    df: pd.DataFrame,
    weights: dict = None,
    score_transform: str = "log1p",      # "log1p" | "none"
    twas_boost: bool = True,
    twas_boost_factor: float = 0.12,     # slightly lower if TWAS coverage is partial
    twas_sig_bonus: float = 0.0,         # extra multiplier per TWAS-significant gene
    normalize: bool = True,
    reference_drug: str = "chlorpromazine",
    missing_gene_strategy: str = "mean_abs_z",   # "mean_abs_z" | "zero"
) -> pd.DataFrame:
    """
    Calculate a per-drug Metabolic Risk Score for antipsychotics.
    See original docstring for the fairness handling of missing TWAS genes.
    """
    if weights is None:
        weights = METABOLIC_WEIGHTS

    df = df.copy()

    # only rows with a recognised gene and a valid affinity score
    mask = df["gene"].notna() & df["affinity_dose_score"].notna()

    # ---- report TWAS coverage among scored metabolic receptors ----
    metab_mask = mask & df["gene"].isin(weights.keys())
    n_metab = int(metab_mask.sum())
    n_metab_twas = int((metab_mask & df["twas_z"].notna()).sum())
    if twas_boost and n_metab > 0:
        cov = n_metab_twas / n_metab
        if cov < 0.5:
            print(f"  [note] TWAS coverage is low: only {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS z-score. "
                  f"The TWAS boost will be partial; consider lowering "
                  f"twas_boost_factor or setting twas_boost=False.")
        else:
            print(f"  [note] TWAS coverage: {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS z-score.")

    if score_transform == "log1p":
        affinity = np.log1p(df.loc[mask, "affinity_dose_score"].clip(lower=0))
    elif score_transform == "none":
        affinity = df.loc[mask, "affinity_dose_score"]
    else:
        raise ValueError(f"Unknown score_transform: {score_transform!r}")

    df.loc[mask, "receptor_weight"] = df.loc[mask, "gene"].map(weights).fillna(0.0)
    df.loc[mask, "weighted_contribution"] = affinity * df.loc[mask, "receptor_weight"]

    drug_scores = (
        df.groupby("drug", dropna=False)["weighted_contribution"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_contribution": "base_score"})
    )

    # === TWAS |z| boost (with fairness handling for missing genes) ===
    if twas_boost:
        # Work on a copy so the imputation never leaks into the returned df
        df_filled = df.copy()

        if missing_gene_strategy == "mean_abs_z":
            available_z = df_filled.loc[
                metab_mask & df_filled["twas_z"].notna(), "twas_z"
            ].abs()
            mean_abs_z = float(available_z.mean()) if len(available_z) > 0 else 0.0

            impute_mask = (
                mask
                & df_filled["twas_z"].isna()
                & df_filled["gene"].isin(weights.keys())
            )
            n_imputed = int(impute_mask.sum())
            df_filled.loc[impute_mask, "twas_z"] = mean_abs_z
            print(f"  [fairness] missing_gene_strategy='mean_abs_z': imputed "
                  f"{n_imputed} missing metabolic-gene row(s) with "
                  f"mean |twas_z| = {mean_abs_z:.3f}")
        elif missing_gene_strategy == "zero":
            print("  [fairness] missing_gene_strategy='zero': missing genes "
                  "receive no TWAS boost (legacy behaviour).")
        else:
            raise ValueError(
                f"Unknown missing_gene_strategy: {missing_gene_strategy!r} "
                f"(expected 'mean_abs_z' or 'zero')"
            )

        twas_signal = (
            df_filled.groupby("drug")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
            .reset_index()
            .rename(columns={"twas_z": "max_abs_twas_z"})
        )
        drug_scores = drug_scores.merge(twas_signal, on="drug", how="left")
        drug_scores["max_abs_twas_z"] = drug_scores["max_abs_twas_z"].fillna(0.0)
        boost = 1 + twas_boost_factor * drug_scores["max_abs_twas_z"]
    else:
        boost = 1.0

    # bonus for number of TWAS-significant metabolic receptors
    if twas_sig_bonus and "twas_significant" in df.columns:
        sig = df[mask & df["twas_significant"].fillna(False)
                 & (df["receptor_weight"] > 0)]
        n_sig = (
            sig.groupby("drug").size()
            .reset_index(name="n_twas_sig_metabolic")
        )
        drug_scores = drug_scores.merge(n_sig, on="drug", how="left")
        drug_scores["n_twas_sig_metabolic"] = (
            drug_scores["n_twas_sig_metabolic"].fillna(0).astype(int)
        )
        sig_mult = 1 + twas_sig_bonus * drug_scores["n_twas_sig_metabolic"]
    else:
        sig_mult = 1.0

    drug_scores["metabolic_risk_score"] = (
        drug_scores["base_score"] * boost * sig_mult
    )

    # normalise relative to reference drug
    if normalize:
        ref_mask = drug_scores["drug"].astype(str).str.lower() \
            == reference_drug.lower()
        ref_score = drug_scores.loc[ref_mask, "metabolic_risk_score"]

        if not ref_score.empty and ref_score.values[0] > 0:
            ref = ref_score.values[0]
        else:
            print(f"  [warn] reference drug '{reference_drug}' not found or "
                  f"zero-scored; normalising to the max score instead.")
            ref = drug_scores["metabolic_risk_score"].max()

        if ref and ref > 0:
            drug_scores["metabolic_risk_score"] = (
                drug_scores["metabolic_risk_score"] / ref * 100
            )
            drug_scores["risk_vs_reference"] = (
                drug_scores["metabolic_risk_score"] - 100
            )

    # --- drop drugs whose metabolic_risk_score == 0 -------------------
    n_zero_drugs = int((drug_scores["metabolic_risk_score"] == 0).sum())
    if n_zero_drugs:
        drug_scores = drug_scores[drug_scores["metabolic_risk_score"] != 0]
        print(f"  [drop-zero] removed {n_zero_drugs} drug(s) with "
              f"metabolic_risk_score == 0.")

    drug_scores["risk_rank"] = (
        drug_scores["metabolic_risk_score"]
        .rank(ascending=False, method="min")
        .astype("Int64")
    )
    drug_scores = drug_scores.sort_values(
        "metabolic_risk_score", ascending=False
    ).reset_index(drop=True)

    preferred = [
        "risk_rank", "drug", "metabolic_risk_score", "base_score",
        "max_abs_twas_z", "n_twas_sig_metabolic", "risk_vs_reference",
    ]
    cols = [c for c in preferred if c in drug_scores.columns]
    return drug_scores[cols]


# ======================================================================
# Core pipeline  (now per TWAS directory)
# ======================================================================
def run_pipeline(twas_dir: str, output_dir: str) -> tuple[pd.DataFrame, dict]:
    """
    Run the full Ki -> DDD -> TWAS pipeline for a SINGLE TWAS directory.
    Returns (results_dataframe, run_stats_dict).
    """
    os.makedirs(output_dir, exist_ok=True)
    stats: dict = {"twas_dir": twas_dir, "output_dir": output_dir}

    # --- build TWAS lookup for THIS directory -----------------------
    print(f"Building TWAS lookup for: {twas_dir}")
    twas_lookup = build_twas_lookup(twas_dir)
    stats["n_genes_twas"] = len(twas_lookup)

    # quick sanity check on what actually loaded
    sample = list(twas_lookup.keys())[:10] if twas_lookup else "EMPTY"
    print(f"  Sample genes in TWAS lookup: {sample}")
    for g in ("HTR2C", "HRH1", "DRD2", "CHRM1", "CHRM3", "HTR2A"):
        print(f"    {g} present? {g in (twas_lookup or {})}")

    # --- load inputs -------------------------------------------------
    print("Loading Ki database ...")
    ki = read_csv_robust(KI_DB_PATH)
    print(f"  Ki rows: {len(ki):,}")

    print("Loading drug list ...")
    drugs = read_csv_robust(DRUGS_CSV_PATH)
    print(f"  Drugs: {len(drugs):,}")
    stats["n_drugs_input"] = len(drugs)

    # Identify the relevant Ki-DB columns (robust to header variations)
    ligand_col   = find_col(ki, ["Test Ligands", "Test Ligand", "Ligand", "ligand"])
    receptor_col = find_col(ki, ["Receptor", "receptor", "Target"])
    kival_col    = find_col(ki, ["Ki Value", "Ki", "Ki_nM", "Ki (nM)"])
    if not all([ligand_col, receptor_col, kival_col]):
        raise ValueError(
            f"Could not locate required Ki columns. Found: "
            f"ligand={ligand_col}, receptor={receptor_col}, ki={kival_col}\n"
            f"Available columns: {list(ki.columns)}"
        )

    # Pre-compute normalised, numeric helper columns on the Ki DB
    ki["_ligand_norm"] = ki[ligand_col].map(normalise_name)
    ki["_ki_nM"]       = ki[kival_col].map(parse_ki)

    # Unique ligand vocabulary for fuzzy matching
    ligand_vocab = sorted(set(ki["_ligand_norm"]) - {""})

    rows = []
    missing_twas: set[str] = set()   # genes with no TWAS data (reported once)

    print("\nMatching drugs and building drug x receptor table ...")
    for _, drug in tqdm(drugs.iterrows(), total=len(drugs)):
        drug_name = drug.get("name")
        norm = normalise_name(drug_name)
        ddd_mg = ddd_to_mg(drug.get("ddd"), drug.get("unit"))
        inv_ddd = (1.0 / ddd_mg) if (not _is_nan(ddd_mg) and ddd_mg > 0) else np.nan

        if not norm:
            continue

        # ---- fuzzy match drug name -> ligand label ----
        match = process.extractOne(norm, ligand_vocab, scorer=fuzz.WRatio)
        if match is None or match[1] < FUZZY_THRESHOLD:
            # record an unmatched drug so it isn't silently lost
            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": None,
                "match_score": (match[1] if match else np.nan),
                "receptor": None, "gene": None,
                "Ki_nM": np.nan, "inv_Ki": np.nan,
                "DDD_mg": ddd_mg, "inv_DDD": inv_ddd,
                "affinity_dose_score": np.nan,
                "n_measurements": 0,
                "twas_z": np.nan, "twas_p": np.nan,
                "strong_binder": False, "twas_significant": False,
                "ki_imputed": False, "twas_imputed": False,
            })
            continue

        matched_ligand, score, _ = match
        subset = ki[ki["_ligand_norm"] == matched_ligand]

        # ---- aggregate Ki per receptor ----
        grouped = subset.groupby(receptor_col)
        for receptor, grp in grouped:
            ki_nM = aggregate_ki(grp["_ki_nM"], KI_AGGREGATION)
            n_meas = int(grp["_ki_nM"].notna().sum())
            inv_ki = (1.0 / ki_nM) if (not _is_nan(ki_nM) and ki_nM > 0) else np.nan

            gene = receptor_to_gene(receptor)

            # combined score = inv_Ki * log(DDD_mg + 1)
            # (falls back to inv_Ki if DDD missing)
            if not _is_nan(inv_ki) and not _is_nan(ddd_mg) and ddd_mg > 0:
                score_combined = inv_ki * np.log(ddd_mg + 1.0)
            elif not _is_nan(inv_ki):
                score_combined = inv_ki
            else:
                score_combined = np.nan

            # TWAS lookup (fast dict access; misses collected quietly)
            twas = get_twas_for_gene(gene, twas_lookup) if gene else None
            if gene and twas is None:
                missing_twas.add(gene)
            twas_z = twas.get("twas_z", np.nan) if twas else np.nan
            twas_p = twas.get("twas_p", np.nan) if twas else np.nan

            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": matched_ligand,
                "match_score": round(score, 1),
                "receptor": receptor,
                "gene": gene,
                "Ki_nM": ki_nM,
                "inv_Ki": inv_ki,
                "DDD_mg": ddd_mg,
                "inv_DDD": inv_ddd,
                "affinity_dose_score": score_combined,
                "n_measurements": n_meas,
                "twas_z": twas_z,
                "twas_p": twas_p,
                "strong_binder": (not _is_nan(ki_nM)) and ki_nM < STRONG_BINDER_KI_NM,
                "twas_significant": (not _is_nan(twas_p)) and twas_p < TWAS_P_SIGNIFICANT,
                "ki_imputed": False,
                "twas_imputed": False,
            })

    results = pd.DataFrame(rows)

    # one concise report of genes that had no TWAS data
    stats["n_missing_twas_genes"] = len(missing_twas)
    stats["missing_twas_sample"] = sorted(missing_twas)[:12]
    if missing_twas:
        sample = sorted(missing_twas)[:8]
        print(f"\n[TWAS] {len(missing_twas)} receptor gene(s) had no TWAS data "
              f"(e.g. {', '.join(sample)}"
              f"{', ...' if len(missing_twas) > len(sample) else ''})")

    # ---- Only keep drugs that have a valid DDD --------------------
    if not results.empty:
        n_before = results["drug"].nunique()
        valid_ddd_drugs = results[results["DDD_mg"].notna()]["drug"].unique()
        results = results[results["drug"].isin(valid_ddd_drugs)].reset_index(drop=True)
        n_after = results["drug"].nunique()
        stats["n_drugs_ddd_dropped"] = int(n_before - n_after)
        print(f"\n[DDD filter] kept {n_after}/{n_before} drug(s) with a valid DDD "
              f"({n_before - n_after} dropped).")

    # --- optional imputation of missing Ki / TWAS values ------------
    if not results.empty:
        print("\nImputation step ...")
        results = impute_missing_ki(results, KI_IMPUTE_STRATEGY)
        results = impute_missing_twas(results, TWAS_IMPUTE_STRATEGY)

    # --- DROP entries with affinity_dose_score == 0 -----------------
    results, n_zero_dropped = drop_zero_score_rows(results)
    stats["n_zero_score_dropped"] = n_zero_dropped

    # rank within each drug by combined score (strongest first)
    if not results.empty:
        results["rank_in_drug"] = (
            results.groupby("drug")["affinity_dose_score"]
            .rank(ascending=False, method="min")
        )
        results = results.sort_values(
            ["drug", "affinity_dose_score"],
            ascending=[True, False]
        ).reset_index(drop=True)

    # --- tally stats -------------------------------------------------
    if not results.empty:
        stats["n_drugs_matched"] = int(
            results.loc[results["receptor"].notna(), "drug"].nunique()
        )
        stats["n_pairs"] = int(results["receptor"].notna().sum())
        stats["n_ki_imputed"] = int(results.get("ki_imputed",
                                     pd.Series(dtype=bool)).sum())
        stats["n_twas_imputed"] = int(results.get("twas_imputed",
                                       pd.Series(dtype=bool)).sum())
        stats["n_strong_binders"] = int(results["strong_binder"].sum())
        stats["n_twas_significant"] = int(results["twas_significant"].sum())
    else:
        stats.update({
            "n_drugs_matched": 0, "n_pairs": 0, "n_ki_imputed": 0,
            "n_twas_imputed": 0, "n_strong_binders": 0,
            "n_twas_significant": 0,
        })

    return results, stats


def write_outputs(results: pd.DataFrame, output_dir: str) -> dict:
    """Write CSV / XLSX / per-drug summary for one TWAS run."""
    paths = {}
    if results.empty:
        print("No results to write.")
        return paths

    # ---- main long-format CSV ----
    csv_path = os.path.join(output_dir, "n05a_ki_twas_full_results.csv")
    results.to_csv(csv_path, index=False)
    print(f"  wrote {csv_path}")
    paths["csv"] = csv_path

    # ---- per-drug summary ----
    matched = results[results["receptor"].notna()]
    summary = (
        matched.groupby(["atc_code", "drug"])
        .agg(
            n_receptors      = ("receptor", "nunique"),
            n_strong_binders = ("strong_binder", "sum"),
            n_twas_sig       = ("twas_significant", "sum"),
            n_ki_imputed     = ("ki_imputed", "sum"),
            n_twas_imputed   = ("twas_imputed", "sum"),
            best_Ki_nM       = ("Ki_nM", "min"),
            top_score        = ("affinity_dose_score", "max"),
            DDD_mg           = ("DDD_mg", "first"),
        )
        .reset_index()
        .sort_values("top_score", ascending=False)
    )
    summary_path = os.path.join(output_dir, "drug_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"  wrote {summary_path}")
    paths["summary"] = summary_path

    # ---- multi-sheet Excel ----
    xlsx_path = os.path.join(output_dir, "n05a_ki_twas_full_results.xlsx")
    top5 = (
        matched[matched["rank_in_drug"] <= 5]
        .sort_values(["drug", "rank_in_drug"])
    )
    twas_sig = matched[matched["twas_significant"]].sort_values(
        "affinity_dose_score", ascending=False
    )
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
            results.to_excel(xw, sheet_name="All", index=False)
            top5.to_excel(xw, sheet_name="Top5_per_drug", index=False)
            twas_sig.to_excel(xw, sheet_name="TWAS_significant", index=False)
            summary.to_excel(xw, sheet_name="Drug_summary", index=False)
        print(f"  wrote {xlsx_path}")
        paths["xlsx"] = xlsx_path
    except Exception as e:
        print(f"  (skipped Excel export: {e})")

    return paths


# ======================================================================
# Detailed final summary (.txt)
# ======================================================================
def write_detailed_summary(run_summaries: list[dict], path: str) -> None:
    """Write a single human-readable summary covering every TWAS run."""
    L: list[str] = []
    bar = "=" * 78
    sub = "-" * 78

    L.append(bar)
    L.append("N05A  Ki -> DDD -> TWAS ENRICHMENT PIPELINE — DETAILED SUMMARY")
    L.append(bar)
    L.append(f"Generated            : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
    L.append(f"Ki database          : {KI_DB_PATH}")
    L.append(f"Drug list            : {DRUGS_CSV_PATH}")
    L.append(f"Output root          : {OUTPUT_DIR}")
    L.append(f"Score formula        : affinity_dose_score = inv_Ki * log(DDD_mg + 1)")
    L.append(f"TWAS dirs processed  : {len(run_summaries)}")
    L.append("")
    L.append("Configuration:")
    L.append(f"  FUZZY_THRESHOLD       = {FUZZY_THRESHOLD}")
    L.append(f"  KI_AGGREGATION        = {KI_AGGREGATION}")
    L.append(f"  STRONG_BINDER_KI_NM   = {STRONG_BINDER_KI_NM}")
    L.append(f"  TWAS_P_SIGNIFICANT    = {TWAS_P_SIGNIFICANT}")
    L.append(f"  KI_IMPUTE_STRATEGY    = {KI_IMPUTE_STRATEGY}")
    L.append(f"  TWAS_IMPUTE_STRATEGY  = {TWAS_IMPUTE_STRATEGY}")
    L.append("")

    # ---- cross-run overview table ----
    L.append(bar)
    L.append("CROSS-RUN OVERVIEW")
    L.append(bar)
    header = (f"{'TWAS label':<18}{'genes':>8}{'matched':>9}{'pairs':>8}"
              f"{'zero-drop':>11}{'ki_imp':>8}{'tw_imp':>8}{'tw_sig':>8}")
    L.append(header)
    L.append(sub)
    for s in run_summaries:
        if s.get("error"):
            L.append(f"{s.get('label',''):<18}  ERROR: {s['error']}")
            continue
        L.append(
            f"{s.get('label',''):<18}"
            f"{s.get('n_genes_twas',0):>8}"
            f"{s.get('n_drugs_matched',0):>9}"
            f"{s.get('n_pairs',0):>8}"
            f"{s.get('n_zero_score_dropped',0):>11}"
            f"{s.get('n_ki_imputed',0):>8}"
            f"{s.get('n_twas_imputed',0):>8}"
            f"{s.get('n_twas_significant',0):>8}"
        )
    L.append("")

    # ---- per-run detail ----
    for s in run_summaries:
        L.append(bar)
        L.append(f"TWAS RUN: {s.get('label','')}")
        L.append(bar)
        L.append(f"  TWAS directory          : {s.get('twas_dir','')}")
        L.append(f"  Output directory        : {s.get('output_dir','')}")
        if s.get("error"):
            L.append(f"  [ERROR] {s['error']}")
            L.append("")
            continue
        L.append(f"  Genes with TWAS data    : {s.get('n_genes_twas',0):,}")
        L.append(f"  Drugs in input list     : {s.get('n_drugs_input',0):,}")
        L.append(f"  Drugs dropped (no DDD)  : {s.get('n_drugs_ddd_dropped',0):,}")
        L.append(f"  Drugs matched to ligand : {s.get('n_drugs_matched',0):,}")
        L.append(f"  Drug x receptor pairs   : {s.get('n_pairs',0):,}")
        L.append(f"  Rows dropped (score==0) : {s.get('n_zero_score_dropped',0):,}")
        L.append(f"  Ki values imputed       : {s.get('n_ki_imputed',0):,}")
        L.append(f"  TWAS values imputed     : {s.get('n_twas_imputed',0):,}")
        L.append(f"  Strong binders          : {s.get('n_strong_binders',0):,}")
        L.append(f"  TWAS-significant rows   : {s.get('n_twas_significant',0):,}")
        L.append(f"  Receptor genes w/o TWAS : {s.get('n_missing_twas_genes',0):,}")
        if s.get("missing_twas_sample"):
            L.append(f"    e.g. {', '.join(s['missing_twas_sample'])}")
        L.append("")

        # top metabolic-risk drugs
        top_risk = s.get("top_risk")
        if top_risk is not None and len(top_risk):
            L.append("  Top metabolic-risk drugs (reference = 100):")
            L.append(f"    {'rank':>4}  {'drug':<28}{'risk':>10}{'maxTWASz':>10}")
            for _, r in top_risk.iterrows():
                L.append(
                    f"    {str(r.get('risk_rank','')):>4}  "
                    f"{str(r.get('drug',''))[:27]:<28}"
                    f"{r.get('metabolic_risk_score', float('nan')):>10.2f}"
                    f"{r.get('max_abs_twas_z', float('nan')):>10.3f}"
                )
            L.append("")

        # top strong + TWAS-significant hits
        top_hits = s.get("top_hits")
        if top_hits is not None and len(top_hits):
            L.append("  Top strong-binding + TWAS-significant hits:")
            L.append(f"    {'drug':<22}{'gene':<9}{'Ki_nM':>10}"
                     f"{'score':>12}{'twas_z':>9}{'twas_p':>11}")
            for _, r in top_hits.iterrows():
                L.append(
                    f"    {str(r.get('drug',''))[:21]:<22}"
                    f"{str(r.get('gene',''))[:8]:<9}"
                    f"{r.get('Ki_nM', float('nan')):>10.3g}"
                    f"{r.get('affinity_dose_score', float('nan')):>12.4g}"
                    f"{r.get('twas_z', float('nan')):>9.3f}"
                    f"{r.get('twas_p', float('nan')):>11.3g}"
                )
            L.append("")

    L.append(bar)
    L.append("END OF SUMMARY")
    L.append(bar)

    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"\n  wrote detailed summary -> {path}")


# ======================================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_summaries: list[dict] = []

    for twas_dir in TWAS_DIRS:
        label = _safe_label(twas_dir)
        out_dir = os.path.join(OUTPUT_DIR, label)
        print("\n" + "#" * 78)
        print(f"# PROCESSING TWAS DIR: {twas_dir}   (label='{label}')")
        print("#" * 78)

        try:
            results, stats = run_pipeline(twas_dir, out_dir)
        except Exception as e:
            print(f"  [ERROR] pipeline failed for {twas_dir}: {e}")
            run_summaries.append({
                "label": label, "twas_dir": twas_dir, "output_dir": out_dir,
                "error": str(e),
            })
            continue

        stats["label"] = label

        n_drugs_matched = stats.get("n_drugs_matched", 0)
        n_pairs = stats.get("n_pairs", 0)
        print(f"\nMatched {n_drugs_matched} drugs to ligands; "
              f"{n_pairs} drug x receptor pairs built.")
        print(f"  imputed Ki rows: {stats.get('n_ki_imputed',0)}  |  "
              f"imputed TWAS rows: {stats.get('n_twas_imputed',0)}")

        print("\nWriting outputs ...")
        write_outputs(results, out_dir)

        # === Metabolic Risk Score ===
        print("\nCalculating Metabolic Risk Score ...")
        risk_df = calculate_metabolic_risk_score(
            results,
            weights=METABOLIC_WEIGHTS,
            score_transform="log1p",            # set "none" for a plain additive sum
            twas_boost=True,                    # only applied where TWAS data exist
            twas_boost_factor=0.12,             # slightly lower because coverage may be partial
            twas_sig_bonus=0.10,                # 0.0 disables the significance bonus
            normalize=True,
            reference_drug="chlorpromazine",
            missing_gene_strategy="mean_abs_z", # fair neutral boost for missing genes
        )
        risk_path = os.path.join(out_dir, "metabolic_risk_score_per_drug.csv")
        risk_df.to_csv(risk_path, index=False)
        print(f"  wrote {risk_path}")

        print("\nN05A drugs ranked by metabolic risk (reference = 100):")
        print(risk_df.head(20).to_string(index=False))

        # capture preview tables for the final summary
        stats["top_risk"] = risk_df.head(15).copy()

        preview = results[
            results["twas_significant"] & results["affinity_dose_score"].notna()
        ].sort_values("affinity_dose_score", ascending=False).head(15)
        stats["top_hits"] = preview[
            ["drug", "gene", "Ki_nM", "affinity_dose_score", "twas_z", "twas_p"]
        ].copy() if not preview.empty else None

        if preview.empty:
            print("\n(No TWAS-significant hits found — check TWAS_DIR / column names.)")

        run_summaries.append(stats)
        print(f"\nDone with TWAS dir '{label}'.")

    # === single detailed summary across all TWAS dirs ===
    summary_txt = os.path.join(OUTPUT_DIR, "pipeline_summary.txt")
    write_detailed_summary(run_summaries, summary_txt)

    print("\nAll TWAS directories processed. Done.")


##############################################################################
# PROCESSING TWAS DIR: /content/lipids/ldl   (label='ldl')
##############################################################################
Building TWAS lookup for: /content/lipids/ldl
  Pre-loaded TWAS data for 18,057 genes from 6/6 file(s)
  Sample genes in TWAS lookup: ['PCSK9', 'SMARCA4', 'CELSR2', 'ANKDD1B', 'CETP', 'FEN1', 'ATP13A1', 'FADS1', 'SYPL2', 'YIPF2']
    HTR2C present? False
    HRH1 present? True
    DRD2 present? True
    CHRM1 present? True
    CHRM3 present? True
    HTR2A present? True
Loading Ki database ...
  Ki rows: 98,764
Loading drug list ...
  Drugs: 70

Matching drugs and building drug x receptor table ...


100%|██████████| 70/70 [00:04<00:00, 16.92it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.160, twas_p=0.239); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log/ldl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/ldl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/ldl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [fairness] missing_gene_strategy='mean_abs_z': imputed 0 missing metabolic-g

100%|██████████| 70/70 [00:03<00:00, 22.84it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.420, twas_p=0.207); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log/hdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/hdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/hdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [fairness] missing_gene_strategy='mean_abs_z': imputed 0 missing metabolic-g

...


[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.393, twas_p=0.171); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log/logtg/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/logtg/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/logtg/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [fairness] missing_gene_strategy='mean_abs_z': imputed 0 missing metab

100%|██████████| 70/70 [00:02<00:00, 23.43it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.089, twas_p=0.327); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log/nonhdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/nonhdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/nonhdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [fairness] missing_gene_strategy='mean_abs_z': imputed 0 missing me

100%|██████████| 70/70 [00:03<00:00, 21.70it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.367, twas_p=0.21); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log/tc/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/tc/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log/tc/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [fairness] missing_gene_strategy='mean_abs_z': imputed 0 missing metabolic-gene 

# Version 2 Linear per-receptor scaling

In [8]:
"""
N05A (Antipsychotics) → Ki → DDD → TWAS enrichment pipeline  (multi-TWAS)
========================================================================
TWAS INTEGRATION: VERSION 1  -- Linear per-receptor scaling.

For every TWAS directory in `TWAS_DIRS`, and for every drug in
`atc_n05a_drugs.csv`:
    1. Fuzzy-match the drug name to the "Test Ligands" column of the Ki database.
    2. Extract Ki (nM) for every receptor the drug binds.
    3. Aggregate multiple Ki measurements per drug-receptor pair (default = "min").
    4. Compute inv_Ki, DDD_mg, affinity_dose_score = inv_Ki * log(DDD_mg + 1).
    5. Map each receptor -> official gene symbol.
    6. Look up TWAS enrichment (z / p) for that gene via a per-dir lookup.
    7. Optionally impute missing Ki and/or TWAS values.
    8. Drop entries whose affinity_dose_score == 0.
    9. Write long CSV, multi-sheet XLSX, and a per-drug summary.

Metabolic Risk Score TWAS integration (Version 1):
    * TWAS influence is applied as LINEAR per-receptor scaling
      twas_scale = 1 + twas_boost_factor * |twas_z|, multiplied into each
      receptor's weighted_contribution BEFORE aggregation.
    * base_score is recomputed from the TWAS-scaled contributions.
    * A small secondary drug-level boost (0.04 * max|z|) is retained.

Dependencies:
    pip install pandas numpy rapidfuzz tqdm openpyxl
"""

import os
import re
import glob
import datetime
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

# ======================================================================
# CONFIG  --  EDIT THESE PATHS
# ======================================================================
KI_DB_PATH     = "/content/Z_database/KiDatabase_2026-06-22.csv"
DRUGS_CSV_PATH = "/content/Z_database/atc_n05a_drugs.csv"

TWAS_DIRS = [
    "/content/lipids/ldl",
    "/content/lipids/hdl",
    "/content/lipids/logtg",
    "/content/lipids/nonhdl",
    "/content/lipids/tc",
]

OUTPUT_DIR = "/content/pipeline_output/n05a_ki_twas_log_v1_linear"

FUZZY_THRESHOLD = 80
KI_AGGREGATION = "min"
STRONG_BINDER_KI_NM = 10.0
TWAS_P_SIGNIFICANT = 0.05
KI_IMPUTE_STRATEGY = "mean"
TWAS_IMPUTE_STRATEGY = "mean"
CSV_ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

TWAS_SYMBOL_COL_CANDIDATES = [
    "gene_name", "genename", "gene_symbol", "symbol", "hgnc_symbol",
]
TWAS_ENSG_COL_CANDIDATES = [
    "gene", "ensembl_gene_id", "gene_id", "geneid", "ensembl", "id",
]
TWAS_Z_COL_CANDIDATES = [
    "twas.z", "twas_z", "zscore", "z_score", "zstat", "z", "zscore",
]
TWAS_P_COL_CANDIDATES = [
    "twas.p", "twas_p", "pvalue", "p_value", "pval", "p",
]


# ======================================================================
# Receptor (Ki DB label) -> official HGNC gene symbol
# ======================================================================
RECEPTOR_TO_GENE = {
    "5-HT1A": "HTR1A", "5HT1A": "HTR1A",
    "5-HT1B": "HTR1B", "5HT1B": "HTR1B",
    "5-HT1D": "HTR1D",
    "5-HT1E": "HTR1E",
    "5-HT1F": "HTR1F",
    "5-HT2A": "HTR2A", "5HT2A": "HTR2A",
    "5-HT2B": "HTR2B",
    "5-HT2C": "HTR2C", "5HT2C": "HTR2C",
    "5-HT3":  "HTR3A",
    "5-HT5A": "HTR5A",
    "5-HT6":  "HTR6",
    "5-HT7":  "HTR7",
    "D1": "DRD1", "D2": "DRD2", "D3": "DRD3", "D4": "DRD4", "D5": "DRD5",
    "DRD1": "DRD1", "DRD2": "DRD2", "DRD3": "DRD3", "DRD4": "DRD4", "DRD5": "DRD5",
    "alpha1A": "ADRA1A", "alpha1B": "ADRA1B", "alpha1D": "ADRA1D",
    "alpha2A": "ADRA2A", "alpha2B": "ADRA2B", "alpha2C": "ADRA2C",
    "alpha1":  "ADRA1A", "alpha2": "ADRA2A",
    "beta1": "ADRB1", "beta2": "ADRB2", "beta3": "ADRB3",
    "H1": "HRH1", "H2": "HRH2", "H3": "HRH3", "H4": "HRH4",
    "M1": "CHRM1", "M2": "CHRM2", "M3": "CHRM3", "M4": "CHRM4", "M5": "CHRM5",
    "Muscarinic": "CHRM1",
    "Sigma1": "SIGMAR1", "Sigma 1": "SIGMAR1",
    "SERT": "SLC6A4", "DAT": "SLC6A3", "NET": "SLC6A2",
}
RECEPTOR_TO_GENE = {k.strip().lower(): v for k, v in RECEPTOR_TO_GENE.items()}


# ======================================================================
# METABOLIC / DIABETES RISK WEIGHTS FOR ANTIPSYCHOTICS
# ======================================================================
METABOLIC_WEIGHTS = {
    "HRH1":    1.00,
    "HTR2C":   0.92,
    "CHRM3":   0.85,
    "HTR2A":   0.55,
    "ADRA1A":  0.48,
    "ADRA1B":  0.48,
    "HTR6":    0.42,
    "ADRA2A":  0.32,
    "ADRA2B":  0.30,
    "ADRA2C":  0.30,
    "CHRM1":   0.28,
    "CHRM4":   0.22,
    "CHRM5":   0.20,
    "HTR7":    0.25,
    "DRD3":    0.18,
    "DRD2":    0.12,
    "DRD4":    0.10,
    "HTR1A":   0.08,
    "ADRB1":   0.08,
    "ADRB2":   0.07,
    "SIGMAR1": 0.05,
    "SLC6A4":  0.06,
    "SLC6A2":  0.05,
    "SLC6A3":  0.04,
}


# ======================================================================
# Helpers
# ======================================================================
def _is_nan(x) -> bool:
    if x is None:
        return True
    try:
        return bool(np.isnan(x))
    except (TypeError, ValueError):
        return False


def read_csv_robust(path: str, **kwargs) -> pd.DataFrame:
    kwargs.setdefault("low_memory", False)
    last_err = None
    for enc in CSV_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", **kwargs)
    except Exception:
        raise last_err if last_err else RuntimeError(f"Could not read {path}")


def normalise_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip().lower())


def receptor_to_gene(receptor: str):
    if not isinstance(receptor, str):
        return None
    key = receptor.strip().lower()
    if key in RECEPTOR_TO_GENE:
        return RECEPTOR_TO_GENE[key]
    key2 = re.sub(r"[\s\-]", "", key)
    for k, v in RECEPTOR_TO_GENE.items():
        if re.sub(r"[\s\-]", "", k) == key2:
            return v
    return None


def parse_ki(value) -> float:
    if value is None:
        return np.nan
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"na", "nan", "nd", "n/a", "-"}:
        return np.nan
    s = s.replace(",", "")
    s = re.sub(r"^[><=~]+", "", s)
    m = re.search(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", s)
    return float(m.group()) if m else np.nan


def ddd_to_mg(ddd, unit) -> float:
    ddd_val = parse_ki(ddd)
    if _is_nan(ddd_val):
        return np.nan
    u = (str(unit).strip().lower() if unit is not None else "")
    factors = {"g": 1000.0, "mg": 1.0, "mcg": 0.001, "µg": 0.001, "ug": 0.001}
    factor = factors.get(u, 1.0)
    return ddd_val * factor


def aggregate_ki(values: pd.Series, how: str) -> float:
    vals = values.dropna()
    if vals.empty:
        return np.nan
    if how == "min":
        return float(vals.min())
    if how == "median":
        return float(vals.median())
    if how == "mean":
        return float(vals.mean())
    return float(vals.min())


def find_col(df: pd.DataFrame, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _strip_ensg_version(s) -> str:
    return re.sub(r"\.\d+$", "", str(s).strip().upper())


def _safe_label(path: str) -> str:
    base = os.path.basename(os.path.normpath(path))
    if not base:
        base = re.sub(r"[^A-Za-z0-9_.-]+", "_", path).strip("_")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", base).strip("_") or "twas"


# ======================================================================
# Recompute affinity-derived columns
# ======================================================================
def _recompute_affinity_columns(results: pd.DataFrame) -> pd.DataFrame:
    ki_nM  = results["Ki_nM"]
    ddd_mg = results["DDD_mg"]

    has_ki = ki_nM.notna() & (ki_nM > 0)
    results["inv_Ki"] = np.where(has_ki, 1.0 / ki_nM, np.nan)

    has_inv_ki = results["inv_Ki"].notna()
    has_ddd    = ddd_mg.notna() & (ddd_mg > 0)
    log_ddd    = np.log(ddd_mg.where(has_ddd) + 1.0)
    results["affinity_dose_score"] = np.where(
        has_inv_ki & has_ddd, results["inv_Ki"] * log_ddd,
        np.where(has_inv_ki, results["inv_Ki"], np.nan),
    )

    results["strong_binder"] = ki_nM.notna() & (ki_nM < STRONG_BINDER_KI_NM)
    return results


def drop_zero_score_rows(results: pd.DataFrame):
    if results.empty or "affinity_dose_score" not in results.columns:
        return results, 0
    zero_mask = results["affinity_dose_score"] == 0
    n_dropped = int(zero_mask.sum())
    if n_dropped:
        results = results.loc[~zero_mask].reset_index(drop=True)
        print(f"  [drop-zero] removed {n_dropped} row(s) with affinity_dose_score == 0.")
    else:
        print("  [drop-zero] no rows with affinity_dose_score == 0.")
    return results, n_dropped


def impute_missing_ki(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "ki_imputed" not in results.columns:
        results["ki_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown KI_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid = results.loc[results["Ki_nM"].notna(), ["receptor", "Ki_nM"]]
    if valid.empty:
        print("  [ki-impute] no Ki values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        per_receptor = valid.groupby("receptor")["Ki_nM"].mean()
        global_ref = float(valid["Ki_nM"].mean())
    else:
        per_receptor = valid.groupby("receptor")["Ki_nM"].median()
        global_ref = float(valid["Ki_nM"].median())
    need = results["Ki_nM"].isna() & results["receptor"].notna()
    n_need = int(need.sum())
    if n_need == 0:
        print(f"  [ki-impute] strategy='{strategy}': no missing Ki values to fill.")
        return results
    filled_ref = results.loc[need, "receptor"].map(per_receptor)
    filled_ref = filled_ref.fillna(global_ref)
    results.loc[need, "Ki_nM"] = filled_ref.values
    results.loc[need, "ki_imputed"] = True
    results = _recompute_affinity_columns(results)
    print(f"  [ki-impute] strategy='{strategy}': filled {n_need} missing Ki "
          f"value(s) (global fallback Ki = {global_ref:.3g} nM).")
    return results


def impute_missing_twas(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "twas_imputed" not in results.columns:
        results["twas_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown TWAS_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid_z = results.loc[results["twas_z"].notna(), "twas_z"]
    valid_p = results.loc[results["twas_p"].notna(), "twas_p"]
    if valid_z.empty and valid_p.empty:
        print("  [twas-impute] no TWAS values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        fill_z = float(valid_z.mean()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.mean()) if not valid_p.empty else np.nan
    else:
        fill_z = float(valid_z.median()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.median()) if not valid_p.empty else np.nan
    need_z = results["gene"].notna() & results["twas_z"].isna()
    need_p = results["gene"].notna() & results["twas_p"].isna()
    n_need = int((need_z | need_p).sum())
    if n_need == 0:
        print(f"  [twas-impute] strategy='{strategy}': no missing TWAS values to fill.")
        return results
    if not _is_nan(fill_z):
        results.loc[need_z, "twas_z"] = fill_z
    if not _is_nan(fill_p):
        results.loc[need_p, "twas_p"] = fill_p
    results.loc[need_z | need_p, "twas_imputed"] = True
    print(f"  [twas-impute] strategy='{strategy}': filled {n_need} gene row(s) "
          f"(twas_z={fill_z:.3f}, twas_p={fill_p:.3g}); "
          f"imputed rows kept NON-significant.")
    return results


# ======================================================================
# TWAS lookup
# ======================================================================
def _read_twas_file(fpath: str):
    for sep in [",", "\t", None]:
        for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
            try:
                df = pd.read_csv(
                    fpath, sep=sep, encoding=enc, engine="python",
                    low_memory=False, on_bad_lines="skip"
                )
                if df is not None and not df.empty and df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    try:
        df = pd.read_csv(fpath, engine="python", on_bad_lines="skip")
        if df is not None and not df.empty:
            return df
    except Exception:
        pass
    return None


def build_twas_lookup(twas_dir: str):
    lookup = {}
    files = glob.glob(os.path.join(twas_dir, "*.*"))
    if not files:
        print(f"  [warn] No files found in {twas_dir}")
        return lookup
    successful_files = 0
    skipped_files = []
    for fpath in files:
        try:
            df = _read_twas_file(fpath)
            if df is None or df.empty:
                skipped_files.append((os.path.basename(fpath), "unreadable/empty"))
                continue
            sym_col  = find_col(df, TWAS_SYMBOL_COL_CANDIDATES)
            ensg_col = find_col(df, TWAS_ENSG_COL_CANDIDATES)
            z_col    = find_col(df, TWAS_Z_COL_CANDIDATES)
            p_col    = find_col(df, TWAS_P_COL_CANDIDATES)
            if ensg_col is not None and ensg_col == sym_col:
                ensg_col = None
            if z_col is None or (sym_col is None and ensg_col is None):
                skipped_files.append((
                    os.path.basename(fpath),
                    f"missing cols (z={z_col}, sym={sym_col}, ensg={ensg_col}); "
                    f"had {list(df.columns)[:8]}"
                ))
                continue
            n = len(df)
            sym_vals  = (df[sym_col].astype(str).str.strip().str.upper()
                         if sym_col else pd.Series([""] * n, index=df.index))
            ensg_vals = (df[ensg_col].map(_strip_ensg_version)
                         if ensg_col else pd.Series([""] * n, index=df.index))
            z_vals = df[z_col].map(parse_ki)
            p_vals = (df[p_col].map(parse_ki)
                      if p_col else pd.Series([np.nan] * n, index=df.index))
            bad = {"NAN", "NA", "NONE", ""}
            for i in df.index:
                gene = sym_vals[i]
                if gene in bad:
                    gene = ensg_vals[i]
                if gene in bad:
                    continue
                z = z_vals[i]
                p = p_vals[i]
                if _is_nan(z) and _is_nan(p):
                    continue
                prev = lookup.get(gene)
                if prev is None:
                    lookup[gene] = {"twas_z": z, "twas_p": p}
                else:
                    prev_p = prev.get("twas_p", np.nan)
                    if not _is_nan(p) and (_is_nan(prev_p) or p < prev_p):
                        lookup[gene] = {"twas_z": z, "twas_p": p}
            successful_files += 1
        except Exception as e:
            skipped_files.append((os.path.basename(fpath), f"error: {e}"))
            continue
    print(f"  Pre-loaded TWAS data for {len(lookup):,} genes "
          f"from {successful_files}/{len(files)} file(s)")
    if skipped_files:
        print(f"  [warn] {len(skipped_files)} file(s) skipped:")
        for name, reason in skipped_files[:8]:
            print(f"         - {name}: {reason}")
    return lookup


def get_twas_for_gene(gene: str, lookup):
    if gene is None or lookup is None:
        return None
    return lookup.get(str(gene).strip().upper())


# ======================================================================
# Metabolic risk score  --  VERSION 1: LINEAR per-receptor scaling
# ======================================================================
def calculate_metabolic_risk_score(
    df: pd.DataFrame,
    weights: dict = None,
    score_transform: str = "log1p",
    twas_boost: bool = True,
    twas_boost_factor: float = 0.24,     # per-receptor linear factor (recommended 0.20-0.28)
    twas_sig_bonus: float = 0.0,
    normalize: bool = True,
    reference_drug: str = "chlorpromazine",
    missing_gene_strategy: str = "mean_abs_z",
) -> pd.DataFrame:
    """
    Per-drug Metabolic Risk Score for antipsychotics.

    TWAS INTEGRATION -- VERSION 1 (Linear per-receptor scaling):
      Each receptor's weighted_contribution is multiplied by
      twas_scale = 1 + twas_boost_factor * |twas_z| BEFORE aggregation.
      base_score is then recomputed from the TWAS-scaled contributions.
      A small secondary drug-level boost (0.04 * max|z|) is retained.
    """
    if weights is None:
        weights = METABOLIC_WEIGHTS

    df = df.copy()
    mask = df["gene"].notna() & df["affinity_dose_score"].notna()

    metab_mask = mask & df["gene"].isin(weights.keys())
    n_metab = int(metab_mask.sum())
    n_metab_twas = int((metab_mask & df["twas_z"].notna()).sum())
    if twas_boost and n_metab > 0:
        cov = n_metab_twas / n_metab
        if cov < 0.5:
            print(f"  [note] TWAS coverage is low: only {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS z-score. "
                  f"Per-receptor scaling will be partial.")
        else:
            print(f"  [note] TWAS coverage: {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS z-score.")

    if score_transform == "log1p":
        affinity = np.log1p(df.loc[mask, "affinity_dose_score"].clip(lower=0))
    elif score_transform == "none":
        affinity = df.loc[mask, "affinity_dose_score"]
    else:
        raise ValueError(f"Unknown score_transform: {score_transform!r}")

    df.loc[mask, "receptor_weight"] = df.loc[mask, "gene"].map(weights).fillna(0.0)
    df.loc[mask, "weighted_contribution"] = affinity * df.loc[mask, "receptor_weight"]

    drug_scores = (
        df.groupby("drug", dropna=False)["weighted_contribution"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_contribution": "base_score"})
    )

    # ==================================================================
    # TWAS integration - Version 1: Linear per-receptor scaling
    # ==================================================================
    if twas_boost:
        df_filled = df.copy()

        # --- Fairness imputation for missing metabolic genes ---
        if missing_gene_strategy == "mean_abs_z":
            available_z = df_filled.loc[
                metab_mask & df_filled["twas_z"].notna(), "twas_z"
            ].abs()
            mean_abs_z = float(available_z.mean()) if len(available_z) > 0 else 0.0

            impute_mask = (
                mask
                & df_filled["twas_z"].isna()
                & df_filled["gene"].isin(weights.keys())
            )
            n_imputed = int(impute_mask.sum())
            df_filled.loc[impute_mask, "twas_z"] = mean_abs_z
            if n_imputed > 0:
                print(f"  [fairness] imputed {n_imputed} missing metabolic-gene "
                      f"row(s) with mean |twas_z| = {mean_abs_z:.3f}")
        elif missing_gene_strategy == "zero":
            print("  [fairness] missing_gene_strategy='zero' "
                  "(no TWAS boost for missing genes)")
        else:
            raise ValueError(
                f"Unknown missing_gene_strategy: {missing_gene_strategy!r} "
                f"(expected 'mean_abs_z' or 'zero')"
            )

        # --- Per-receptor TWAS scaling (LINEAR) ---
        twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
        df_filled.loc[mask, "twas_scale"] = 1.0 + twas_boost_factor * twas_abs

        # Apply TWAS scale directly to each receptor's contribution
        df_filled.loc[mask, "weighted_contribution"] = (
            affinity
            * df_filled.loc[mask, "receptor_weight"]
            * df_filled.loc[mask, "twas_scale"]
        )

        # Recompute base_score from the TWAS-scaled per-receptor contributions
        drug_scores = (
            df_filled.groupby("drug", dropna=False)["weighted_contribution"]
            .sum()
            .reset_index()
            .rename(columns={"weighted_contribution": "base_score"})
        )

        # Optional small *secondary* drug-level boost from the strongest hit
        twas_signal = (
            df_filled.groupby("drug")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
            .reset_index()
            .rename(columns={"twas_z": "max_abs_twas_z"})
        )
        drug_scores = drug_scores.merge(twas_signal, on="drug", how="left")
        drug_scores["max_abs_twas_z"] = drug_scores["max_abs_twas_z"].fillna(0.0)

        secondary_boost_factor = 0.04
        boost = 1 + secondary_boost_factor * drug_scores["max_abs_twas_z"]

        print(f"  [TWAS-linear] per-receptor scaling active "
              f"(twas_boost_factor={twas_boost_factor}). "
              f"Secondary drug-level boost factor = {secondary_boost_factor}")
    else:
        boost = 1.0

    if twas_sig_bonus and "twas_significant" in df.columns:
        sig = df[mask & df["twas_significant"].fillna(False)
                 & (df["receptor_weight"] > 0)]
        n_sig = (
            sig.groupby("drug").size()
            .reset_index(name="n_twas_sig_metabolic")
        )
        drug_scores = drug_scores.merge(n_sig, on="drug", how="left")
        drug_scores["n_twas_sig_metabolic"] = (
            drug_scores["n_twas_sig_metabolic"].fillna(0).astype(int)
        )
        sig_mult = 1 + twas_sig_bonus * drug_scores["n_twas_sig_metabolic"]
    else:
        sig_mult = 1.0

    drug_scores["metabolic_risk_score"] = (
        drug_scores["base_score"] * boost * sig_mult
    )

    if normalize:
        ref_mask = drug_scores["drug"].astype(str).str.lower() \
            == reference_drug.lower()
        ref_score = drug_scores.loc[ref_mask, "metabolic_risk_score"]
        if not ref_score.empty and ref_score.values[0] > 0:
            ref = ref_score.values[0]
        else:
            print(f"  [warn] reference drug '{reference_drug}' not found or "
                  f"zero-scored; normalising to the max score instead.")
            ref = drug_scores["metabolic_risk_score"].max()
        if ref and ref > 0:
            drug_scores["metabolic_risk_score"] = (
                drug_scores["metabolic_risk_score"] / ref * 100
            )
            drug_scores["risk_vs_reference"] = (
                drug_scores["metabolic_risk_score"] - 100
            )

    n_zero_drugs = int((drug_scores["metabolic_risk_score"] == 0).sum())
    if n_zero_drugs:
        drug_scores = drug_scores[drug_scores["metabolic_risk_score"] != 0]
        print(f"  [drop-zero] removed {n_zero_drugs} drug(s) with "
              f"metabolic_risk_score == 0.")

    drug_scores["risk_rank"] = (
        drug_scores["metabolic_risk_score"]
        .rank(ascending=False, method="min")
        .astype("Int64")
    )
    drug_scores = drug_scores.sort_values(
        "metabolic_risk_score", ascending=False
    ).reset_index(drop=True)

    preferred = [
        "risk_rank", "drug", "metabolic_risk_score", "base_score",
        "max_abs_twas_z", "n_twas_sig_metabolic", "risk_vs_reference",
    ]
    cols = [c for c in preferred if c in drug_scores.columns]
    return drug_scores[cols]


# ======================================================================
# Core pipeline
# ======================================================================
def run_pipeline(twas_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    stats = {"twas_dir": twas_dir, "output_dir": output_dir}

    print(f"Building TWAS lookup for: {twas_dir}")
    twas_lookup = build_twas_lookup(twas_dir)
    stats["n_genes_twas"] = len(twas_lookup)

    sample = list(twas_lookup.keys())[:10] if twas_lookup else "EMPTY"
    print(f"  Sample genes in TWAS lookup: {sample}")
    for g in ("HTR2C", "HRH1", "DRD2", "CHRM1", "CHRM3", "HTR2A"):
        print(f"    {g} present? {g in (twas_lookup or {})}")

    print("Loading Ki database ...")
    ki = read_csv_robust(KI_DB_PATH)
    print(f"  Ki rows: {len(ki):,}")

    print("Loading drug list ...")
    drugs = read_csv_robust(DRUGS_CSV_PATH)
    print(f"  Drugs: {len(drugs):,}")
    stats["n_drugs_input"] = len(drugs)

    ligand_col   = find_col(ki, ["Test Ligands", "Test Ligand", "Ligand", "ligand"])
    receptor_col = find_col(ki, ["Receptor", "receptor", "Target"])
    kival_col    = find_col(ki, ["Ki Value", "Ki", "Ki_nM", "Ki (nM)"])
    if not all([ligand_col, receptor_col, kival_col]):
        raise ValueError(
            f"Could not locate required Ki columns. Found: "
            f"ligand={ligand_col}, receptor={receptor_col}, ki={kival_col}\n"
            f"Available columns: {list(ki.columns)}"
        )

    ki["_ligand_norm"] = ki[ligand_col].map(normalise_name)
    ki["_ki_nM"]       = ki[kival_col].map(parse_ki)
    ligand_vocab = sorted(set(ki["_ligand_norm"]) - {""})

    rows = []
    missing_twas = set()

    print("\nMatching drugs and building drug x receptor table ...")
    for _, drug in tqdm(drugs.iterrows(), total=len(drugs)):
        drug_name = drug.get("name")
        norm = normalise_name(drug_name)
        ddd_mg = ddd_to_mg(drug.get("ddd"), drug.get("unit"))
        inv_ddd = (1.0 / ddd_mg) if (not _is_nan(ddd_mg) and ddd_mg > 0) else np.nan

        if not norm:
            continue

        match = process.extractOne(norm, ligand_vocab, scorer=fuzz.WRatio)
        if match is None or match[1] < FUZZY_THRESHOLD:
            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": None,
                "match_score": (match[1] if match else np.nan),
                "receptor": None, "gene": None,
                "Ki_nM": np.nan, "inv_Ki": np.nan,
                "DDD_mg": ddd_mg, "inv_DDD": inv_ddd,
                "affinity_dose_score": np.nan,
                "n_measurements": 0,
                "twas_z": np.nan, "twas_p": np.nan,
                "strong_binder": False, "twas_significant": False,
                "ki_imputed": False, "twas_imputed": False,
            })
            continue

        matched_ligand, score, _ = match
        subset = ki[ki["_ligand_norm"] == matched_ligand]

        grouped = subset.groupby(receptor_col)
        for receptor, grp in grouped:
            ki_nM = aggregate_ki(grp["_ki_nM"], KI_AGGREGATION)
            n_meas = int(grp["_ki_nM"].notna().sum())
            inv_ki = (1.0 / ki_nM) if (not _is_nan(ki_nM) and ki_nM > 0) else np.nan
            gene = receptor_to_gene(receptor)

            if not _is_nan(inv_ki) and not _is_nan(ddd_mg) and ddd_mg > 0:
                score_combined = inv_ki * np.log(ddd_mg + 1.0)
            elif not _is_nan(inv_ki):
                score_combined = inv_ki
            else:
                score_combined = np.nan

            twas = get_twas_for_gene(gene, twas_lookup) if gene else None
            if gene and twas is None:
                missing_twas.add(gene)
            twas_z = twas.get("twas_z", np.nan) if twas else np.nan
            twas_p = twas.get("twas_p", np.nan) if twas else np.nan

            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": matched_ligand,
                "match_score": round(score, 1),
                "receptor": receptor,
                "gene": gene,
                "Ki_nM": ki_nM,
                "inv_Ki": inv_ki,
                "DDD_mg": ddd_mg,
                "inv_DDD": inv_ddd,
                "affinity_dose_score": score_combined,
                "n_measurements": n_meas,
                "twas_z": twas_z,
                "twas_p": twas_p,
                "strong_binder": (not _is_nan(ki_nM)) and ki_nM < STRONG_BINDER_KI_NM,
                "twas_significant": (not _is_nan(twas_p)) and twas_p < TWAS_P_SIGNIFICANT,
                "ki_imputed": False,
                "twas_imputed": False,
            })

    results = pd.DataFrame(rows)

    stats["n_missing_twas_genes"] = len(missing_twas)
    stats["missing_twas_sample"] = sorted(missing_twas)[:12]
    if missing_twas:
        sample = sorted(missing_twas)[:8]
        print(f"\n[TWAS] {len(missing_twas)} receptor gene(s) had no TWAS data "
              f"(e.g. {', '.join(sample)}"
              f"{', ...' if len(missing_twas) > len(sample) else ''})")

    if not results.empty:
        n_before = results["drug"].nunique()
        valid_ddd_drugs = results[results["DDD_mg"].notna()]["drug"].unique()
        results = results[results["drug"].isin(valid_ddd_drugs)].reset_index(drop=True)
        n_after = results["drug"].nunique()
        stats["n_drugs_ddd_dropped"] = int(n_before - n_after)
        print(f"\n[DDD filter] kept {n_after}/{n_before} drug(s) with a valid DDD "
              f"({n_before - n_after} dropped).")

    if not results.empty:
        print("\nImputation step ...")
        results = impute_missing_ki(results, KI_IMPUTE_STRATEGY)
        results = impute_missing_twas(results, TWAS_IMPUTE_STRATEGY)

    results, n_zero_dropped = drop_zero_score_rows(results)
    stats["n_zero_score_dropped"] = n_zero_dropped

    if not results.empty:
        results["rank_in_drug"] = (
            results.groupby("drug")["affinity_dose_score"]
            .rank(ascending=False, method="min")
        )
        results = results.sort_values(
            ["drug", "affinity_dose_score"],
            ascending=[True, False]
        ).reset_index(drop=True)

    if not results.empty:
        stats["n_drugs_matched"] = int(
            results.loc[results["receptor"].notna(), "drug"].nunique()
        )
        stats["n_pairs"] = int(results["receptor"].notna().sum())
        stats["n_ki_imputed"] = int(results.get("ki_imputed",
                                     pd.Series(dtype=bool)).sum())
        stats["n_twas_imputed"] = int(results.get("twas_imputed",
                                       pd.Series(dtype=bool)).sum())
        stats["n_strong_binders"] = int(results["strong_binder"].sum())
        stats["n_twas_significant"] = int(results["twas_significant"].sum())
    else:
        stats.update({
            "n_drugs_matched": 0, "n_pairs": 0, "n_ki_imputed": 0,
            "n_twas_imputed": 0, "n_strong_binders": 0,
            "n_twas_significant": 0,
        })

    return results, stats


def write_outputs(results: pd.DataFrame, output_dir: str):
    paths = {}
    if results.empty:
        print("No results to write.")
        return paths

    csv_path = os.path.join(output_dir, "n05a_ki_twas_full_results.csv")
    results.to_csv(csv_path, index=False)
    print(f"  wrote {csv_path}")
    paths["csv"] = csv_path

    matched = results[results["receptor"].notna()]
    summary = (
        matched.groupby(["atc_code", "drug"])
        .agg(
            n_receptors      = ("receptor", "nunique"),
            n_strong_binders = ("strong_binder", "sum"),
            n_twas_sig       = ("twas_significant", "sum"),
            n_ki_imputed     = ("ki_imputed", "sum"),
            n_twas_imputed   = ("twas_imputed", "sum"),
            best_Ki_nM       = ("Ki_nM", "min"),
            top_score        = ("affinity_dose_score", "max"),
            DDD_mg           = ("DDD_mg", "first"),
        )
        .reset_index()
        .sort_values("top_score", ascending=False)
    )
    summary_path = os.path.join(output_dir, "drug_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"  wrote {summary_path}")
    paths["summary"] = summary_path

    xlsx_path = os.path.join(output_dir, "n05a_ki_twas_full_results.xlsx")
    top5 = (
        matched[matched["rank_in_drug"] <= 5]
        .sort_values(["drug", "rank_in_drug"])
    )
    twas_sig = matched[matched["twas_significant"]].sort_values(
        "affinity_dose_score", ascending=False
    )
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
            results.to_excel(xw, sheet_name="All", index=False)
            top5.to_excel(xw, sheet_name="Top5_per_drug", index=False)
            twas_sig.to_excel(xw, sheet_name="TWAS_significant", index=False)
            summary.to_excel(xw, sheet_name="Drug_summary", index=False)
        print(f"  wrote {xlsx_path}")
        paths["xlsx"] = xlsx_path
    except Exception as e:
        print(f"  (skipped Excel export: {e})")

    return paths


def write_detailed_summary(run_summaries, path: str) -> None:
    L = []
    bar = "=" * 78
    sub = "-" * 78
    L.append(bar)
    L.append("N05A  Ki -> DDD -> TWAS ENRICHMENT PIPELINE — DETAILED SUMMARY")
    L.append("TWAS INTEGRATION: VERSION 1 (Linear per-receptor scaling)")
    L.append(bar)
    L.append(f"Generated            : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
    L.append(f"Ki database          : {KI_DB_PATH}")
    L.append(f"Drug list            : {DRUGS_CSV_PATH}")
    L.append(f"Output root          : {OUTPUT_DIR}")
    L.append(f"Score formula        : affinity_dose_score = inv_Ki * log(DDD_mg + 1)")
    L.append(f"TWAS dirs processed  : {len(run_summaries)}")
    L.append("")
    L.append("Configuration:")
    L.append(f"  FUZZY_THRESHOLD       = {FUZZY_THRESHOLD}")
    L.append(f"  KI_AGGREGATION        = {KI_AGGREGATION}")
    L.append(f"  STRONG_BINDER_KI_NM   = {STRONG_BINDER_KI_NM}")
    L.append(f"  TWAS_P_SIGNIFICANT    = {TWAS_P_SIGNIFICANT}")
    L.append(f"  KI_IMPUTE_STRATEGY    = {KI_IMPUTE_STRATEGY}")
    L.append(f"  TWAS_IMPUTE_STRATEGY  = {TWAS_IMPUTE_STRATEGY}")
    L.append("")
    L.append(bar)
    L.append("CROSS-RUN OVERVIEW")
    L.append(bar)
    header = (f"{'TWAS label':<18}{'genes':>8}{'matched':>9}{'pairs':>8}"
              f"{'zero-drop':>11}{'ki_imp':>8}{'tw_imp':>8}{'tw_sig':>8}")
    L.append(header)
    L.append(sub)
    for s in run_summaries:
        if s.get("error"):
            L.append(f"{s.get('label',''):<18}  ERROR: {s['error']}")
            continue
        L.append(
            f"{s.get('label',''):<18}"
            f"{s.get('n_genes_twas',0):>8}"
            f"{s.get('n_drugs_matched',0):>9}"
            f"{s.get('n_pairs',0):>8}"
            f"{s.get('n_zero_score_dropped',0):>11}"
            f"{s.get('n_ki_imputed',0):>8}"
            f"{s.get('n_twas_imputed',0):>8}"
            f"{s.get('n_twas_significant',0):>8}"
        )
    L.append("")
    for s in run_summaries:
        L.append(bar)
        L.append(f"TWAS RUN: {s.get('label','')}")
        L.append(bar)
        L.append(f"  TWAS directory          : {s.get('twas_dir','')}")
        L.append(f"  Output directory        : {s.get('output_dir','')}")
        if s.get("error"):
            L.append(f"  [ERROR] {s['error']}")
            L.append("")
            continue
        L.append(f"  Genes with TWAS data    : {s.get('n_genes_twas',0):,}")
        L.append(f"  Drugs in input list     : {s.get('n_drugs_input',0):,}")
        L.append(f"  Drugs dropped (no DDD)  : {s.get('n_drugs_ddd_dropped',0):,}")
        L.append(f"  Drugs matched to ligand : {s.get('n_drugs_matched',0):,}")
        L.append(f"  Drug x receptor pairs   : {s.get('n_pairs',0):,}")
        L.append(f"  Rows dropped (score==0) : {s.get('n_zero_score_dropped',0):,}")
        L.append(f"  Ki values imputed       : {s.get('n_ki_imputed',0):,}")
        L.append(f"  TWAS values imputed     : {s.get('n_twas_imputed',0):,}")
        L.append(f"  Strong binders          : {s.get('n_strong_binders',0):,}")
        L.append(f"  TWAS-significant rows   : {s.get('n_twas_significant',0):,}")
        L.append(f"  Receptor genes w/o TWAS : {s.get('n_missing_twas_genes',0):,}")
        if s.get("missing_twas_sample"):
            L.append(f"    e.g. {', '.join(s['missing_twas_sample'])}")
        L.append("")
        top_risk = s.get("top_risk")
        if top_risk is not None and len(top_risk):
            L.append("  Top metabolic-risk drugs (reference = 100):")
            L.append(f"    {'rank':>4}  {'drug':<28}{'risk':>10}{'maxTWASz':>10}")
            for _, r in top_risk.iterrows():
                L.append(
                    f"    {str(r.get('risk_rank','')):>4}  "
                    f"{str(r.get('drug',''))[:27]:<28}"
                    f"{r.get('metabolic_risk_score', float('nan')):>10.2f}"
                    f"{r.get('max_abs_twas_z', float('nan')):>10.3f}"
                )
            L.append("")
        top_hits = s.get("top_hits")
        if top_hits is not None and len(top_hits):
            L.append("  Top strong-binding + TWAS-significant hits:")
            L.append(f"    {'drug':<22}{'gene':<9}{'Ki_nM':>10}"
                     f"{'score':>12}{'twas_z':>9}{'twas_p':>11}")
            for _, r in top_hits.iterrows():
                L.append(
                    f"    {str(r.get('drug',''))[:21]:<22}"
                    f"{str(r.get('gene',''))[:8]:<9}"
                    f"{r.get('Ki_nM', float('nan')):>10.3g}"
                    f"{r.get('affinity_dose_score', float('nan')):>12.4g}"
                    f"{r.get('twas_z', float('nan')):>9.3f}"
                    f"{r.get('twas_p', float('nan')):>11.3g}"
                )
            L.append("")
    L.append(bar)
    L.append("END OF SUMMARY")
    L.append(bar)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"\n  wrote detailed summary -> {path}")


# ======================================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_summaries = []

    for twas_dir in TWAS_DIRS:
        label = _safe_label(twas_dir)
        out_dir = os.path.join(OUTPUT_DIR, label)
        print("\n" + "#" * 78)
        print(f"# PROCESSING TWAS DIR: {twas_dir}   (label='{label}')")
        print("# TWAS integration = VERSION 1 (linear per-receptor scaling)")
        print("#" * 78)

        try:
            results, stats = run_pipeline(twas_dir, out_dir)
        except Exception as e:
            print(f"  [ERROR] pipeline failed for {twas_dir}: {e}")
            run_summaries.append({
                "label": label, "twas_dir": twas_dir, "output_dir": out_dir,
                "error": str(e),
            })
            continue

        stats["label"] = label
        n_drugs_matched = stats.get("n_drugs_matched", 0)
        n_pairs = stats.get("n_pairs", 0)
        print(f"\nMatched {n_drugs_matched} drugs to ligands; "
              f"{n_pairs} drug x receptor pairs built.")
        print(f"  imputed Ki rows: {stats.get('n_ki_imputed',0)}  |  "
              f"imputed TWAS rows: {stats.get('n_twas_imputed',0)}")

        print("\nWriting outputs ...")
        write_outputs(results, out_dir)

        print("\nCalculating Metabolic Risk Score (Version 1: linear) ...")
        risk_df = calculate_metabolic_risk_score(
            results,
            weights=METABOLIC_WEIGHTS,
            score_transform="log1p",
            twas_boost=True,
            twas_boost_factor=0.24,             # per-receptor linear factor
            twas_sig_bonus=0.10,
            normalize=True,
            reference_drug="chlorpromazine",
            missing_gene_strategy="mean_abs_z",
        )
        risk_path = os.path.join(out_dir, "metabolic_risk_score_per_drug.csv")
        risk_df.to_csv(risk_path, index=False)
        print(f"  wrote {risk_path}")

        print("\nN05A drugs ranked by metabolic risk (reference = 100):")
        print(risk_df.head(20).to_string(index=False))

        stats["top_risk"] = risk_df.head(15).copy()
        preview = results[
            results["twas_significant"] & results["affinity_dose_score"].notna()
        ].sort_values("affinity_dose_score", ascending=False).head(15)
        stats["top_hits"] = preview[
            ["drug", "gene", "Ki_nM", "affinity_dose_score", "twas_z", "twas_p"]
        ].copy() if not preview.empty else None
        if preview.empty:
            print("\n(No TWAS-significant hits found — check TWAS_DIR / column names.)")

        run_summaries.append(stats)
        print(f"\nDone with TWAS dir '{label}'.")

    summary_txt = os.path.join(OUTPUT_DIR, "pipeline_summary.txt")
    write_detailed_summary(run_summaries, summary_txt)
    print("\nAll TWAS directories processed (Version 1 - linear). Done.")


##############################################################################
# PROCESSING TWAS DIR: /content/lipids/ldl   (label='ldl')
# TWAS integration = VERSION 1 (linear per-receptor scaling)
##############################################################################
Building TWAS lookup for: /content/lipids/ldl
  Pre-loaded TWAS data for 18,057 genes from 6/6 file(s)
  Sample genes in TWAS lookup: ['PCSK9', 'SMARCA4', 'CELSR2', 'ANKDD1B', 'CETP', 'FEN1', 'ATP13A1', 'FADS1', 'SYPL2', 'YIPF2']
    HTR2C present? False
    HRH1 present? True
    DRD2 present? True
    CHRM1 present? True
    CHRM3 present? True
    HTR2A present? True
Loading Ki database ...
  Ki rows: 98,764
Loading drug list ...
  Drugs: 70

Matching drugs and building drug x receptor table ...


100%|██████████| 70/70 [00:03<00:00, 23.04it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.160, twas_p=0.239); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/ldl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/ldl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/ldl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 1: linear) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-linear] per-receptor

100%|██████████| 70/70 [00:03<00:00, 22.89it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.420, twas_p=0.207); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/hdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/hdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/hdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 1: linear) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-linear] per-receptor

...


[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.393, twas_p=0.171); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/logtg/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/logtg/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/logtg/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 1: linear) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-linear] per-re

100%|██████████| 70/70 [00:03<00:00, 23.20it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.089, twas_p=0.327); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/nonhdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/nonhdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/nonhdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 1: linear) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-linear] per

100%|██████████| 70/70 [00:03<00:00, 19.72it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.367, twas_p=0.21); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/tc/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/tc/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v1_linear/tc/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 1: linear) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-linear] per-receptor sca

# Version 3 Mild exponential per-receptor scaling

In [9]:
"""
N05A (Antipsychotics) → Ki → DDD → TWAS enrichment pipeline  (multi-TWAS)
========================================================================
TWAS INTEGRATION: VERSION 2  -- Mild exponential per-receptor scaling.

Same pipeline as Version 1, except the Metabolic Risk Score applies
per-receptor scaling as:
    twas_scale = exp(beta * |twas_z|)          (beta ~ 0.06 - 0.10)
This accelerates the influence of high-|z| TWAS hits while remaining
stable. base_score is recomputed from the TWAS-scaled contributions.
A very small (or zero) secondary drug-level boost is retained.

Dependencies:
    pip install pandas numpy rapidfuzz tqdm openpyxl
"""

import os
import re
import glob
import datetime
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

# ======================================================================
# CONFIG  --  EDIT THESE PATHS
# ======================================================================
KI_DB_PATH     = "/content/Z_database/KiDatabase_2026-06-22.csv"
DRUGS_CSV_PATH = "/content/Z_database/atc_n05a_drugs.csv"

TWAS_DIRS = [
    "/content/lipids/ldl",
    "/content/lipids/hdl",
    "/content/lipids/logtg",
    "/content/lipids/nonhdl",
    "/content/lipids/tc",
]

OUTPUT_DIR = "/content/pipeline_output/n05a_ki_twas_log_v2_expo"

FUZZY_THRESHOLD = 80
KI_AGGREGATION = "min"
STRONG_BINDER_KI_NM = 10.0
TWAS_P_SIGNIFICANT = 0.05
KI_IMPUTE_STRATEGY = "mean"
TWAS_IMPUTE_STRATEGY = "mean"
CSV_ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

TWAS_SYMBOL_COL_CANDIDATES = [
    "gene_name", "genename", "gene_symbol", "symbol", "hgnc_symbol",
]
TWAS_ENSG_COL_CANDIDATES = [
    "gene", "ensembl_gene_id", "gene_id", "geneid", "ensembl", "id",
]
TWAS_Z_COL_CANDIDATES = [
    "twas.z", "twas_z", "zscore", "z_score", "zstat", "z", "zscore",
]
TWAS_P_COL_CANDIDATES = [
    "twas.p", "twas_p", "pvalue", "p_value", "pval", "p",
]


# ======================================================================
# Receptor (Ki DB label) -> official HGNC gene symbol
# ======================================================================
RECEPTOR_TO_GENE = {
    "5-HT1A": "HTR1A", "5HT1A": "HTR1A",
    "5-HT1B": "HTR1B", "5HT1B": "HTR1B",
    "5-HT1D": "HTR1D",
    "5-HT1E": "HTR1E",
    "5-HT1F": "HTR1F",
    "5-HT2A": "HTR2A", "5HT2A": "HTR2A",
    "5-HT2B": "HTR2B",
    "5-HT2C": "HTR2C", "5HT2C": "HTR2C",
    "5-HT3":  "HTR3A",
    "5-HT5A": "HTR5A",
    "5-HT6":  "HTR6",
    "5-HT7":  "HTR7",
    "D1": "DRD1", "D2": "DRD2", "D3": "DRD3", "D4": "DRD4", "D5": "DRD5",
    "DRD1": "DRD1", "DRD2": "DRD2", "DRD3": "DRD3", "DRD4": "DRD4", "DRD5": "DRD5",
    "alpha1A": "ADRA1A", "alpha1B": "ADRA1B", "alpha1D": "ADRA1D",
    "alpha2A": "ADRA2A", "alpha2B": "ADRA2B", "alpha2C": "ADRA2C",
    "alpha1":  "ADRA1A", "alpha2": "ADRA2A",
    "beta1": "ADRB1", "beta2": "ADRB2", "beta3": "ADRB3",
    "H1": "HRH1", "H2": "HRH2", "H3": "HRH3", "H4": "HRH4",
    "M1": "CHRM1", "M2": "CHRM2", "M3": "CHRM3", "M4": "CHRM4", "M5": "CHRM5",
    "Muscarinic": "CHRM1",
    "Sigma1": "SIGMAR1", "Sigma 1": "SIGMAR1",
    "SERT": "SLC6A4", "DAT": "SLC6A3", "NET": "SLC6A2",
}
RECEPTOR_TO_GENE = {k.strip().lower(): v for k, v in RECEPTOR_TO_GENE.items()}


# ======================================================================
# METABOLIC / DIABETES RISK WEIGHTS FOR ANTIPSYCHOTICS
# ======================================================================
METABOLIC_WEIGHTS = {
    "HRH1":    1.00,
    "HTR2C":   0.92,
    "CHRM3":   0.85,
    "HTR2A":   0.55,
    "ADRA1A":  0.48,
    "ADRA1B":  0.48,
    "HTR6":    0.42,
    "ADRA2A":  0.32,
    "ADRA2B":  0.30,
    "ADRA2C":  0.30,
    "CHRM1":   0.28,
    "CHRM4":   0.22,
    "CHRM5":   0.20,
    "HTR7":    0.25,
    "DRD3":    0.18,
    "DRD2":    0.12,
    "DRD4":    0.10,
    "HTR1A":   0.08,
    "ADRB1":   0.08,
    "ADRB2":   0.07,
    "SIGMAR1": 0.05,
    "SLC6A4":  0.06,
    "SLC6A2":  0.05,
    "SLC6A3":  0.04,
}


# ======================================================================
# Helpers
# ======================================================================
def _is_nan(x) -> bool:
    if x is None:
        return True
    try:
        return bool(np.isnan(x))
    except (TypeError, ValueError):
        return False


def read_csv_robust(path: str, **kwargs) -> pd.DataFrame:
    kwargs.setdefault("low_memory", False)
    last_err = None
    for enc in CSV_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", **kwargs)
    except Exception:
        raise last_err if last_err else RuntimeError(f"Could not read {path}")


def normalise_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip().lower())


def receptor_to_gene(receptor: str):
    if not isinstance(receptor, str):
        return None
    key = receptor.strip().lower()
    if key in RECEPTOR_TO_GENE:
        return RECEPTOR_TO_GENE[key]
    key2 = re.sub(r"[\s\-]", "", key)
    for k, v in RECEPTOR_TO_GENE.items():
        if re.sub(r"[\s\-]", "", k) == key2:
            return v
    return None


def parse_ki(value) -> float:
    if value is None:
        return np.nan
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"na", "nan", "nd", "n/a", "-"}:
        return np.nan
    s = s.replace(",", "")
    s = re.sub(r"^[><=~]+", "", s)
    m = re.search(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", s)
    return float(m.group()) if m else np.nan


def ddd_to_mg(ddd, unit) -> float:
    ddd_val = parse_ki(ddd)
    if _is_nan(ddd_val):
        return np.nan
    u = (str(unit).strip().lower() if unit is not None else "")
    factors = {"g": 1000.0, "mg": 1.0, "mcg": 0.001, "µg": 0.001, "ug": 0.001}
    factor = factors.get(u, 1.0)
    return ddd_val * factor


def aggregate_ki(values: pd.Series, how: str) -> float:
    vals = values.dropna()
    if vals.empty:
        return np.nan
    if how == "min":
        return float(vals.min())
    if how == "median":
        return float(vals.median())
    if how == "mean":
        return float(vals.mean())
    return float(vals.min())


def find_col(df: pd.DataFrame, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _strip_ensg_version(s) -> str:
    return re.sub(r"\.\d+$", "", str(s).strip().upper())


def _safe_label(path: str) -> str:
    base = os.path.basename(os.path.normpath(path))
    if not base:
        base = re.sub(r"[^A-Za-z0-9_.-]+", "_", path).strip("_")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", base).strip("_") or "twas"


# ======================================================================
# Recompute affinity-derived columns
# ======================================================================
def _recompute_affinity_columns(results: pd.DataFrame) -> pd.DataFrame:
    ki_nM  = results["Ki_nM"]
    ddd_mg = results["DDD_mg"]

    has_ki = ki_nM.notna() & (ki_nM > 0)
    results["inv_Ki"] = np.where(has_ki, 1.0 / ki_nM, np.nan)

    has_inv_ki = results["inv_Ki"].notna()
    has_ddd    = ddd_mg.notna() & (ddd_mg > 0)
    log_ddd    = np.log(ddd_mg.where(has_ddd) + 1.0)
    results["affinity_dose_score"] = np.where(
        has_inv_ki & has_ddd, results["inv_Ki"] * log_ddd,
        np.where(has_inv_ki, results["inv_Ki"], np.nan),
    )

    results["strong_binder"] = ki_nM.notna() & (ki_nM < STRONG_BINDER_KI_NM)
    return results


def drop_zero_score_rows(results: pd.DataFrame):
    if results.empty or "affinity_dose_score" not in results.columns:
        return results, 0
    zero_mask = results["affinity_dose_score"] == 0
    n_dropped = int(zero_mask.sum())
    if n_dropped:
        results = results.loc[~zero_mask].reset_index(drop=True)
        print(f"  [drop-zero] removed {n_dropped} row(s) with affinity_dose_score == 0.")
    else:
        print("  [drop-zero] no rows with affinity_dose_score == 0.")
    return results, n_dropped


def impute_missing_ki(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "ki_imputed" not in results.columns:
        results["ki_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown KI_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid = results.loc[results["Ki_nM"].notna(), ["receptor", "Ki_nM"]]
    if valid.empty:
        print("  [ki-impute] no Ki values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        per_receptor = valid.groupby("receptor")["Ki_nM"].mean()
        global_ref = float(valid["Ki_nM"].mean())
    else:
        per_receptor = valid.groupby("receptor")["Ki_nM"].median()
        global_ref = float(valid["Ki_nM"].median())
    need = results["Ki_nM"].isna() & results["receptor"].notna()
    n_need = int(need.sum())
    if n_need == 0:
        print(f"  [ki-impute] strategy='{strategy}': no missing Ki values to fill.")
        return results
    filled_ref = results.loc[need, "receptor"].map(per_receptor)
    filled_ref = filled_ref.fillna(global_ref)
    results.loc[need, "Ki_nM"] = filled_ref.values
    results.loc[need, "ki_imputed"] = True
    results = _recompute_affinity_columns(results)
    print(f"  [ki-impute] strategy='{strategy}': filled {n_need} missing Ki "
          f"value(s) (global fallback Ki = {global_ref:.3g} nM).")
    return results


def impute_missing_twas(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "twas_imputed" not in results.columns:
        results["twas_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown TWAS_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid_z = results.loc[results["twas_z"].notna(), "twas_z"]
    valid_p = results.loc[results["twas_p"].notna(), "twas_p"]
    if valid_z.empty and valid_p.empty:
        print("  [twas-impute] no TWAS values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        fill_z = float(valid_z.mean()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.mean()) if not valid_p.empty else np.nan
    else:
        fill_z = float(valid_z.median()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.median()) if not valid_p.empty else np.nan
    need_z = results["gene"].notna() & results["twas_z"].isna()
    need_p = results["gene"].notna() & results["twas_p"].isna()
    n_need = int((need_z | need_p).sum())
    if n_need == 0:
        print(f"  [twas-impute] strategy='{strategy}': no missing TWAS values to fill.")
        return results
    if not _is_nan(fill_z):
        results.loc[need_z, "twas_z"] = fill_z
    if not _is_nan(fill_p):
        results.loc[need_p, "twas_p"] = fill_p
    results.loc[need_z | need_p, "twas_imputed"] = True
    print(f"  [twas-impute] strategy='{strategy}': filled {n_need} gene row(s) "
          f"(twas_z={fill_z:.3f}, twas_p={fill_p:.3g}); "
          f"imputed rows kept NON-significant.")
    return results


# ======================================================================
# TWAS lookup
# ======================================================================
def _read_twas_file(fpath: str):
    for sep in [",", "\t", None]:
        for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
            try:
                df = pd.read_csv(
                    fpath, sep=sep, encoding=enc, engine="python",
                    low_memory=False, on_bad_lines="skip"
                )
                if df is not None and not df.empty and df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    try:
        df = pd.read_csv(fpath, engine="python", on_bad_lines="skip")
        if df is not None and not df.empty:
            return df
    except Exception:
        pass
    return None


def build_twas_lookup(twas_dir: str):
    lookup = {}
    files = glob.glob(os.path.join(twas_dir, "*.*"))
    if not files:
        print(f"  [warn] No files found in {twas_dir}")
        return lookup
    successful_files = 0
    skipped_files = []
    for fpath in files:
        try:
            df = _read_twas_file(fpath)
            if df is None or df.empty:
                skipped_files.append((os.path.basename(fpath), "unreadable/empty"))
                continue
            sym_col  = find_col(df, TWAS_SYMBOL_COL_CANDIDATES)
            ensg_col = find_col(df, TWAS_ENSG_COL_CANDIDATES)
            z_col    = find_col(df, TWAS_Z_COL_CANDIDATES)
            p_col    = find_col(df, TWAS_P_COL_CANDIDATES)
            if ensg_col is not None and ensg_col == sym_col:
                ensg_col = None
            if z_col is None or (sym_col is None and ensg_col is None):
                skipped_files.append((
                    os.path.basename(fpath),
                    f"missing cols (z={z_col}, sym={sym_col}, ensg={ensg_col}); "
                    f"had {list(df.columns)[:8]}"
                ))
                continue
            n = len(df)
            sym_vals  = (df[sym_col].astype(str).str.strip().str.upper()
                         if sym_col else pd.Series([""] * n, index=df.index))
            ensg_vals = (df[ensg_col].map(_strip_ensg_version)
                         if ensg_col else pd.Series([""] * n, index=df.index))
            z_vals = df[z_col].map(parse_ki)
            p_vals = (df[p_col].map(parse_ki)
                      if p_col else pd.Series([np.nan] * n, index=df.index))
            bad = {"NAN", "NA", "NONE", ""}
            for i in df.index:
                gene = sym_vals[i]
                if gene in bad:
                    gene = ensg_vals[i]
                if gene in bad:
                    continue
                z = z_vals[i]
                p = p_vals[i]
                if _is_nan(z) and _is_nan(p):
                    continue
                prev = lookup.get(gene)
                if prev is None:
                    lookup[gene] = {"twas_z": z, "twas_p": p}
                else:
                    prev_p = prev.get("twas_p", np.nan)
                    if not _is_nan(p) and (_is_nan(prev_p) or p < prev_p):
                        lookup[gene] = {"twas_z": z, "twas_p": p}
            successful_files += 1
        except Exception as e:
            skipped_files.append((os.path.basename(fpath), f"error: {e}"))
            continue
    print(f"  Pre-loaded TWAS data for {len(lookup):,} genes "
          f"from {successful_files}/{len(files)} file(s)")
    if skipped_files:
        print(f"  [warn] {len(skipped_files)} file(s) skipped:")
        for name, reason in skipped_files[:8]:
            print(f"         - {name}: {reason}")
    return lookup


def get_twas_for_gene(gene: str, lookup):
    if gene is None or lookup is None:
        return None
    return lookup.get(str(gene).strip().upper())


# ======================================================================
# Metabolic risk score  --  VERSION 2: MILD EXPONENTIAL per-receptor scaling
# ======================================================================
def calculate_metabolic_risk_score(
    df: pd.DataFrame,
    weights: dict = None,
    score_transform: str = "log1p",
    twas_boost: bool = True,
    twas_boost_factor: float = 0.24,     # unused by v2 scaling (kept for signature compat)
    twas_sig_bonus: float = 0.0,
    normalize: bool = True,
    reference_drug: str = "chlorpromazine",
    missing_gene_strategy: str = "mean_abs_z",
    beta: float = 0.08,                  # exponential rate (tune 0.06 - 0.10)
) -> pd.DataFrame:
    """
    Per-drug Metabolic Risk Score for antipsychotics.

    TWAS INTEGRATION -- VERSION 2 (Mild exponential per-receptor scaling):
      Each receptor's weighted_contribution is multiplied by
      twas_scale = exp(beta * |twas_z|) BEFORE aggregation, accelerating
      the influence of strong TWAS hits. base_score is then recomputed
      from the TWAS-scaled contributions. A very small (or zero) secondary
      drug-level boost is retained.
    """
    if weights is None:
        weights = METABOLIC_WEIGHTS

    df = df.copy()
    mask = df["gene"].notna() & df["affinity_dose_score"].notna()

    metab_mask = mask & df["gene"].isin(weights.keys())
    n_metab = int(metab_mask.sum())
    n_metab_twas = int((metab_mask & df["twas_z"].notna()).sum())
    if twas_boost and n_metab > 0:
        cov = n_metab_twas / n_metab
        if cov < 0.5:
            print(f"  [note] TWAS coverage is low: only {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS z-score. "
                  f"Per-receptor scaling will be partial.")
        else:
            print(f"  [note] TWAS coverage: {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS z-score.")

    if score_transform == "log1p":
        affinity = np.log1p(df.loc[mask, "affinity_dose_score"].clip(lower=0))
    elif score_transform == "none":
        affinity = df.loc[mask, "affinity_dose_score"]
    else:
        raise ValueError(f"Unknown score_transform: {score_transform!r}")

    df.loc[mask, "receptor_weight"] = df.loc[mask, "gene"].map(weights).fillna(0.0)
    df.loc[mask, "weighted_contribution"] = affinity * df.loc[mask, "receptor_weight"]

    drug_scores = (
        df.groupby("drug", dropna=False)["weighted_contribution"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_contribution": "base_score"})
    )

    # ==================================================================
    # TWAS integration - Version 2: Mild exponential per-receptor scaling
    # ==================================================================
    if twas_boost:
        df_filled = df.copy()

        # --- Fairness imputation for missing metabolic genes ---
        if missing_gene_strategy == "mean_abs_z":
            available_z = df_filled.loc[
                metab_mask & df_filled["twas_z"].notna(), "twas_z"
            ].abs()
            mean_abs_z = float(available_z.mean()) if len(available_z) > 0 else 0.0

            impute_mask = (
                mask
                & df_filled["twas_z"].isna()
                & df_filled["gene"].isin(weights.keys())
            )
            n_imputed = int(impute_mask.sum())
            df_filled.loc[impute_mask, "twas_z"] = mean_abs_z
            if n_imputed > 0:
                print(f"  [fairness] imputed {n_imputed} missing metabolic-gene "
                      f"row(s) with mean |twas_z| = {mean_abs_z:.3f}")
        elif missing_gene_strategy == "zero":
            print("  [fairness] missing_gene_strategy='zero' "
                  "(no TWAS boost for missing genes)")
        else:
            raise ValueError(
                f"Unknown missing_gene_strategy: {missing_gene_strategy!r} "
                f"(expected 'mean_abs_z' or 'zero')"
            )

        # --- Per-receptor TWAS scaling (MILD EXPONENTIAL) ---
        twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
        df_filled.loc[mask, "twas_scale"] = np.exp(beta * twas_abs)

        df_filled.loc[mask, "weighted_contribution"] = (
            affinity
            * df_filled.loc[mask, "receptor_weight"]
            * df_filled.loc[mask, "twas_scale"]
        )

        # Recompute base_score from the TWAS-scaled per-receptor contributions
        drug_scores = (
            df_filled.groupby("drug", dropna=False)["weighted_contribution"]
            .sum()
            .reset_index()
            .rename(columns={"weighted_contribution": "base_score"})
        )

        # Optional tiny secondary drug-level boost (can even set to 0.0)
        twas_signal = (
            df_filled.groupby("drug")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
            .reset_index()
            .rename(columns={"twas_z": "max_abs_twas_z"})
        )
        drug_scores = drug_scores.merge(twas_signal, on="drug", how="left")
        drug_scores["max_abs_twas_z"] = drug_scores["max_abs_twas_z"].fillna(0.0)

        secondary_boost_factor = 0.02
        boost = 1 + secondary_boost_factor * drug_scores["max_abs_twas_z"]

        print(f"  [TWAS-expo] mild exponential per-receptor scaling active "
              f"(beta={beta}). Secondary boost factor = {secondary_boost_factor}")
    else:
        boost = 1.0

    if twas_sig_bonus and "twas_significant" in df.columns:
        sig = df[mask & df["twas_significant"].fillna(False)
                 & (df["receptor_weight"] > 0)]
        n_sig = (
            sig.groupby("drug").size()
            .reset_index(name="n_twas_sig_metabolic")
        )
        drug_scores = drug_scores.merge(n_sig, on="drug", how="left")
        drug_scores["n_twas_sig_metabolic"] = (
            drug_scores["n_twas_sig_metabolic"].fillna(0).astype(int)
        )
        sig_mult = 1 + twas_sig_bonus * drug_scores["n_twas_sig_metabolic"]
    else:
        sig_mult = 1.0

    drug_scores["metabolic_risk_score"] = (
        drug_scores["base_score"] * boost * sig_mult
    )

    if normalize:
        ref_mask = drug_scores["drug"].astype(str).str.lower() \
            == reference_drug.lower()
        ref_score = drug_scores.loc[ref_mask, "metabolic_risk_score"]
        if not ref_score.empty and ref_score.values[0] > 0:
            ref = ref_score.values[0]
        else:
            print(f"  [warn] reference drug '{reference_drug}' not found or "
                  f"zero-scored; normalising to the max score instead.")
            ref = drug_scores["metabolic_risk_score"].max()
        if ref and ref > 0:
            drug_scores["metabolic_risk_score"] = (
                drug_scores["metabolic_risk_score"] / ref * 100
            )
            drug_scores["risk_vs_reference"] = (
                drug_scores["metabolic_risk_score"] - 100
            )

    n_zero_drugs = int((drug_scores["metabolic_risk_score"] == 0).sum())
    if n_zero_drugs:
        drug_scores = drug_scores[drug_scores["metabolic_risk_score"] != 0]
        print(f"  [drop-zero] removed {n_zero_drugs} drug(s) with "
              f"metabolic_risk_score == 0.")

    drug_scores["risk_rank"] = (
        drug_scores["metabolic_risk_score"]
        .rank(ascending=False, method="min")
        .astype("Int64")
    )
    drug_scores = drug_scores.sort_values(
        "metabolic_risk_score", ascending=False
    ).reset_index(drop=True)

    preferred = [
        "risk_rank", "drug", "metabolic_risk_score", "base_score",
        "max_abs_twas_z", "n_twas_sig_metabolic", "risk_vs_reference",
    ]
    cols = [c for c in preferred if c in drug_scores.columns]
    return drug_scores[cols]


# ======================================================================
# Core pipeline
# ======================================================================
def run_pipeline(twas_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    stats = {"twas_dir": twas_dir, "output_dir": output_dir}

    print(f"Building TWAS lookup for: {twas_dir}")
    twas_lookup = build_twas_lookup(twas_dir)
    stats["n_genes_twas"] = len(twas_lookup)

    sample = list(twas_lookup.keys())[:10] if twas_lookup else "EMPTY"
    print(f"  Sample genes in TWAS lookup: {sample}")
    for g in ("HTR2C", "HRH1", "DRD2", "CHRM1", "CHRM3", "HTR2A"):
        print(f"    {g} present? {g in (twas_lookup or {})}")

    print("Loading Ki database ...")
    ki = read_csv_robust(KI_DB_PATH)
    print(f"  Ki rows: {len(ki):,}")

    print("Loading drug list ...")
    drugs = read_csv_robust(DRUGS_CSV_PATH)
    print(f"  Drugs: {len(drugs):,}")
    stats["n_drugs_input"] = len(drugs)

    ligand_col   = find_col(ki, ["Test Ligands", "Test Ligand", "Ligand", "ligand"])
    receptor_col = find_col(ki, ["Receptor", "receptor", "Target"])
    kival_col    = find_col(ki, ["Ki Value", "Ki", "Ki_nM", "Ki (nM)"])
    if not all([ligand_col, receptor_col, kival_col]):
        raise ValueError(
            f"Could not locate required Ki columns. Found: "
            f"ligand={ligand_col}, receptor={receptor_col}, ki={kival_col}\n"
            f"Available columns: {list(ki.columns)}"
        )

    ki["_ligand_norm"] = ki[ligand_col].map(normalise_name)
    ki["_ki_nM"]       = ki[kival_col].map(parse_ki)
    ligand_vocab = sorted(set(ki["_ligand_norm"]) - {""})

    rows = []
    missing_twas = set()

    print("\nMatching drugs and building drug x receptor table ...")
    for _, drug in tqdm(drugs.iterrows(), total=len(drugs)):
        drug_name = drug.get("name")
        norm = normalise_name(drug_name)
        ddd_mg = ddd_to_mg(drug.get("ddd"), drug.get("unit"))
        inv_ddd = (1.0 / ddd_mg) if (not _is_nan(ddd_mg) and ddd_mg > 0) else np.nan

        if not norm:
            continue

        match = process.extractOne(norm, ligand_vocab, scorer=fuzz.WRatio)
        if match is None or match[1] < FUZZY_THRESHOLD:
            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": None,
                "match_score": (match[1] if match else np.nan),
                "receptor": None, "gene": None,
                "Ki_nM": np.nan, "inv_Ki": np.nan,
                "DDD_mg": ddd_mg, "inv_DDD": inv_ddd,
                "affinity_dose_score": np.nan,
                "n_measurements": 0,
                "twas_z": np.nan, "twas_p": np.nan,
                "strong_binder": False, "twas_significant": False,
                "ki_imputed": False, "twas_imputed": False,
            })
            continue

        matched_ligand, score, _ = match
        subset = ki[ki["_ligand_norm"] == matched_ligand]

        grouped = subset.groupby(receptor_col)
        for receptor, grp in grouped:
            ki_nM = aggregate_ki(grp["_ki_nM"], KI_AGGREGATION)
            n_meas = int(grp["_ki_nM"].notna().sum())
            inv_ki = (1.0 / ki_nM) if (not _is_nan(ki_nM) and ki_nM > 0) else np.nan
            gene = receptor_to_gene(receptor)

            if not _is_nan(inv_ki) and not _is_nan(ddd_mg) and ddd_mg > 0:
                score_combined = inv_ki * np.log(ddd_mg + 1.0)
            elif not _is_nan(inv_ki):
                score_combined = inv_ki
            else:
                score_combined = np.nan

            twas = get_twas_for_gene(gene, twas_lookup) if gene else None
            if gene and twas is None:
                missing_twas.add(gene)
            twas_z = twas.get("twas_z", np.nan) if twas else np.nan
            twas_p = twas.get("twas_p", np.nan) if twas else np.nan

            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": matched_ligand,
                "match_score": round(score, 1),
                "receptor": receptor,
                "gene": gene,
                "Ki_nM": ki_nM,
                "inv_Ki": inv_ki,
                "DDD_mg": ddd_mg,
                "inv_DDD": inv_ddd,
                "affinity_dose_score": score_combined,
                "n_measurements": n_meas,
                "twas_z": twas_z,
                "twas_p": twas_p,
                "strong_binder": (not _is_nan(ki_nM)) and ki_nM < STRONG_BINDER_KI_NM,
                "twas_significant": (not _is_nan(twas_p)) and twas_p < TWAS_P_SIGNIFICANT,
                "ki_imputed": False,
                "twas_imputed": False,
            })

    results = pd.DataFrame(rows)

    stats["n_missing_twas_genes"] = len(missing_twas)
    stats["missing_twas_sample"] = sorted(missing_twas)[:12]
    if missing_twas:
        sample = sorted(missing_twas)[:8]
        print(f"\n[TWAS] {len(missing_twas)} receptor gene(s) had no TWAS data "
              f"(e.g. {', '.join(sample)}"
              f"{', ...' if len(missing_twas) > len(sample) else ''})")

    if not results.empty:
        n_before = results["drug"].nunique()
        valid_ddd_drugs = results[results["DDD_mg"].notna()]["drug"].unique()
        results = results[results["drug"].isin(valid_ddd_drugs)].reset_index(drop=True)
        n_after = results["drug"].nunique()
        stats["n_drugs_ddd_dropped"] = int(n_before - n_after)
        print(f"\n[DDD filter] kept {n_after}/{n_before} drug(s) with a valid DDD "
              f"({n_before - n_after} dropped).")

    if not results.empty:
        print("\nImputation step ...")
        results = impute_missing_ki(results, KI_IMPUTE_STRATEGY)
        results = impute_missing_twas(results, TWAS_IMPUTE_STRATEGY)

    results, n_zero_dropped = drop_zero_score_rows(results)
    stats["n_zero_score_dropped"] = n_zero_dropped

    if not results.empty:
        results["rank_in_drug"] = (
            results.groupby("drug")["affinity_dose_score"]
            .rank(ascending=False, method="min")
        )
        results = results.sort_values(
            ["drug", "affinity_dose_score"],
            ascending=[True, False]
        ).reset_index(drop=True)

    if not results.empty:
        stats["n_drugs_matched"] = int(
            results.loc[results["receptor"].notna(), "drug"].nunique()
        )
        stats["n_pairs"] = int(results["receptor"].notna().sum())
        stats["n_ki_imputed"] = int(results.get("ki_imputed",
                                     pd.Series(dtype=bool)).sum())
        stats["n_twas_imputed"] = int(results.get("twas_imputed",
                                       pd.Series(dtype=bool)).sum())
        stats["n_strong_binders"] = int(results["strong_binder"].sum())
        stats["n_twas_significant"] = int(results["twas_significant"].sum())
    else:
        stats.update({
            "n_drugs_matched": 0, "n_pairs": 0, "n_ki_imputed": 0,
            "n_twas_imputed": 0, "n_strong_binders": 0,
            "n_twas_significant": 0,
        })

    return results, stats


def write_outputs(results: pd.DataFrame, output_dir: str):
    paths = {}
    if results.empty:
        print("No results to write.")
        return paths

    csv_path = os.path.join(output_dir, "n05a_ki_twas_full_results.csv")
    results.to_csv(csv_path, index=False)
    print(f"  wrote {csv_path}")
    paths["csv"] = csv_path

    matched = results[results["receptor"].notna()]
    summary = (
        matched.groupby(["atc_code", "drug"])
        .agg(
            n_receptors      = ("receptor", "nunique"),
            n_strong_binders = ("strong_binder", "sum"),
            n_twas_sig       = ("twas_significant", "sum"),
            n_ki_imputed     = ("ki_imputed", "sum"),
            n_twas_imputed   = ("twas_imputed", "sum"),
            best_Ki_nM       = ("Ki_nM", "min"),
            top_score        = ("affinity_dose_score", "max"),
            DDD_mg           = ("DDD_mg", "first"),
        )
        .reset_index()
        .sort_values("top_score", ascending=False)
    )
    summary_path = os.path.join(output_dir, "drug_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"  wrote {summary_path}")
    paths["summary"] = summary_path

    xlsx_path = os.path.join(output_dir, "n05a_ki_twas_full_results.xlsx")
    top5 = (
        matched[matched["rank_in_drug"] <= 5]
        .sort_values(["drug", "rank_in_drug"])
    )
    twas_sig = matched[matched["twas_significant"]].sort_values(
        "affinity_dose_score", ascending=False
    )
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
            results.to_excel(xw, sheet_name="All", index=False)
            top5.to_excel(xw, sheet_name="Top5_per_drug", index=False)
            twas_sig.to_excel(xw, sheet_name="TWAS_significant", index=False)
            summary.to_excel(xw, sheet_name="Drug_summary", index=False)
        print(f"  wrote {xlsx_path}")
        paths["xlsx"] = xlsx_path
    except Exception as e:
        print(f"  (skipped Excel export: {e})")

    return paths


def write_detailed_summary(run_summaries, path: str) -> None:
    L = []
    bar = "=" * 78
    sub = "-" * 78
    L.append(bar)
    L.append("N05A  Ki -> DDD -> TWAS ENRICHMENT PIPELINE — DETAILED SUMMARY")
    L.append("TWAS INTEGRATION: VERSION 2 (Mild exponential per-receptor scaling)")
    L.append(bar)
    L.append(f"Generated            : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
    L.append(f"Ki database          : {KI_DB_PATH}")
    L.append(f"Drug list            : {DRUGS_CSV_PATH}")
    L.append(f"Output root          : {OUTPUT_DIR}")
    L.append(f"Score formula        : affinity_dose_score = inv_Ki * log(DDD_mg + 1)")
    L.append(f"TWAS dirs processed  : {len(run_summaries)}")
    L.append("")
    L.append("Configuration:")
    L.append(f"  FUZZY_THRESHOLD       = {FUZZY_THRESHOLD}")
    L.append(f"  KI_AGGREGATION        = {KI_AGGREGATION}")
    L.append(f"  STRONG_BINDER_KI_NM   = {STRONG_BINDER_KI_NM}")
    L.append(f"  TWAS_P_SIGNIFICANT    = {TWAS_P_SIGNIFICANT}")
    L.append(f"  KI_IMPUTE_STRATEGY    = {KI_IMPUTE_STRATEGY}")
    L.append(f"  TWAS_IMPUTE_STRATEGY  = {TWAS_IMPUTE_STRATEGY}")
    L.append("")
    L.append(bar)
    L.append("CROSS-RUN OVERVIEW")
    L.append(bar)
    header = (f"{'TWAS label':<18}{'genes':>8}{'matched':>9}{'pairs':>8}"
              f"{'zero-drop':>11}{'ki_imp':>8}{'tw_imp':>8}{'tw_sig':>8}")
    L.append(header)
    L.append(sub)
    for s in run_summaries:
        if s.get("error"):
            L.append(f"{s.get('label',''):<18}  ERROR: {s['error']}")
            continue
        L.append(
            f"{s.get('label',''):<18}"
            f"{s.get('n_genes_twas',0):>8}"
            f"{s.get('n_drugs_matched',0):>9}"
            f"{s.get('n_pairs',0):>8}"
            f"{s.get('n_zero_score_dropped',0):>11}"
            f"{s.get('n_ki_imputed',0):>8}"
            f"{s.get('n_twas_imputed',0):>8}"
            f"{s.get('n_twas_significant',0):>8}"
        )
    L.append("")
    for s in run_summaries:
        L.append(bar)
        L.append(f"TWAS RUN: {s.get('label','')}")
        L.append(bar)
        L.append(f"  TWAS directory          : {s.get('twas_dir','')}")
        L.append(f"  Output directory        : {s.get('output_dir','')}")
        if s.get("error"):
            L.append(f"  [ERROR] {s['error']}")
            L.append("")
            continue
        L.append(f"  Genes with TWAS data    : {s.get('n_genes_twas',0):,}")
        L.append(f"  Drugs in input list     : {s.get('n_drugs_input',0):,}")
        L.append(f"  Drugs dropped (no DDD)  : {s.get('n_drugs_ddd_dropped',0):,}")
        L.append(f"  Drugs matched to ligand : {s.get('n_drugs_matched',0):,}")
        L.append(f"  Drug x receptor pairs   : {s.get('n_pairs',0):,}")
        L.append(f"  Rows dropped (score==0) : {s.get('n_zero_score_dropped',0):,}")
        L.append(f"  Ki values imputed       : {s.get('n_ki_imputed',0):,}")
        L.append(f"  TWAS values imputed     : {s.get('n_twas_imputed',0):,}")
        L.append(f"  Strong binders          : {s.get('n_strong_binders',0):,}")
        L.append(f"  TWAS-significant rows   : {s.get('n_twas_significant',0):,}")
        L.append(f"  Receptor genes w/o TWAS : {s.get('n_missing_twas_genes',0):,}")
        if s.get("missing_twas_sample"):
            L.append(f"    e.g. {', '.join(s['missing_twas_sample'])}")
        L.append("")
        top_risk = s.get("top_risk")
        if top_risk is not None and len(top_risk):
            L.append("  Top metabolic-risk drugs (reference = 100):")
            L.append(f"    {'rank':>4}  {'drug':<28}{'risk':>10}{'maxTWASz':>10}")
            for _, r in top_risk.iterrows():
                L.append(
                    f"    {str(r.get('risk_rank','')):>4}  "
                    f"{str(r.get('drug',''))[:27]:<28}"
                    f"{r.get('metabolic_risk_score', float('nan')):>10.2f}"
                    f"{r.get('max_abs_twas_z', float('nan')):>10.3f}"
                )
            L.append("")
        top_hits = s.get("top_hits")
        if top_hits is not None and len(top_hits):
            L.append("  Top strong-binding + TWAS-significant hits:")
            L.append(f"    {'drug':<22}{'gene':<9}{'Ki_nM':>10}"
                     f"{'score':>12}{'twas_z':>9}{'twas_p':>11}")
            for _, r in top_hits.iterrows():
                L.append(
                    f"    {str(r.get('drug',''))[:21]:<22}"
                    f"{str(r.get('gene',''))[:8]:<9}"
                    f"{r.get('Ki_nM', float('nan')):>10.3g}"
                    f"{r.get('affinity_dose_score', float('nan')):>12.4g}"
                    f"{r.get('twas_z', float('nan')):>9.3f}"
                    f"{r.get('twas_p', float('nan')):>11.3g}"
                )
            L.append("")
    L.append(bar)
    L.append("END OF SUMMARY")
    L.append(bar)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"\n  wrote detailed summary -> {path}")


# ======================================================================
if __name__ == "__main__":
    os.makedirs(OUTPUT_DIR, exist_ok=True)
    run_summaries = []

    for twas_dir in TWAS_DIRS:
        label = _safe_label(twas_dir)
        out_dir = os.path.join(OUTPUT_DIR, label)
        print("\n" + "#" * 78)
        print(f"# PROCESSING TWAS DIR: {twas_dir}   (label='{label}')")
        print("# TWAS integration = VERSION 2 (mild exponential per-receptor scaling)")
        print("#" * 78)

        try:
            results, stats = run_pipeline(twas_dir, out_dir)
        except Exception as e:
            print(f"  [ERROR] pipeline failed for {twas_dir}: {e}")
            run_summaries.append({
                "label": label, "twas_dir": twas_dir, "output_dir": out_dir,
                "error": str(e),
            })
            continue

        stats["label"] = label
        n_drugs_matched = stats.get("n_drugs_matched", 0)
        n_pairs = stats.get("n_pairs", 0)
        print(f"\nMatched {n_drugs_matched} drugs to ligands; "
              f"{n_pairs} drug x receptor pairs built.")
        print(f"  imputed Ki rows: {stats.get('n_ki_imputed',0)}  |  "
              f"imputed TWAS rows: {stats.get('n_twas_imputed',0)}")

        print("\nWriting outputs ...")
        write_outputs(results, out_dir)

        print("\nCalculating Metabolic Risk Score (Version 2: mild exponential) ...")
        risk_df = calculate_metabolic_risk_score(
            results,
            weights=METABOLIC_WEIGHTS,
            score_transform="log1p",
            twas_boost=True,
            twas_boost_factor=0.24,             # unused by v2 scaling
            twas_sig_bonus=0.10,
            normalize=True,
            reference_drug="chlorpromazine",
            missing_gene_strategy="mean_abs_z",
            beta=0.08,                          # exponential rate (tune 0.06 - 0.10)
        )
        risk_path = os.path.join(out_dir, "metabolic_risk_score_per_drug.csv")
        risk_df.to_csv(risk_path, index=False)
        print(f"  wrote {risk_path}")

        print("\nN05A drugs ranked by metabolic risk (reference = 100):")
        print(risk_df.head(20).to_string(index=False))

        stats["top_risk"] = risk_df.head(15).copy()
        preview = results[
            results["twas_significant"] & results["affinity_dose_score"].notna()
        ].sort_values("affinity_dose_score", ascending=False).head(15)
        stats["top_hits"] = preview[
            ["drug", "gene", "Ki_nM", "affinity_dose_score", "twas_z", "twas_p"]
        ].copy() if not preview.empty else None
        if preview.empty:
            print("\n(No TWAS-significant hits found — check TWAS_DIR / column names.)")

        run_summaries.append(stats)
        print(f"\nDone with TWAS dir '{label}'.")

    summary_txt = os.path.join(OUTPUT_DIR, "pipeline_summary.txt")
    write_detailed_summary(run_summaries, summary_txt)
    print("\nAll TWAS directories processed (Version 2 - mild exponential). Done.")


##############################################################################
# PROCESSING TWAS DIR: /content/lipids/ldl   (label='ldl')
# TWAS integration = VERSION 2 (mild exponential per-receptor scaling)
##############################################################################
Building TWAS lookup for: /content/lipids/ldl
  Pre-loaded TWAS data for 18,057 genes from 6/6 file(s)
  Sample genes in TWAS lookup: ['PCSK9', 'SMARCA4', 'CELSR2', 'ANKDD1B', 'CETP', 'FEN1', 'ATP13A1', 'FADS1', 'SYPL2', 'YIPF2']
    HTR2C present? False
    HRH1 present? True
    DRD2 present? True
    CHRM1 present? True
    CHRM3 present? True
    HTR2A present? True
Loading Ki database ...
  Ki rows: 98,764
Loading drug list ...
  Drugs: 70

Matching drugs and building drug x receptor table ...


100%|██████████| 70/70 [00:03<00:00, 22.91it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.160, twas_p=0.239); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/ldl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/ldl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/ldl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 2: mild exponential) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-expo] mild expon

100%|██████████| 70/70 [00:03<00:00, 18.43it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.420, twas_p=0.207); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/hdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/hdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/hdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 2: mild exponential) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-expo] mild expon

...


[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.393, twas_p=0.171); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/logtg/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/logtg/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/logtg/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 2: mild exponential) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-expo] mild

100%|██████████| 70/70 [00:04<00:00, 16.92it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.089, twas_p=0.327); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/nonhdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/nonhdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/nonhdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 2: mild exponential) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-expo] m

100%|██████████| 70/70 [00:03<00:00, 22.85it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.367, twas_p=0.21); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/tc/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/tc/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v2_expo/tc/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 2: mild exponential) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS z-score.
  [TWAS-expo] mild exponenti

# Version 4 MILD EXPONENTIAL SCALING on -log10(p)

In [6]:
# ======================================================================
# VERSION 3 (new):  N05A → Ki → DDD → TWAS pipeline
# TWAS INTEGRATION: MILD EXPONENTIAL SCALING on -log10(p)
#   twas_scale = exp(beta * -log10(p))     (beta ~ 0.03 - 0.06)
# Self-contained: run as a single Colab cell.
#   Requires: pip install pandas numpy rapidfuzz tqdm openpyxl
# ======================================================================

import os
import re
import glob
import datetime
import numpy as np
import pandas as pd
from rapidfuzz import process, fuzz
from tqdm import tqdm

# ======================================================================
# CONFIG  --  EDIT THESE PATHS
# ======================================================================
KI_DB_PATH     = "/content/Z_database/KiDatabase_2026-06-22.csv"
DRUGS_CSV_PATH = "/content/Z_database/atc_n05a_drugs.csv"

TWAS_DIRS = [
    "/content/lipids/ldl",
    "/content/lipids/hdl",
    "/content/lipids/logtg",
    "/content/lipids/nonhdl",
    "/content/lipids/tc",
]

# Distinct output root for the -log10(p) version
OUTPUT_DIR = "/content/pipeline_output/n05a_ki_twas_log_v3_logp"

FUZZY_THRESHOLD = 80
KI_AGGREGATION = "min"
STRONG_BINDER_KI_NM = 10.0
TWAS_P_SIGNIFICANT = 0.05
KI_IMPUTE_STRATEGY = "mean"
TWAS_IMPUTE_STRATEGY = "mean"
CSV_ENCODINGS = ["utf-8", "utf-8-sig", "latin-1", "cp1252"]

# ---- Version 3 (logp_expo) scaling parameters ----
LOGP_BETA           = 0.045   # exponential rate on -log10(p) (tune 0.03 - 0.06)
MAX_NEGLOG10_P      = 25.0    # cap to prevent explosion from ultra-small p-values
SECONDARY_BOOST_FAC = 0.02    # tiny secondary drug-level boost (max|z|)

TWAS_SYMBOL_COL_CANDIDATES = [
    "gene_name", "genename", "gene_symbol", "symbol", "hgnc_symbol",
]
TWAS_ENSG_COL_CANDIDATES = [
    "gene", "ensembl_gene_id", "gene_id", "geneid", "ensembl", "id",
]
TWAS_Z_COL_CANDIDATES = [
    "twas.z", "twas_z", "zscore", "z_score", "zstat", "z", "zscore",
]
TWAS_P_COL_CANDIDATES = [
    "twas.p", "twas_p", "pvalue", "p_value", "pval", "p",
]


# ======================================================================
# Receptor (Ki DB label) -> official HGNC gene symbol
# ======================================================================
RECEPTOR_TO_GENE = {
    "5-HT1A": "HTR1A", "5HT1A": "HTR1A",
    "5-HT1B": "HTR1B", "5HT1B": "HTR1B",
    "5-HT1D": "HTR1D",
    "5-HT1E": "HTR1E",
    "5-HT1F": "HTR1F",
    "5-HT2A": "HTR2A", "5HT2A": "HTR2A",
    "5-HT2B": "HTR2B",
    "5-HT2C": "HTR2C", "5HT2C": "HTR2C",
    "5-HT3":  "HTR3A",
    "5-HT5A": "HTR5A",
    "5-HT6":  "HTR6",
    "5-HT7":  "HTR7",
    "D1": "DRD1", "D2": "DRD2", "D3": "DRD3", "D4": "DRD4", "D5": "DRD5",
    "DRD1": "DRD1", "DRD2": "DRD2", "DRD3": "DRD3", "DRD4": "DRD4", "DRD5": "DRD5",
    "alpha1A": "ADRA1A", "alpha1B": "ADRA1B", "alpha1D": "ADRA1D",
    "alpha2A": "ADRA2A", "alpha2B": "ADRA2B", "alpha2C": "ADRA2C",
    "alpha1":  "ADRA1A", "alpha2": "ADRA2A",
    "beta1": "ADRB1", "beta2": "ADRB2", "beta3": "ADRB3",
    "H1": "HRH1", "H2": "HRH2", "H3": "HRH3", "H4": "HRH4",
    "M1": "CHRM1", "M2": "CHRM2", "M3": "CHRM3", "M4": "CHRM4", "M5": "CHRM5",
    "Muscarinic": "CHRM1",
    "Sigma1": "SIGMAR1", "Sigma 1": "SIGMAR1",
    "SERT": "SLC6A4", "DAT": "SLC6A3", "NET": "SLC6A2",
}
RECEPTOR_TO_GENE = {k.strip().lower(): v for k, v in RECEPTOR_TO_GENE.items()}


# ======================================================================
# METABOLIC / DIABETES RISK WEIGHTS FOR ANTIPSYCHOTICS
# ======================================================================
METABOLIC_WEIGHTS = {
    "HRH1":    1.00,
    "HTR2C":   0.92,
    "CHRM3":   0.85,
    "HTR2A":   0.55,
    "ADRA1A":  0.48,
    "ADRA1B":  0.48,
    "HTR6":    0.42,
    "ADRA2A":  0.32,
    "ADRA2B":  0.30,
    "ADRA2C":  0.30,
    "CHRM1":   0.28,
    "CHRM4":   0.22,
    "CHRM5":   0.20,
    "HTR7":    0.25,
    "DRD3":    0.18,
    "DRD2":    0.12,
    "DRD4":    0.10,
    "HTR1A":   0.08,
    "ADRB1":   0.08,
    "ADRB2":   0.07,
    "SIGMAR1": 0.05,
    "SLC6A4":  0.06,
    "SLC6A2":  0.05,
    "SLC6A3":  0.04,
}


# ======================================================================
# Helpers
# ======================================================================
def _is_nan(x) -> bool:
    if x is None:
        return True
    try:
        return bool(np.isnan(x))
    except (TypeError, ValueError):
        return False


def read_csv_robust(path: str, **kwargs) -> pd.DataFrame:
    kwargs.setdefault("low_memory", False)
    last_err = None
    for enc in CSV_ENCODINGS:
        try:
            return pd.read_csv(path, encoding=enc, **kwargs)
        except UnicodeDecodeError as e:
            last_err = e
            continue
    try:
        return pd.read_csv(path, encoding="latin-1",
                           encoding_errors="replace", **kwargs)
    except Exception:
        raise last_err if last_err else RuntimeError(f"Could not read {path}")


def normalise_name(s: str) -> str:
    if not isinstance(s, str):
        return ""
    return re.sub(r"\s+", " ", s.strip().lower())


def receptor_to_gene(receptor: str):
    if not isinstance(receptor, str):
        return None
    key = receptor.strip().lower()
    if key in RECEPTOR_TO_GENE:
        return RECEPTOR_TO_GENE[key]
    key2 = re.sub(r"[\s\-]", "", key)
    for k, v in RECEPTOR_TO_GENE.items():
        if re.sub(r"[\s\-]", "", k) == key2:
            return v
    return None


def parse_ki(value) -> float:
    if value is None:
        return np.nan
    if isinstance(value, (int, float)) and not isinstance(value, bool):
        return float(value)
    s = str(value).strip()
    if s == "" or s.lower() in {"na", "nan", "nd", "n/a", "-"}:
        return np.nan
    s = s.replace(",", "")
    s = re.sub(r"^[><=~]+", "", s)
    m = re.search(r"-?\d+\.?\d*(?:[eE][-+]?\d+)?", s)
    return float(m.group()) if m else np.nan


def ddd_to_mg(ddd, unit) -> float:
    ddd_val = parse_ki(ddd)
    if _is_nan(ddd_val):
        return np.nan
    u = (str(unit).strip().lower() if unit is not None else "")
    factors = {"g": 1000.0, "mg": 1.0, "mcg": 0.001, "µg": 0.001, "ug": 0.001}
    factor = factors.get(u, 1.0)
    return ddd_val * factor


def aggregate_ki(values: pd.Series, how: str) -> float:
    vals = values.dropna()
    if vals.empty:
        return np.nan
    if how == "min":
        return float(vals.min())
    if how == "median":
        return float(vals.median())
    if how == "mean":
        return float(vals.mean())
    return float(vals.min())


def find_col(df: pd.DataFrame, candidates):
    lower_map = {str(c).strip().lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


def _strip_ensg_version(s) -> str:
    return re.sub(r"\.\d+$", "", str(s).strip().upper())


def _safe_label(path: str) -> str:
    base = os.path.basename(os.path.normpath(path))
    if not base:
        base = re.sub(r"[^A-Za-z0-9_.-]+", "_", path).strip("_")
    return re.sub(r"[^A-Za-z0-9_.-]+", "_", base).strip("_") or "twas"


# ======================================================================
# Recompute affinity-derived columns
# ======================================================================
def _recompute_affinity_columns(results: pd.DataFrame) -> pd.DataFrame:
    ki_nM  = results["Ki_nM"]
    ddd_mg = results["DDD_mg"]

    has_ki = ki_nM.notna() & (ki_nM > 0)
    results["inv_Ki"] = np.where(has_ki, 1.0 / ki_nM, np.nan)

    has_inv_ki = results["inv_Ki"].notna()
    has_ddd    = ddd_mg.notna() & (ddd_mg > 0)
    log_ddd    = np.log(ddd_mg.where(has_ddd) + 1.0)
    results["affinity_dose_score"] = np.where(
        has_inv_ki & has_ddd, results["inv_Ki"] * log_ddd,
        np.where(has_inv_ki, results["inv_Ki"], np.nan),
    )

    results["strong_binder"] = ki_nM.notna() & (ki_nM < STRONG_BINDER_KI_NM)
    return results


def drop_zero_score_rows(results: pd.DataFrame):
    if results.empty or "affinity_dose_score" not in results.columns:
        return results, 0
    zero_mask = results["affinity_dose_score"] == 0
    n_dropped = int(zero_mask.sum())
    if n_dropped:
        results = results.loc[~zero_mask].reset_index(drop=True)
        print(f"  [drop-zero] removed {n_dropped} row(s) with affinity_dose_score == 0.")
    else:
        print("  [drop-zero] no rows with affinity_dose_score == 0.")
    return results, n_dropped


def impute_missing_ki(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "ki_imputed" not in results.columns:
        results["ki_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown KI_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid = results.loc[results["Ki_nM"].notna(), ["receptor", "Ki_nM"]]
    if valid.empty:
        print("  [ki-impute] no Ki values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        per_receptor = valid.groupby("receptor")["Ki_nM"].mean()
        global_ref = float(valid["Ki_nM"].mean())
    else:
        per_receptor = valid.groupby("receptor")["Ki_nM"].median()
        global_ref = float(valid["Ki_nM"].median())
    need = results["Ki_nM"].isna() & results["receptor"].notna()
    n_need = int(need.sum())
    if n_need == 0:
        print(f"  [ki-impute] strategy='{strategy}': no missing Ki values to fill.")
        return results
    filled_ref = results.loc[need, "receptor"].map(per_receptor)
    filled_ref = filled_ref.fillna(global_ref)
    results.loc[need, "Ki_nM"] = filled_ref.values
    results.loc[need, "ki_imputed"] = True
    results = _recompute_affinity_columns(results)
    print(f"  [ki-impute] strategy='{strategy}': filled {n_need} missing Ki "
          f"value(s) (global fallback Ki = {global_ref:.3g} nM).")
    return results


def impute_missing_twas(results: pd.DataFrame, strategy: str = "none") -> pd.DataFrame:
    if results.empty:
        return results
    if "twas_imputed" not in results.columns:
        results["twas_imputed"] = False
    if strategy == "none":
        return results
    if strategy not in {"mean", "median"}:
        raise ValueError(
            f"Unknown TWAS_IMPUTE_STRATEGY: {strategy!r} "
            f"(expected 'none', 'mean' or 'median')"
        )
    valid_z = results.loc[results["twas_z"].notna(), "twas_z"]
    valid_p = results.loc[results["twas_p"].notna(), "twas_p"]
    if valid_z.empty and valid_p.empty:
        print("  [twas-impute] no TWAS values available to build a reference; skipped.")
        return results
    if strategy == "mean":
        fill_z = float(valid_z.mean()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.mean()) if not valid_p.empty else np.nan
    else:
        fill_z = float(valid_z.median()) if not valid_z.empty else np.nan
        fill_p = float(valid_p.median()) if not valid_p.empty else np.nan
    need_z = results["gene"].notna() & results["twas_z"].isna()
    need_p = results["gene"].notna() & results["twas_p"].isna()
    n_need = int((need_z | need_p).sum())
    if n_need == 0:
        print(f"  [twas-impute] strategy='{strategy}': no missing TWAS values to fill.")
        return results
    if not _is_nan(fill_z):
        results.loc[need_z, "twas_z"] = fill_z
    if not _is_nan(fill_p):
        results.loc[need_p, "twas_p"] = fill_p
    results.loc[need_z | need_p, "twas_imputed"] = True
    print(f"  [twas-impute] strategy='{strategy}': filled {n_need} gene row(s) "
          f"(twas_z={fill_z:.3f}, twas_p={fill_p:.3g}); "
          f"imputed rows kept NON-significant.")
    return results


# ======================================================================
# TWAS lookup
# ======================================================================
def _read_twas_file(fpath: str):
    for sep in [",", "\t", None]:
        for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
            try:
                df = pd.read_csv(
                    fpath, sep=sep, encoding=enc, engine="python",
                    low_memory=False, on_bad_lines="skip"
                )
                if df is not None and not df.empty and df.shape[1] >= 2:
                    return df
            except Exception:
                continue
    try:
        df = pd.read_csv(fpath, engine="python", on_bad_lines="skip")
        if df is not None and not df.empty:
            return df
    except Exception:
        pass
    return None


def build_twas_lookup(twas_dir: str):
    lookup = {}
    files = glob.glob(os.path.join(twas_dir, "*.*"))
    if not files:
        print(f"  [warn] No files found in {twas_dir}")
        return lookup
    successful_files = 0
    skipped_files = []
    for fpath in files:
        try:
            df = _read_twas_file(fpath)
            if df is None or df.empty:
                skipped_files.append((os.path.basename(fpath), "unreadable/empty"))
                continue
            sym_col  = find_col(df, TWAS_SYMBOL_COL_CANDIDATES)
            ensg_col = find_col(df, TWAS_ENSG_COL_CANDIDATES)
            z_col    = find_col(df, TWAS_Z_COL_CANDIDATES)
            p_col    = find_col(df, TWAS_P_COL_CANDIDATES)
            if ensg_col is not None and ensg_col == sym_col:
                ensg_col = None
            if z_col is None or (sym_col is None and ensg_col is None):
                skipped_files.append((
                    os.path.basename(fpath),
                    f"missing cols (z={z_col}, sym={sym_col}, ensg={ensg_col}); "
                    f"had {list(df.columns)[:8]}"
                ))
                continue
            n = len(df)
            sym_vals  = (df[sym_col].astype(str).str.strip().str.upper()
                         if sym_col else pd.Series([""] * n, index=df.index))
            ensg_vals = (df[ensg_col].map(_strip_ensg_version)
                         if ensg_col else pd.Series([""] * n, index=df.index))
            z_vals = df[z_col].map(parse_ki)
            p_vals = (df[p_col].map(parse_ki)
                      if p_col else pd.Series([np.nan] * n, index=df.index))
            bad = {"NAN", "NA", "NONE", ""}
            for i in df.index:
                gene = sym_vals[i]
                if gene in bad:
                    gene = ensg_vals[i]
                if gene in bad:
                    continue
                z = z_vals[i]
                p = p_vals[i]
                if _is_nan(z) and _is_nan(p):
                    continue
                prev = lookup.get(gene)
                if prev is None:
                    lookup[gene] = {"twas_z": z, "twas_p": p}
                else:
                    prev_p = prev.get("twas_p", np.nan)
                    if not _is_nan(p) and (_is_nan(prev_p) or p < prev_p):
                        lookup[gene] = {"twas_z": z, "twas_p": p}
            successful_files += 1
        except Exception as e:
            skipped_files.append((os.path.basename(fpath), f"error: {e}"))
            continue
    print(f"  Pre-loaded TWAS data for {len(lookup):,} genes "
          f"from {successful_files}/{len(files)} file(s)")
    if skipped_files:
        print(f"  [warn] {len(skipped_files)} file(s) skipped:")
        for name, reason in skipped_files[:8]:
            print(f"         - {name}: {reason}")
    return lookup


def get_twas_for_gene(gene: str, lookup):
    if gene is None or lookup is None:
        return None
    return lookup.get(str(gene).strip().upper())


# ======================================================================
# Metabolic risk score  --  VERSION 3: EXPONENTIAL scaling on -log10(p)
# ======================================================================
def calculate_metabolic_risk_score(
    df: pd.DataFrame,
    weights: dict = None,
    score_transform: str = "log1p",
    twas_boost: bool = True,
    twas_scaling: str = "logp_expo",     # "logp_expo" | "logp_linear" | "z_expo" | "z_linear" | "none"
    beta: float = 0.045,                 # exponential rate for logp_expo / z_expo
    gamma: float = 0.12,                 # linear factor for logp_linear
    twas_sig_bonus: float = 0.10,
    normalize: bool = True,
    reference_drug: str = "chlorpromazine",
    missing_gene_strategy: str = "mean_abs_z",
    max_neglog10_p: float = 25.0,        # cap to prevent explosion from ultra-small p
    secondary_boost_factor: float = 0.02,
) -> pd.DataFrame:
    """
    Per-drug Metabolic Risk Score for antipsychotics.

    TWAS INTEGRATION -- VERSION 3 (Mild exponential scaling on -log10(p)):
      Each receptor's weighted_contribution is multiplied by
      twas_scale = exp(beta * -log10(p)) BEFORE aggregation. This gives
      stronger emphasis to highly significant TWAS hits while remaining
      stable. -log10(p) is capped at `max_neglog10_p`. base_score is
      recomputed from the TWAS-scaled contributions; a tiny secondary
      drug-level boost (secondary_boost_factor * max|z|) is retained.

    twas_scaling options:
        "logp_expo"   : exp(beta  * -log10(p))     <- Version 3 default
        "logp_linear" : 1 + gamma * -log10(p)
        "z_expo"      : exp(beta  * |z|)           (v2 behaviour)
        "z_linear"    : 1 + beta  * |z|            (v1 behaviour)
        "none"        : no per-receptor TWAS scaling
    """
    if weights is None:
        weights = METABOLIC_WEIGHTS

    df = df.copy()
    mask = df["gene"].notna() & df["affinity_dose_score"].notna()
    metab_mask = mask & df["gene"].isin(weights.keys())

    # ---- report TWAS coverage among scored metabolic receptors ----
    n_metab = int(metab_mask.sum())
    uses_p = twas_scaling in ("logp_expo", "logp_linear")
    cov_col = "twas_p" if uses_p else "twas_z"
    n_metab_twas = int((metab_mask & df[cov_col].notna()).sum())
    if twas_boost and n_metab > 0:
        cov = n_metab_twas / n_metab
        if cov < 0.5:
            print(f"  [note] TWAS coverage is low: only {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS {cov_col}. "
                  f"Per-receptor scaling will be partial.")
        else:
            print(f"  [note] TWAS coverage: {n_metab_twas}/{n_metab} "
                  f"({cov:.0%}) metabolic-receptor rows have a TWAS {cov_col}.")

    if score_transform == "log1p":
        affinity = np.log1p(df.loc[mask, "affinity_dose_score"].clip(lower=0))
    elif score_transform == "none":
        affinity = df.loc[mask, "affinity_dose_score"]
    else:
        raise ValueError(f"Unknown score_transform: {score_transform!r}")

    df.loc[mask, "receptor_weight"] = df.loc[mask, "gene"].map(weights).fillna(0.0)
    df.loc[mask, "weighted_contribution"] = affinity * df.loc[mask, "receptor_weight"]

    drug_scores = (
        df.groupby("drug", dropna=False)["weighted_contribution"]
        .sum()
        .reset_index()
        .rename(columns={"weighted_contribution": "base_score"})
    )

    # ==================================================================
    # TWAS integration - Version 3: exponential on -log10(p)
    # ==================================================================
    if twas_boost and twas_scaling != "none":
        df_filled = df.copy()

        # --- Fairness imputation for missing metabolic genes ---
        if missing_gene_strategy == "mean_abs_z":
            if uses_p:
                # impute missing metabolic p-values via the mean -log10(p)
                avail_p = df_filled.loc[
                    metab_mask & df_filled["twas_p"].notna(), "twas_p"
                ].clip(lower=1e-300)
                if len(avail_p) > 0:
                    mean_neglogp = float((-np.log10(avail_p)).mean())
                else:
                    mean_neglogp = 0.0
                impute_mask = (
                    mask
                    & df_filled["twas_p"].isna()
                    & df_filled["gene"].isin(weights.keys())
                )
                n_imputed = int(impute_mask.sum())
                if n_imputed > 0:
                    df_filled.loc[impute_mask, "twas_p"] = 10.0 ** (-mean_neglogp)
                    print(f"  [fairness] imputed {n_imputed} missing metabolic "
                          f"p-value(s) with mean -log10(p) = {mean_neglogp:.3f}")
            else:
                available_z = df_filled.loc[
                    metab_mask & df_filled["twas_z"].notna(), "twas_z"
                ].abs()
                mean_abs_z = float(available_z.mean()) if len(available_z) > 0 else 0.0
                impute_mask = (
                    mask
                    & df_filled["twas_z"].isna()
                    & df_filled["gene"].isin(weights.keys())
                )
                n_imputed = int(impute_mask.sum())
                if n_imputed > 0:
                    df_filled.loc[impute_mask, "twas_z"] = mean_abs_z
                    print(f"  [fairness] imputed {n_imputed} missing metabolic-gene "
                          f"row(s) with mean |twas_z| = {mean_abs_z:.3f}")
        elif missing_gene_strategy == "zero":
            print("  [fairness] missing_gene_strategy='zero' "
                  "(no TWAS boost for missing genes)")
        else:
            raise ValueError(
                f"Unknown missing_gene_strategy: {missing_gene_strategy!r} "
                f"(expected 'mean_abs_z' or 'zero')"
            )

        # --- Per-receptor TWAS scaling ---
        if twas_scaling == "logp_expo":
            p_safe = df_filled.loc[mask, "twas_p"].clip(lower=1e-300)
            neglog10_p = (-np.log10(p_safe)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = np.exp(beta * neglog10_p)
            print(f"  [TWAS-logp] exponential scaling on -log10(p) active "
                  f"(beta={beta}, cap={max_neglog10_p}). "
                  f"Secondary drug-level boost factor = {secondary_boost_factor}")
        elif twas_scaling == "logp_linear":
            p_safe = df_filled.loc[mask, "twas_p"].clip(lower=1e-300)
            neglog10_p = (-np.log10(p_safe)).clip(upper=max_neglog10_p).fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = 1.0 + gamma * neglog10_p
            print(f"  [TWAS-logp] linear scaling on -log10(p) active "
                  f"(gamma={gamma}, cap={max_neglog10_p}).")
        elif twas_scaling == "z_expo":
            twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = np.exp(beta * twas_abs)
            print(f"  [TWAS-z] exponential scaling on |z| active (beta={beta}).")
        elif twas_scaling == "z_linear":
            twas_abs = df_filled.loc[mask, "twas_z"].abs().fillna(0.0)
            df_filled.loc[mask, "twas_scale"] = 1.0 + beta * twas_abs
            print(f"  [TWAS-z] linear scaling on |z| active (factor={beta}).")
        else:
            raise ValueError(f"Unknown twas_scaling: {twas_scaling!r}")

        # Apply TWAS scale directly to each receptor's contribution
        df_filled.loc[mask, "weighted_contribution"] = (
            affinity
            * df_filled.loc[mask, "receptor_weight"]
            * df_filled.loc[mask, "twas_scale"]
        )

        # Recompute base_score from the TWAS-scaled per-receptor contributions
        drug_scores = (
            df_filled.groupby("drug", dropna=False)["weighted_contribution"]
            .sum()
            .reset_index()
            .rename(columns={"weighted_contribution": "base_score"})
        )

        # Optional tiny secondary drug-level boost (uses max|z| for stability)
        twas_signal = (
            df_filled.groupby("drug")["twas_z"]
            .apply(lambda x: x.abs().max() if x.notna().any() else 0.0)
            .reset_index()
            .rename(columns={"twas_z": "max_abs_twas_z"})
        )
        drug_scores = drug_scores.merge(twas_signal, on="drug", how="left")
        drug_scores["max_abs_twas_z"] = drug_scores["max_abs_twas_z"].fillna(0.0)
        boost = 1 + secondary_boost_factor * drug_scores["max_abs_twas_z"]
    else:
        boost = 1.0

    if twas_sig_bonus and "twas_significant" in df.columns:
        sig = df[mask & df["twas_significant"].fillna(False)
                 & (df["receptor_weight"] > 0)]
        n_sig = (
            sig.groupby("drug").size()
            .reset_index(name="n_twas_sig_metabolic")
        )
        drug_scores = drug_scores.merge(n_sig, on="drug", how="left")
        drug_scores["n_twas_sig_metabolic"] = (
            drug_scores["n_twas_sig_metabolic"].fillna(0).astype(int)
        )
        sig_mult = 1 + twas_sig_bonus * drug_scores["n_twas_sig_metabolic"]
    else:
        sig_mult = 1.0

    drug_scores["metabolic_risk_score"] = (
        drug_scores["base_score"] * boost * sig_mult
    )

    if normalize:
        ref_mask = drug_scores["drug"].astype(str).str.lower() \
            == reference_drug.lower()
        ref_score = drug_scores.loc[ref_mask, "metabolic_risk_score"]
        if not ref_score.empty and ref_score.values[0] > 0:
            ref = ref_score.values[0]
        else:
            print(f"  [warn] reference drug '{reference_drug}' not found or "
                  f"zero-scored; normalising to the max score instead.")
            ref = drug_scores["metabolic_risk_score"].max()
        if ref and ref > 0:
            drug_scores["metabolic_risk_score"] = (
                drug_scores["metabolic_risk_score"] / ref * 100
            )
            drug_scores["risk_vs_reference"] = (
                drug_scores["metabolic_risk_score"] - 100
            )

    n_zero_drugs = int((drug_scores["metabolic_risk_score"] == 0).sum())
    if n_zero_drugs:
        drug_scores = drug_scores[drug_scores["metabolic_risk_score"] != 0]
        print(f"  [drop-zero] removed {n_zero_drugs} drug(s) with "
              f"metabolic_risk_score == 0.")

    drug_scores["risk_rank"] = (
        drug_scores["metabolic_risk_score"]
        .rank(ascending=False, method="min")
        .astype("Int64")
    )
    drug_scores = drug_scores.sort_values(
        "metabolic_risk_score", ascending=False
    ).reset_index(drop=True)

    preferred = [
        "risk_rank", "drug", "metabolic_risk_score", "base_score",
        "max_abs_twas_z", "n_twas_sig_metabolic", "risk_vs_reference",
    ]
    cols = [c for c in preferred if c in drug_scores.columns]
    return drug_scores[cols]


# ======================================================================
# Core pipeline
# ======================================================================
def run_pipeline(twas_dir: str, output_dir: str):
    os.makedirs(output_dir, exist_ok=True)
    stats = {"twas_dir": twas_dir, "output_dir": output_dir}

    print(f"Building TWAS lookup for: {twas_dir}")
    twas_lookup = build_twas_lookup(twas_dir)
    stats["n_genes_twas"] = len(twas_lookup)

    sample = list(twas_lookup.keys())[:10] if twas_lookup else "EMPTY"
    print(f"  Sample genes in TWAS lookup: {sample}")
    for g in ("HTR2C", "HRH1", "DRD2", "CHRM1", "CHRM3", "HTR2A"):
        print(f"    {g} present? {g in (twas_lookup or {})}")

    print("Loading Ki database ...")
    ki = read_csv_robust(KI_DB_PATH)
    print(f"  Ki rows: {len(ki):,}")

    print("Loading drug list ...")
    drugs = read_csv_robust(DRUGS_CSV_PATH)
    print(f"  Drugs: {len(drugs):,}")
    stats["n_drugs_input"] = len(drugs)

    ligand_col   = find_col(ki, ["Test Ligands", "Test Ligand", "Ligand", "ligand"])
    receptor_col = find_col(ki, ["Receptor", "receptor", "Target"])
    kival_col    = find_col(ki, ["Ki Value", "Ki", "Ki_nM", "Ki (nM)"])
    if not all([ligand_col, receptor_col, kival_col]):
        raise ValueError(
            f"Could not locate required Ki columns. Found: "
            f"ligand={ligand_col}, receptor={receptor_col}, ki={kival_col}\n"
            f"Available columns: {list(ki.columns)}"
        )

    ki["_ligand_norm"] = ki[ligand_col].map(normalise_name)
    ki["_ki_nM"]       = ki[kival_col].map(parse_ki)
    ligand_vocab = sorted(set(ki["_ligand_norm"]) - {""})

    rows = []
    missing_twas = set()

    print("\nMatching drugs and building drug x receptor table ...")
    for _, drug in tqdm(drugs.iterrows(), total=len(drugs)):
        drug_name = drug.get("name")
        norm = normalise_name(drug_name)
        ddd_mg = ddd_to_mg(drug.get("ddd"), drug.get("unit"))
        inv_ddd = (1.0 / ddd_mg) if (not _is_nan(ddd_mg) and ddd_mg > 0) else np.nan

        if not norm:
            continue

        match = process.extractOne(norm, ligand_vocab, scorer=fuzz.WRatio)
        if match is None or match[1] < FUZZY_THRESHOLD:
            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": None,
                "match_score": (match[1] if match else np.nan),
                "receptor": None, "gene": None,
                "Ki_nM": np.nan, "inv_Ki": np.nan,
                "DDD_mg": ddd_mg, "inv_DDD": inv_ddd,
                "affinity_dose_score": np.nan,
                "n_measurements": 0,
                "twas_z": np.nan, "twas_p": np.nan,
                "strong_binder": False, "twas_significant": False,
                "ki_imputed": False, "twas_imputed": False,
            })
            continue

        matched_ligand, score, _ = match
        subset = ki[ki["_ligand_norm"] == matched_ligand]

        grouped = subset.groupby(receptor_col)
        for receptor, grp in grouped:
            ki_nM = aggregate_ki(grp["_ki_nM"], KI_AGGREGATION)
            n_meas = int(grp["_ki_nM"].notna().sum())
            inv_ki = (1.0 / ki_nM) if (not _is_nan(ki_nM) and ki_nM > 0) else np.nan
            gene = receptor_to_gene(receptor)

            if not _is_nan(inv_ki) and not _is_nan(ddd_mg) and ddd_mg > 0:
                score_combined = inv_ki * np.log(ddd_mg + 1.0)
            elif not _is_nan(inv_ki):
                score_combined = inv_ki
            else:
                score_combined = np.nan

            twas = get_twas_for_gene(gene, twas_lookup) if gene else None
            if gene and twas is None:
                missing_twas.add(gene)
            twas_z = twas.get("twas_z", np.nan) if twas else np.nan
            twas_p = twas.get("twas_p", np.nan) if twas else np.nan

            rows.append({
                "atc_code": drug.get("atc_code"),
                "drug": drug_name,
                "matched_ligand": matched_ligand,
                "match_score": round(score, 1),
                "receptor": receptor,
                "gene": gene,
                "Ki_nM": ki_nM,
                "inv_Ki": inv_ki,
                "DDD_mg": ddd_mg,
                "inv_DDD": inv_ddd,
                "affinity_dose_score": score_combined,
                "n_measurements": n_meas,
                "twas_z": twas_z,
                "twas_p": twas_p,
                "strong_binder": (not _is_nan(ki_nM)) and ki_nM < STRONG_BINDER_KI_NM,
                "twas_significant": (not _is_nan(twas_p)) and twas_p < TWAS_P_SIGNIFICANT,
                "ki_imputed": False,
                "twas_imputed": False,
            })

    results = pd.DataFrame(rows)

    stats["n_missing_twas_genes"] = len(missing_twas)
    stats["missing_twas_sample"] = sorted(missing_twas)[:12]
    if missing_twas:
        sample = sorted(missing_twas)[:8]
        print(f"\n[TWAS] {len(missing_twas)} receptor gene(s) had no TWAS data "
              f"(e.g. {', '.join(sample)}"
              f"{', ...' if len(missing_twas) > len(sample) else ''})")

    if not results.empty:
        n_before = results["drug"].nunique()
        valid_ddd_drugs = results[results["DDD_mg"].notna()]["drug"].unique()
        results = results[results["drug"].isin(valid_ddd_drugs)].reset_index(drop=True)
        n_after = results["drug"].nunique()
        stats["n_drugs_ddd_dropped"] = int(n_before - n_after)
        print(f"\n[DDD filter] kept {n_after}/{n_before} drug(s) with a valid DDD "
              f"({n_before - n_after} dropped).")

    if not results.empty:
        print("\nImputation step ...")
        results = impute_missing_ki(results, KI_IMPUTE_STRATEGY)
        results = impute_missing_twas(results, TWAS_IMPUTE_STRATEGY)

    results, n_zero_dropped = drop_zero_score_rows(results)
    stats["n_zero_score_dropped"] = n_zero_dropped

    if not results.empty:
        results["rank_in_drug"] = (
            results.groupby("drug")["affinity_dose_score"]
            .rank(ascending=False, method="min")
        )
        results = results.sort_values(
            ["drug", "affinity_dose_score"],
            ascending=[True, False]
        ).reset_index(drop=True)

    if not results.empty:
        stats["n_drugs_matched"] = int(
            results.loc[results["receptor"].notna(), "drug"].nunique()
        )
        stats["n_pairs"] = int(results["receptor"].notna().sum())
        stats["n_ki_imputed"] = int(results.get("ki_imputed",
                                     pd.Series(dtype=bool)).sum())
        stats["n_twas_imputed"] = int(results.get("twas_imputed",
                                       pd.Series(dtype=bool)).sum())
        stats["n_strong_binders"] = int(results["strong_binder"].sum())
        stats["n_twas_significant"] = int(results["twas_significant"].sum())
    else:
        stats.update({
            "n_drugs_matched": 0, "n_pairs": 0, "n_ki_imputed": 0,
            "n_twas_imputed": 0, "n_strong_binders": 0,
            "n_twas_significant": 0,
        })

    return results, stats


def write_outputs(results: pd.DataFrame, output_dir: str):
    paths = {}
    if results.empty:
        print("No results to write.")
        return paths

    csv_path = os.path.join(output_dir, "n05a_ki_twas_full_results.csv")
    results.to_csv(csv_path, index=False)
    print(f"  wrote {csv_path}")
    paths["csv"] = csv_path

    matched = results[results["receptor"].notna()]
    summary = (
        matched.groupby(["atc_code", "drug"])
        .agg(
            n_receptors      = ("receptor", "nunique"),
            n_strong_binders = ("strong_binder", "sum"),
            n_twas_sig       = ("twas_significant", "sum"),
            n_ki_imputed     = ("ki_imputed", "sum"),
            n_twas_imputed   = ("twas_imputed", "sum"),
            best_Ki_nM       = ("Ki_nM", "min"),
            top_score        = ("affinity_dose_score", "max"),
            DDD_mg           = ("DDD_mg", "first"),
        )
        .reset_index()
        .sort_values("top_score", ascending=False)
    )
    summary_path = os.path.join(output_dir, "drug_summary.csv")
    summary.to_csv(summary_path, index=False)
    print(f"  wrote {summary_path}")
    paths["summary"] = summary_path

    xlsx_path = os.path.join(output_dir, "n05a_ki_twas_full_results.xlsx")
    top5 = (
        matched[matched["rank_in_drug"] <= 5]
        .sort_values(["drug", "rank_in_drug"])
    )
    twas_sig = matched[matched["twas_significant"]].sort_values(
        "affinity_dose_score", ascending=False
    )
    try:
        with pd.ExcelWriter(xlsx_path, engine="openpyxl") as xw:
            results.to_excel(xw, sheet_name="All", index=False)
            top5.to_excel(xw, sheet_name="Top5_per_drug", index=False)
            twas_sig.to_excel(xw, sheet_name="TWAS_significant", index=False)
            summary.to_excel(xw, sheet_name="Drug_summary", index=False)
        print(f"  wrote {xlsx_path}")
        paths["xlsx"] = xlsx_path
    except Exception as e:
        print(f"  (skipped Excel export: {e})")

    return paths


def write_detailed_summary(run_summaries, path: str) -> None:
    L = []
    bar = "=" * 78
    sub = "-" * 78
    L.append(bar)
    L.append("N05A  Ki -> DDD -> TWAS ENRICHMENT PIPELINE — DETAILED SUMMARY")
    L.append("TWAS INTEGRATION: VERSION 3 (Mild exponential scaling on -log10(p))")
    L.append(bar)
    L.append(f"Generated            : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
    L.append(f"Ki database          : {KI_DB_PATH}")
    L.append(f"Drug list            : {DRUGS_CSV_PATH}")
    L.append(f"Output root          : {OUTPUT_DIR}")
    L.append(f"Score formula        : affinity_dose_score = inv_Ki * log(DDD_mg + 1)")
    L.append(f"TWAS scale (v3)      : twas_scale = exp(beta * -log10(p)), "
             f"beta={LOGP_BETA}, cap={MAX_NEGLOG10_P}")
    L.append(f"TWAS dirs processed  : {len(run_summaries)}")
    L.append("")
    L.append("Configuration:")
    L.append(f"  FUZZY_THRESHOLD       = {FUZZY_THRESHOLD}")
    L.append(f"  KI_AGGREGATION        = {KI_AGGREGATION}")
    L.append(f"  STRONG_BINDER_KI_NM   = {STRONG_BINDER_KI_NM}")
    L.append(f"  TWAS_P_SIGNIFICANT    = {TWAS_P_SIGNIFICANT}")
    L.append(f"  KI_IMPUTE_STRATEGY    = {KI_IMPUTE_STRATEGY}")
    L.append(f"  TWAS_IMPUTE_STRATEGY  = {TWAS_IMPUTE_STRATEGY}")
    L.append(f"  LOGP_BETA             = {LOGP_BETA}")
    L.append(f"  MAX_NEGLOG10_P        = {MAX_NEGLOG10_P}")
    L.append("")
    L.append(bar)
    L.append("CROSS-RUN OVERVIEW")
    L.append(bar)
    header = (f"{'TWAS label':<18}{'genes':>8}{'matched':>9}{'pairs':>8}"
              f"{'zero-drop':>11}{'ki_imp':>8}{'tw_imp':>8}{'tw_sig':>8}")
    L.append(header)
    L.append(sub)
    for s in run_summaries:
        if s.get("error"):
            L.append(f"{s.get('label',''):<18}  ERROR: {s['error']}")
            continue
        L.append(
            f"{s.get('label',''):<18}"
            f"{s.get('n_genes_twas',0):>8}"
            f"{s.get('n_drugs_matched',0):>9}"
            f"{s.get('n_pairs',0):>8}"
            f"{s.get('n_zero_score_dropped',0):>11}"
            f"{s.get('n_ki_imputed',0):>8}"
            f"{s.get('n_twas_imputed',0):>8}"
            f"{s.get('n_twas_significant',0):>8}"
        )
    L.append("")
    for s in run_summaries:
        L.append(bar)
        L.append(f"TWAS RUN: {s.get('label','')}")
        L.append(bar)
        L.append(f"  TWAS directory          : {s.get('twas_dir','')}")
        L.append(f"  Output directory        : {s.get('output_dir','')}")
        if s.get("error"):
            L.append(f"  [ERROR] {s['error']}")
            L.append("")
            continue
        L.append(f"  Genes with TWAS data    : {s.get('n_genes_twas',0):,}")
        L.append(f"  Drugs in input list     : {s.get('n_drugs_input',0):,}")
        L.append(f"  Drugs dropped (no DDD)  : {s.get('n_drugs_ddd_dropped',0):,}")
        L.append(f"  Drugs matched to ligand : {s.get('n_drugs_matched',0):,}")
        L.append(f"  Drug x receptor pairs   : {s.get('n_pairs',0):,}")
        L.append(f"  Rows dropped (score==0) : {s.get('n_zero_score_dropped',0):,}")
        L.append(f"  Ki values imputed       : {s.get('n_ki_imputed',0):,}")
        L.append(f"  TWAS values imputed     : {s.get('n_twas_imputed',0):,}")
        L.append(f"  Strong binders          : {s.get('n_strong_binders',0):,}")
        L.append(f"  TWAS-significant rows   : {s.get('n_twas_significant',0):,}")
        L.append(f"  Receptor genes w/o TWAS : {s.get('n_missing_twas_genes',0):,}")
        if s.get("missing_twas_sample"):
            L.append(f"    e.g. {', '.join(s['missing_twas_sample'])}")
        L.append("")
        top_risk = s.get("top_risk")
        if top_risk is not None and len(top_risk):
            L.append("  Top metabolic-risk drugs (reference = 100):")
            L.append(f"    {'rank':>4}  {'drug':<28}{'risk':>10}{'maxTWASz':>10}")
            for _, r in top_risk.iterrows():
                L.append(
                    f"    {str(r.get('risk_rank','')):>4}  "
                    f"{str(r.get('drug',''))[:27]:<28}"
                    f"{r.get('metabolic_risk_score', float('nan')):>10.2f}"
                    f"{r.get('max_abs_twas_z', float('nan')):>10.3f}"
                )
            L.append("")
        top_hits = s.get("top_hits")
        if top_hits is not None and len(top_hits):
            L.append("  Top strong-binding + TWAS-significant hits:")
            L.append(f"    {'drug':<22}{'gene':<9}{'Ki_nM':>10}"
                     f"{'score':>12}{'twas_z':>9}{'twas_p':>11}")
            for _, r in top_hits.iterrows():
                L.append(
                    f"    {str(r.get('drug',''))[:21]:<22}"
                    f"{str(r.get('gene',''))[:8]:<9}"
                    f"{r.get('Ki_nM', float('nan')):>10.3g}"
                    f"{r.get('affinity_dose_score', float('nan')):>12.4g}"
                    f"{r.get('twas_z', float('nan')):>9.3f}"
                    f"{r.get('twas_p', float('nan')):>11.3g}"
                )
            L.append("")
    L.append(bar)
    L.append("END OF SUMMARY")
    L.append(bar)
    with open(path, "w", encoding="utf-8") as fh:
        fh.write("\n".join(L))
    print(f"\n  wrote detailed summary -> {path}")


# ======================================================================
# MAIN
# ======================================================================
os.makedirs(OUTPUT_DIR, exist_ok=True)
run_summaries = []

for twas_dir in TWAS_DIRS:
    label = _safe_label(twas_dir)
    out_dir = os.path.join(OUTPUT_DIR, label)
    print("\n" + "#" * 78)
    print(f"# PROCESSING TWAS DIR: {twas_dir}   (label='{label}')")
    print("# TWAS integration = VERSION 3 (mild exponential scaling on -log10(p))")
    print("#" * 78)

    try:
        results, stats = run_pipeline(twas_dir, out_dir)
    except Exception as e:
        print(f"  [ERROR] pipeline failed for {twas_dir}: {e}")
        run_summaries.append({
            "label": label, "twas_dir": twas_dir, "output_dir": out_dir,
            "error": str(e),
        })
        continue

    stats["label"] = label
    n_drugs_matched = stats.get("n_drugs_matched", 0)
    n_pairs = stats.get("n_pairs", 0)
    print(f"\nMatched {n_drugs_matched} drugs to ligands; "
          f"{n_pairs} drug x receptor pairs built.")
    print(f"  imputed Ki rows: {stats.get('n_ki_imputed',0)}  |  "
          f"imputed TWAS rows: {stats.get('n_twas_imputed',0)}")

    print("\nWriting outputs ...")
    write_outputs(results, out_dir)

    print("\nCalculating Metabolic Risk Score (Version 3: logp_expo) ...")
    risk_df = calculate_metabolic_risk_score(
        results,
        weights=METABOLIC_WEIGHTS,
        score_transform="log1p",
        twas_boost=True,
        twas_scaling="logp_expo",           # exp(beta * -log10(p))
        beta=LOGP_BETA,                     # 0.045 (tune 0.03 - 0.06)
        twas_sig_bonus=0.10,
        normalize=True,
        reference_drug="chlorpromazine",
        missing_gene_strategy="mean_abs_z",
        max_neglog10_p=MAX_NEGLOG10_P,
        secondary_boost_factor=SECONDARY_BOOST_FAC,
    )
    risk_path = os.path.join(out_dir, "metabolic_risk_score_per_drug.csv")
    risk_df.to_csv(risk_path, index=False)
    print(f"  wrote {risk_path}")

    print("\nN05A drugs ranked by metabolic risk (reference = 100):")
    print(risk_df.head(20).to_string(index=False))

    stats["top_risk"] = risk_df.head(15).copy()
    preview = results[
        results["twas_significant"] & results["affinity_dose_score"].notna()
    ].sort_values("affinity_dose_score", ascending=False).head(15)
    stats["top_hits"] = preview[
        ["drug", "gene", "Ki_nM", "affinity_dose_score", "twas_z", "twas_p"]
    ].copy() if not preview.empty else None
    if preview.empty:
        print("\n(No TWAS-significant hits found — check TWAS_DIR / column names.)")

    run_summaries.append(stats)
    print(f"\nDone with TWAS dir '{label}'.")

summary_txt = os.path.join(OUTPUT_DIR, "pipeline_summary.txt")
write_detailed_summary(run_summaries, summary_txt)
print("\nAll TWAS directories processed (Version 3 - logp_expo). Done.")


##############################################################################
# PROCESSING TWAS DIR: /content/lipids/ldl   (label='ldl')
# TWAS integration = VERSION 3 (mild exponential scaling on -log10(p))
##############################################################################
Building TWAS lookup for: /content/lipids/ldl
  Pre-loaded TWAS data for 18,057 genes from 6/6 file(s)
  Sample genes in TWAS lookup: ['PCSK9', 'SMARCA4', 'CELSR2', 'ANKDD1B', 'CETP', 'FEN1', 'ATP13A1', 'FADS1', 'SYPL2', 'YIPF2']
    HTR2C present? False
    HRH1 present? True
    DRD2 present? True
    CHRM1 present? True
    CHRM3 present? True
    HTR2A present? True
Loading Ki database ...
  Ki rows: 98,764
Loading drug list ...
  Drugs: 70

Matching drugs and building drug x receptor table ...


100%|██████████| 70/70 [00:02<00:00, 23.46it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.160, twas_p=0.239); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/ldl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/ldl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/ldl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential scalin

100%|██████████| 70/70 [00:04<00:00, 14.61it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.420, twas_p=0.207); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/hdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/hdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/hdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential scalin

...


[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.393, twas_p=0.171); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/logtg/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/logtg/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/logtg/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential 

100%|██████████| 70/70 [00:04<00:00, 16.37it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.089, twas_p=0.327); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/nonhdl/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/nonhdl/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/nonhdl/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponenti

100%|██████████| 70/70 [00:03<00:00, 23.21it/s]



[TWAS] 9 receptor gene(s) had no TWAS data (e.g. ADRB3, DRD3, DRD5, HRH3, HTR1A, HTR1D, HTR1E, HTR2C, ...)

[DDD filter] kept 61/70 drug(s) with a valid DDD (9 dropped).

Imputation step ...
  [ki-impute] strategy='mean': no missing Ki values to fill.
  [twas-impute] strategy='mean': filled 210 gene row(s) (twas_z=0.367, twas_p=0.21); imputed rows kept NON-significant.
  [drop-zero] no rows with affinity_dose_score == 0.

Matched 58 drugs to ligands; 1285 drug x receptor pairs built.
  imputed Ki rows: 0  |  imputed TWAS rows: 210

Writing outputs ...
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/tc/n05a_ki_twas_full_results.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/tc/drug_summary.csv
  wrote /content/pipeline_output/n05a_ki_twas_log_v3_logp/tc/n05a_ki_twas_full_results.xlsx

Calculating Metabolic Risk Score (Version 3: logp_expo) ...
  [note] TWAS coverage: 681/681 (100%) metabolic-receptor rows have a TWAS twas_p.
  [TWAS-logp] exponential scaling on

# Downstream

In [10]:
# ======================================================================
#  DOWNSTREAM METHOD-COMPARISON  (4 TWAS-integration methods)
#  Original (v0) vs Linear (v1) vs Expo_z (v2) vs Logp (v3)
#  Uses ONLY the pipeline output files. RUN AS A SINGLE COLAB CELL.
#  Writes a VERY DETAILED summary .txt of all important / significant results.
# ======================================================================
import os
import datetime
import itertools
from pathlib import Path
import numpy as np
import pandas as pd

import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
try:
    import seaborn as sns
    _HAS_SNS = True
except Exception:
    _HAS_SNS = False

try:
    from scipy.stats import friedmanchisquare, wilcoxon
    _HAS_SCIPY = True
except Exception:
    _HAS_SCIPY = False

# ======================================================================
# CONFIG  --  edit only if your paths differ
# ======================================================================
BASE = Path("/content/pipeline_output")

METHODS = {                                    # display name -> output dir
    "Original": BASE / "n05a_ki_twas_log",
    "Linear":   BASE / "n05a_ki_twas_log_v1_linear",
    "Expo_z":   BASE / "n05a_ki_twas_log_v2_expo",
    "Logp":     BASE / "n05a_ki_twas_log_v3_logp",
}

# Per-method per-receptor TWAS scaling (MUST mirror the pipeline maths)
#   Original: twas_scale = 1
#   Linear  : twas_scale = 1 + 0.24*|z|
#   Expo_z  : twas_scale = exp(0.08*|z|)
#   Logp    : twas_scale = exp(0.045 * -log10(p)),  -log10(p) capped at 25
METHOD_SCALE = {
    "Original": ("none",   None),
    "Linear":   ("z_lin",  0.24),
    "Expo_z":   ("z_expo", 0.08),
    "Logp":     ("logp",   0.045),
}
LOGP_CAP = 25.0

TRAITS = ["ldl", "hdl", "logtg", "nonhdl", "tc"]
REFERENCE_DRUG = "chlorpromazine"
LEADER_DRUG    = "clozapine"

TOP_N_LIST = [5, 10]
SENSITIVE_RANK_SHIFT = 3           # flag drugs whose rank moves >= this across methods

OUT_DIR = BASE / "method_comparison_v4"
FIG_DIR = OUT_DIR / "figures"
OUT_DIR.mkdir(parents=True, exist_ok=True)
FIG_DIR.mkdir(parents=True, exist_ok=True)

METABOLIC_WEIGHTS = {
    "HRH1":1.00,"HTR2C":0.92,"CHRM3":0.85,"HTR2A":0.55,"ADRA1A":0.48,
    "ADRA1B":0.48,"HTR6":0.42,"ADRA2A":0.32,"ADRA2B":0.30,"ADRA2C":0.30,
    "CHRM1":0.28,"CHRM4":0.22,"CHRM5":0.20,"HTR7":0.25,"DRD3":0.18,
    "DRD2":0.12,"DRD4":0.10,"HTR1A":0.08,"ADRB1":0.08,"ADRB2":0.07,
    "SIGMAR1":0.05,"SLC6A4":0.06,"SLC6A2":0.05,"SLC6A3":0.04,
}
METAB_GENES = set(METABOLIC_WEIGHTS.keys())

# ======================================================================
# helpers
# ======================================================================
def _read_csv(p):
    for enc in ["utf-8", "utf-8-sig", "latin-1", "cp1252"]:
        try:
            return pd.read_csv(p, encoding=enc, low_memory=False)
        except UnicodeDecodeError:
            continue
    return pd.read_csv(p, encoding="latin-1", encoding_errors="replace",
                       low_memory=False)

def _fmt(v, nd=2):
    try:
        if v is None or (isinstance(v, float) and np.isnan(v)):
            return "NA"
        return f"{v:.{nd}f}"
    except Exception:
        return str(v)

LOG = []
def say(m=""):
    print(m)
    LOG.append(m)

BAR = "=" * 80
SUB = "-" * 80

# ======================================================================
# 0.  Discover available methods / traits
# ======================================================================
say(BAR)
say("DOWNSTREAM METHOD COMPARISON — Original vs Linear vs Expo_z vs Logp")
say(BAR)
say(f"Generated : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
say(f"Base dir  : {BASE}")
say("")

avail = {}    # method -> set(traits with risk file)
for m, d in METHODS.items():
    ts = set()
    for t in TRAITS:
        if (d / t / "metabolic_risk_score_per_drug.csv").is_file():
            ts.add(t)
    if ts:
        avail[m] = ts
        say(f"[ok]   {m:<9}: {len(ts)} trait(s) -> {', '.join(sorted(ts))}")
    else:
        say(f"[warn] {m:<9}: NO risk files found under {d} (skipped)")
say("")

if len(avail) < 2:
    raise FileNotFoundError(
        "Need at least 2 methods with output to compare. "
        "Check METHODS paths / that the pipelines have been run."
    )

METHOD_NAMES  = [m for m in METHODS if m in avail]           # keep display order
COMMON_TRAITS = sorted(set.intersection(*avail.values()))    # traits present in ALL methods
ALL_TRAITS    = sorted(set.union(*avail.values()))
say(f"Methods compared : {', '.join(METHOD_NAMES)}")
say(f"Traits in ALL    : {', '.join(COMMON_TRAITS) if COMMON_TRAITS else '(none)'}")
say(f"Traits in ANY    : {', '.join(ALL_TRAITS)}")
say("")

# ======================================================================
# 1.  Load all risk scores -> long -> wide
# ======================================================================
say(BAR); say("1) LOADING RISK SCORES"); say(BAR)
records = []
per = {}                       # (method,trait) -> risk df
for m in METHOD_NAMES:
    for t in sorted(avail[m]):
        fp = METHODS[m] / t / "metabolic_risk_score_per_drug.csv"
        df = _read_csv(fp)
        per[(m, t)] = df
        keep = df[["drug", "metabolic_risk_score", "risk_rank"]].copy()
        keep["method"] = m
        keep["trait"] = t
        records.append(keep)

long = pd.concat(records, ignore_index=True)
long["drug_l"] = long["drug"].astype(str).str.lower()

score_wide = long.pivot_table(index=["drug", "trait"], columns="method",
                              values="metabolic_risk_score").reset_index()
rank_wide  = long.pivot_table(index=["drug", "trait"], columns="method",
                              values="risk_rank").reset_index()
score_wide.to_csv(OUT_DIR / "risk_score_wide.csv", index=False)
rank_wide.to_csv(OUT_DIR / "risk_rank_wide.csv", index=False)
say(f"[ok] loaded {len(long)} rows across {long['method'].nunique()} method(s) "
    f"x {long['trait'].nunique()} trait(s)")
say(f"[ok] wrote risk_score_wide.csv & risk_rank_wide.csv")
say("")

# ======================================================================
# 2.  Between-method correlation of risk scores (per trait)
# ======================================================================
say(BAR); say("2) BETWEEN-METHOD RISK-SCORE CORRELATION (per trait)"); say(BAR)
corr_rows = []
for t in ALL_TRAITS:
    sub = long[long["trait"] == t]
    piv = sub.pivot_table(index="drug", columns="method",
                          values="metabolic_risk_score").dropna()
    mset = [m for m in METHOD_NAMES if m in piv.columns]
    if len(mset) < 2 or len(piv) < 3:
        say(f"[{t.upper()}] insufficient overlap (drugs={len(piv)}, methods={len(mset)}).")
        continue
    sp = piv[mset].corr(method="spearman")
    pe = piv[mset].corr(method="pearson")
    say(f"[{t.upper()}] n_drugs={len(piv)}")
    for a, b in itertools.combinations(mset, 2):
        say(f"     {a:<9} vs {b:<9}  rho={_fmt(sp.loc[a,b],3)}  "
            f"pearson r={_fmt(pe.loc[a,b],3)}")
        corr_rows.append({"trait": t, "method_a": a, "method_b": b,
                          "spearman": sp.loc[a, b], "pearson": pe.loc[a, b],
                          "n_drugs": len(piv)})
    say("")
corr_df = pd.DataFrame(corr_rows)
mp = pd.DataFrame()
if not corr_df.empty:
    corr_df.to_csv(OUT_DIR / "between_method_correlation.csv", index=False)
    mp = (corr_df.groupby(["method_a", "method_b"])["spearman"]
          .mean().reset_index().sort_values("spearman", ascending=False))
    say("Mean Spearman rho by method pair (averaged over traits):")
    for r in mp.itertuples():
        say(f"     {r.method_a:<9} vs {r.method_b:<9}  mean_rho={_fmt(r.spearman,3)}")
say("")

# ======================================================================
# 3.  Top-N overlap (Jaccard) between methods
# ======================================================================
say(BAR); say("3) TOP-N OVERLAP (Jaccard) BETWEEN METHODS"); say(BAR)
overlap_rows = []
def _topset(m, t, n):
    df = per.get((m, t))
    if df is None:
        return set()
    return set(df.sort_values("metabolic_risk_score", ascending=False)
               .head(n)["drug"].astype(str).str.lower())

for n in TOP_N_LIST:
    say(f"--- Top {n} ---")
    for t in ALL_TRAITS:
        present = [m for m in METHOD_NAMES if (m, t) in per]
        if len(present) < 2:
            continue
        sets = {m: _topset(m, t, n) for m in present}
        line = f"[{t.upper()}] "
        for a, b in itertools.combinations(present, 2):
            inter = len(sets[a] & sets[b]); uni = len(sets[a] | sets[b])
            jac = inter / uni if uni else np.nan
            line += f"{a[:3]}n{b[:3]}={inter}/{n}(J={_fmt(jac,2)})  "
            overlap_rows.append({"trait": t, "top_n": n, "method_a": a,
                                 "method_b": b, "intersect": inter, "jaccard": jac})
        allc = set.intersection(*sets.values()) if sets else set()
        line += f"| all={len(allc)}"
        say(line)
    say("")
if overlap_rows:
    pd.DataFrame(overlap_rows).to_csv(OUT_DIR / "topn_overlap.csv", index=False)

# ======================================================================
# 4.  Score magnitude & leader separation per method/trait
# ======================================================================
say(BAR); say("4) SCORE MAGNITUDE & LEADER SEPARATION"); say(BAR)
mag_rows = []
for m in METHOD_NAMES:
    for t in sorted(avail[m]):
        s = per[(m, t)]["metabolic_risk_score"].dropna()
        if s.empty:
            continue
        top = s.max()
        p90 = np.percentile(s, 90); p50 = np.percentile(s, 50)
        n_above = int((s > 100).sum())
        sep = top / p50 if p50 else np.nan
        mag_rows.append({"method": m, "trait": t, "n_drugs": len(s),
                         "max": top, "p90": p90, "median": p50,
                         "mean": float(s.mean()), "std": float(s.std()),
                         "n_above_100": n_above, "top_over_median": sep})
mag = pd.DataFrame(mag_rows)
mag.to_csv(OUT_DIR / "score_magnitude.csv", index=False)
say(f"{'method':<9}{'trait':<8}{'n':>4}{'max':>9}{'p90':>8}{'median':>8}"
    f"{'std':>8}{'>100':>6}{'top/med':>9}")
say(SUB)
for r in mag.itertuples():
    say(f"{r.method:<9}{r.trait:<8}{int(r.n_drugs):>4}{_fmt(r.max,1):>9}"
        f"{_fmt(r.p90,1):>8}{_fmt(r.median,1):>8}{_fmt(r.std,1):>8}"
        f"{int(r.n_above_100):>6}{_fmt(r.top_over_median,2):>9}")
say("")
sep_by_method = (mag.groupby("method")["top_over_median"]
                 .mean().sort_values(ascending=False))
say("Mean leader separation (top/median) by method — higher = more amplification:")
for m, v in sep_by_method.items():
    say(f"     {m:<9} {_fmt(v,3)}")
say("")

# ======================================================================
# 5.  Rank-shift / sensitivity analysis (across methods, per trait)
# ======================================================================
say(BAR); say(f"5) RANK SENSITIVITY TO METHOD (shift >= {SENSITIVE_RANK_SHIFT})"); say(BAR)
rank_piv = long.pivot_table(index=["drug", "trait"], columns="method",
                            values="risk_rank")
present_methods = [m for m in METHOD_NAMES if m in rank_piv.columns]
rp = rank_piv[present_methods].dropna()
sens = pd.DataFrame()
stab = pd.Series(dtype=float)
if len(rp) and len(present_methods) >= 2:
    rp = rp.copy()
    rp["rank_shift"] = rp[present_methods].max(axis=1) - rp[present_methods].min(axis=1)
    rp = rp.reset_index()
    rp.to_csv(OUT_DIR / "rank_shift_all.csv", index=False)
    sens = (rp[rp["rank_shift"] >= SENSITIVE_RANK_SHIFT]
            .sort_values("rank_shift", ascending=False))
    say(f"Drug x trait combos with rank shift >= {SENSITIVE_RANK_SHIFT}: {len(sens)}")
    say(f"{'drug':<24}{'trait':<8}" +
        "".join(f"{m[:7]:>8}" for m in present_methods) + f"{'shift':>7}")
    say(SUB)
    for r in sens.head(20).itertuples(index=False):
        d = getattr(r, "drug"); t = getattr(r, "trait")
        ranks = "".join(f"{int(getattr(r, m)):>8}" for m in present_methods)
        say(f"{str(d)[:23]:<24}{str(t):<8}{ranks}{int(r.rank_shift):>7}")
    say("")
    stab = (rp.groupby("drug")["rank_shift"].max().sort_values().head(10))
    say("Most method-STABLE drugs (max rank shift across traits):")
    for d, v in stab.items():
        say(f"     {d:<24} max_shift={int(v)}")
    say("")
else:
    say("Insufficient data for rank-sensitivity analysis.")

# ======================================================================
# 6.  Leader gap (clozapine / olanzapine vs reference) per method/trait
# ======================================================================
say(BAR); say(f"6) LEADER GAP  ({LEADER_DRUG} vs {REFERENCE_DRUG}=100)"); say(BAR)
gap_rows = []
for m in METHOD_NAMES:
    for t in sorted(avail[m]):
        df = per[(m, t)]
        dl = df.assign(_l=df["drug"].astype(str).str.lower())
        cz  = dl.loc[dl["_l"] == LEADER_DRUG, "metabolic_risk_score"]
        ola = dl.loc[dl["_l"] == "olanzapine", "metabolic_risk_score"]
        gap_rows.append({
            "method": m, "trait": t,
            "clozapine":  float(cz.iloc[0]) if len(cz) else np.nan,
            "olanzapine": float(ola.iloc[0]) if len(ola) else np.nan,
        })
gap = pd.DataFrame(gap_rows)
gap.to_csv(OUT_DIR / "leader_gap.csv", index=False)
gap_piv = gap.pivot_table(index="trait", columns="method", values="clozapine")
say(f"{LEADER_DRUG.capitalize()} score by method x trait (reference=100):")
say(gap_piv.round(1).to_string())
say("")
if "Original" in gap_piv.columns:
    for m in [x for x in METHOD_NAMES if x != "Original" and x in gap_piv.columns]:
        diff = (gap_piv[m] - gap_piv["Original"]).dropna()
        if len(diff):
            say(f"{LEADER_DRUG.capitalize()} lift {m} - Original: "
                f"mean={_fmt(diff.mean(),1)}, max={_fmt(diff.max(),1)} "
                f"(trait={diff.idxmax().upper()})")
say("")

# ======================================================================
# 7.  Receptor-contribution comparison (RECOMPUTED per method's maths)
# ======================================================================
say(BAR); say("7) RECEPTOR-CONTRIBUTION DRIVERS (recomputed per method)"); say(BAR)

def receptor_totals(method, trait):
    lp = METHODS[method] / trait / "n05a_ki_twas_full_results.csv"
    if not lp.is_file():
        return None
    df = _read_csv(lp)
    if not {"gene", "affinity_dose_score"}.issubset(df.columns):
        return None
    d = df[df["gene"].isin(METAB_GENES) & df["affinity_dose_score"].notna()].copy()
    if d.empty:
        return None
    d["affinity"] = np.log1p(d["affinity_dose_score"].clip(lower=0))
    d["weight"]   = d["gene"].map(METABOLIC_WEIGHTS).fillna(0.0)
    kind, param = METHOD_SCALE[method]
    if kind == "none":
        scale = 1.0
    elif kind == "z_lin":
        z = d.get("twas_z", pd.Series(np.nan, index=d.index)).abs()
        avail_z = z[z.notna()]
        z = z.fillna(float(avail_z.mean()) if len(avail_z) else 0.0)
        scale = 1.0 + param * z
    elif kind == "z_expo":
        z = d.get("twas_z", pd.Series(np.nan, index=d.index)).abs()
        avail_z = z[z.notna()]
        z = z.fillna(float(avail_z.mean()) if len(avail_z) else 0.0)
        scale = np.exp(param * z)
    elif kind == "logp":
        p = d.get("twas_p", pd.Series(np.nan, index=d.index))
        avail_p = p[p.notna()].clip(lower=1e-300)
        mean_neglogp = float((-np.log10(avail_p)).mean()) if len(avail_p) else 0.0
        p_safe = p.clip(lower=1e-300)
        neglogp = (-np.log10(p_safe)).clip(upper=LOGP_CAP)
        neglogp = neglogp.fillna(mean_neglogp)
        scale = np.exp(param * neglogp)
    else:
        scale = 1.0
    d["wc"] = d["affinity"] * d["weight"] * scale
    return d.groupby("gene")["wc"].sum().sort_values(ascending=False)

recep_consensus = {}    # method -> Series(gene -> mean rank over traits)
recep_store = {}        # (method,trait) -> Series
for m in METHOD_NAMES:
    per_trait_rank = {}
    for t in sorted(avail[m]):
        s = receptor_totals(m, t)
        if s is None:
            continue
        recep_store[(m, t)] = s
        per_trait_rank[t] = s.rank(ascending=False)
    if per_trait_rank:
        rf = pd.DataFrame(per_trait_rank)
        recep_consensus[m] = rf.mean(axis=1).sort_values()

common_top3 = set()
if recep_consensus:
    say("Top-8 receptor drivers per method (mean rank across traits, lower=stronger):")
    for m, s in recep_consensus.items():
        say(f"  [{m:<9}] " + ", ".join(f"{g}({_fmt(v,1)})" for g, v in s.head(8).items()))
    say("")
    top3 = {m: set(s.head(3).index) for m, s in recep_consensus.items()}
    common_top3 = set.intersection(*top3.values()) if top3 else set()
    say(f"Receptors in the top-3 of EVERY method: "
        f"{', '.join(sorted(common_top3)) if common_top3 else '(none)'}")
    common_for_recep = [t for t in ALL_TRAITS
                        if all((m, t) in recep_store for m in METHOD_NAMES)]
    if common_for_recep:
        ct = "hdl" if "hdl" in common_for_recep else common_for_recep[0]
        tbl = pd.DataFrame({m: recep_store[(m, ct)] for m in METHOD_NAMES
                            if (m, ct) in recep_store})
        tbl = tbl.sort_values(METHOD_NAMES[0], ascending=False)
        tbl.to_csv(OUT_DIR / f"receptor_contrib_{ct}.csv")
        say(f"\nReceptor contributions on trait '{ct.upper()}' (summed, top 10):")
        say(tbl.head(10).round(2).to_string())
else:
    say("No long-format results available to recompute receptor contributions.")
say("")

# ======================================================================
# 8.  Per-drug amplification factors between methods
# ======================================================================
say(BAR); say("8) PER-DRUG AMPLIFICATION FACTORS (vs Original)"); say(BAR)
amp = pd.DataFrame()
amp_cols = []
if "Original" in METHOD_NAMES:
    wide = long.pivot_table(index=["drug", "trait"], columns="method",
                            values="metabolic_risk_score").reset_index()
    for m in [x for x in METHOD_NAMES if x != "Original"]:
        if m in wide.columns and "Original" in wide.columns:
            col = f"{m}_vs_Original"
            wide[col] = wide[m] / wide["Original"]
            amp_cols.append(col)
    if amp_cols:
        amp = wide
        amp.to_csv(OUT_DIR / "per_drug_amplification_factors.csv", index=False)
        say("Amplification factor summary (across all drug x trait rows):")
        desc = amp[amp_cols].describe().round(3)
        say(desc.to_string())
        say("")
        # top amplified drugs by the strongest-scaling available method
        pref_method = next((c for c in ["Logp_vs_Original", "Expo_z_vs_Original",
                                         "Linear_vs_Original"] if c in amp_cols), None)
        if pref_method:
            base_m = pref_method.replace("_vs_Original", "")
            say(f"Top 10 drug x trait rows most amplified by {base_m} vs Original:")
            cols = ["drug", "trait", "Original", base_m, pref_method]
            cols = [c for c in cols if c in amp.columns]
            top_amp = amp.dropna(subset=[pref_method]).nlargest(10, pref_method)[cols]
            say(top_amp.round(2).to_string(index=False))
    say("")
else:
    say("Original method not present; amplification vs Original skipped.")
say("")

# ======================================================================
# 9.  Cross-method CONSENSUS high-risk drugs
# ======================================================================
say(BAR); say("9) CONSENSUS HIGH-RISK DRUGS (mean across methods & traits)"); say(BAR)
consensus = (long.groupby("drug")
             .agg(mean_risk=("metabolic_risk_score", "mean"),
                  median_risk=("metabolic_risk_score", "median"),
                  mean_rank=("risk_rank", "mean"),
                  n_obs=("metabolic_risk_score", "size"))
             .reset_index()
             .sort_values("mean_rank"))
consensus.to_csv(OUT_DIR / "consensus_ranking.csv", index=False)
say(f"{'#':>3}  {'drug':<26}{'mean_risk':>10}{'mean_rank':>10}{'n_obs':>7}")
say(SUB)
for i, r in enumerate(consensus.head(20).itertuples(), 1):
    say(f"{i:>3}  {str(r.drug)[:25]:<26}{_fmt(r.mean_risk,1):>10}"
        f"{_fmt(r.mean_rank,2):>10}{int(r.n_obs):>7}")
say("")

# ======================================================================
# 10.  Statistical test across methods (Friedman) per trait
# ======================================================================
say(BAR); say("10) STATISTICAL TEST ACROSS METHODS (Friedman, per trait)"); say(BAR)
fried_rows = []
if _HAS_SCIPY:
    for t in ALL_TRAITS:
        sub = long[long["trait"] == t]
        piv = sub.pivot_table(index="drug", columns="method",
                              values="metabolic_risk_score").dropna()
        mset = [m for m in METHOD_NAMES if m in piv.columns]
        if len(mset) < 3 or len(piv) < 3:
            continue
        try:
            stat, pval = friedmanchisquare(*[piv[m].values for m in mset])
            fried_rows.append({"trait": t, "n_drugs": len(piv),
                               "n_methods": len(mset), "chi2": stat, "p": pval})
            say(f"[{t.upper()}] Friedman chi2={_fmt(stat,3)}  p={_fmt(pval,4)}  "
                f"(n_drugs={len(piv)}, methods={len(mset)})")
        except Exception as e:
            say(f"[{t.upper()}] Friedman failed: {e}")
    say("")
    if fried_rows:
        pd.DataFrame(fried_rows).to_csv(OUT_DIR / "friedman_per_trait.csv", index=False)
else:
    say("scipy not available -> Friedman/Wilcoxon tests skipped.")
say("")

# ======================================================================
# FIGURES
# ======================================================================
try:  # leader separation bar
    plt.figure(figsize=(9, 5))
    piv = mag.pivot_table(index="trait", columns="method", values="top_over_median")
    piv = piv.reindex(columns=[m for m in METHOD_NAMES if m in piv.columns])
    piv.plot(kind="bar", ax=plt.gca())
    plt.ylabel("Top / median score")
    plt.title("Leader separation by method & trait")
    plt.xticks(rotation=0); plt.tight_layout()
    plt.savefig(FIG_DIR / "separation_by_method.png", dpi=150); plt.close()
except Exception as e:
    say(f"[warn] separation figure failed: {e}")

try:  # correlation heatmaps per common trait
    for t in COMMON_TRAITS:
        piv = (long[long["trait"] == t]
               .pivot_table(index="drug", columns="method",
                            values="metabolic_risk_score").dropna())
        ms = [m for m in METHOD_NAMES if m in piv.columns]
        if len(ms) < 2 or len(piv) < 3:
            continue
        c = piv[ms].corr(method="spearman")
        plt.figure(figsize=(5, 4))
        if _HAS_SNS:
            sns.heatmap(c, annot=True, cmap="coolwarm", vmin=0.8, vmax=1.0, fmt=".3f")
        else:
            plt.imshow(c, cmap="coolwarm", vmin=0.8, vmax=1.0)
            plt.xticks(range(len(ms)), ms, rotation=45); plt.yticks(range(len(ms)), ms)
            for i in range(len(ms)):
                for j in range(len(ms)):
                    plt.text(j, i, f"{c.iloc[i,j]:.3f}", ha="center", va="center")
            plt.colorbar()
        plt.title(f"Method Spearman rho — {t.upper()}"); plt.tight_layout()
        plt.savefig(FIG_DIR / f"method_corr_{t}.png", dpi=150); plt.close()
except Exception as e:
    say(f"[warn] correlation figures failed: {e}")

try:  # consensus top-15 heatmap
    top_drugs = consensus.head(15)["drug"].tolist()
    heat = (long[long["drug"].isin(top_drugs)]
            .pivot_table(index="drug", columns="method",
                         values="metabolic_risk_score", aggfunc="mean")
            .reindex(index=top_drugs))
    heat = heat.reindex(columns=[m for m in METHOD_NAMES if m in heat.columns])
    plt.figure(figsize=(8, 7))
    if _HAS_SNS:
        sns.heatmap(heat, annot=True, fmt=".1f", cmap="RdYlGn_r", center=100)
    else:
        plt.imshow(heat, cmap="RdYlGn_r", aspect="auto")
        plt.colorbar()
        plt.yticks(range(len(heat.index)), heat.index)
        plt.xticks(range(len(heat.columns)), heat.columns, rotation=45)
    plt.title("Mean metabolic risk score — consensus top 15 drugs")
    plt.tight_layout()
    plt.savefig(FIG_DIR / "consensus_top15_heatmap.png", dpi=150); plt.close()
except Exception as e:
    say(f"[warn] consensus heatmap failed: {e}")

# ======================================================================
# WRITE VERY DETAILED SUMMARY TXT
# ======================================================================
S = []
S.append(BAR)
S.append("METHOD COMPARISON — VERY DETAILED SUMMARY (4 TWAS-integration methods)")
S.append("Original (no scaling) | Linear (1+0.24|z|) | Expo_z (exp0.08|z|) | "
         "Logp (exp0.045*-log10 p)")
S.append(BAR)
S.append(f"Generated        : {datetime.datetime.now():%Y-%m-%d %H:%M:%S}")
S.append(f"Base dir         : {BASE}")
S.append(f"Output dir       : {OUT_DIR}")
S.append(f"Methods compared : {', '.join(METHOD_NAMES)}")
S.append(f"Traits (all)     : {', '.join(ALL_TRAITS)}")
S.append(f"Traits (common)  : {', '.join(COMMON_TRAITS) if COMMON_TRAITS else '(none)'}")
S.append(f"Reference drug   : {REFERENCE_DRUG} (=100)")
S.append(f"Leader drug      : {LEADER_DRUG}")
S.append("")

# --- data availability ---
S.append(BAR); S.append("0) DATA AVAILABILITY"); S.append(BAR)
for m in METHOD_NAMES:
    S.append(f"   {m:<9}: traits = {', '.join(sorted(avail[m]))}")
S.append(f"   Total risk rows loaded: {len(long)}")
S.append("")

# --- A) correlation ---
S.append(BAR); S.append("A) BETWEEN-METHOD AGREEMENT (Spearman rho of risk scores)"); S.append(BAR)
if not corr_df.empty:
    for t in ALL_TRAITS:
        sub = corr_df[corr_df["trait"] == t]
        if sub.empty:
            continue
        S.append(f"[{t.upper()}] (n_drugs={int(sub['n_drugs'].iloc[0])})")
        for r in sub.itertuples():
            S.append(f"   {r.method_a:<9} vs {r.method_b:<9} rho={_fmt(r.spearman,3)} "
                     f"pearson={_fmt(r.pearson,3)}")
    S.append("")
    S.append("Mean rho by pair (over traits):")
    for r in mp.itertuples():
        S.append(f"   {r.method_a:<9} vs {r.method_b:<9} mean_rho={_fmt(r.spearman,3)}")
    hi = mp.iloc[0]; lo = mp.iloc[-1]
    S.append("")
    S.append(f"INTERPRETATION: most similar pair = {hi.method_a}/{hi.method_b} "
             f"(rho={_fmt(hi.spearman,3)}); most divergent = {lo.method_a}/{lo.method_b} "
             f"(rho={_fmt(lo.spearman,3)}).")
else:
    S.append("Insufficient overlap to compute correlations.")
S.append("")

# --- B) top-N overlap ---
S.append(BAR); S.append("B) TOP-N OVERLAP (Jaccard) BETWEEN METHODS"); S.append(BAR)
if overlap_rows:
    ov = pd.DataFrame(overlap_rows)
    for n in TOP_N_LIST:
        S.append(f"--- Top {n} (mean Jaccard over traits) ---")
        sub = ov[ov["top_n"] == n]
        mj = (sub.groupby(["method_a", "method_b"])["jaccard"].mean()
              .reset_index().sort_values("jaccard", ascending=False))
        for r in mj.itertuples():
            S.append(f"   {r.method_a:<9} vs {r.method_b:<9} meanJ={_fmt(r.jaccard,3)}")
        S.append("")
else:
    S.append("No overlap data.")
S.append("")

# --- C) magnitude / separation ---
S.append(BAR); S.append("C) SCORE MAGNITUDE & LEADER SEPARATION"); S.append(BAR)
S.append(f"{'method':<9}{'trait':<8}{'n':>4}{'max':>9}{'median':>8}{'std':>8}"
         f"{'>100':>6}{'top/med':>9}")
for r in mag.itertuples():
    S.append(f"{r.method:<9}{r.trait:<8}{int(r.n_drugs):>4}{_fmt(r.max,1):>9}"
             f"{_fmt(r.median,1):>8}{_fmt(r.std,1):>8}{int(r.n_above_100):>6}"
             f"{_fmt(r.top_over_median,2):>9}")
S.append("")
S.append("Mean leader separation (top/median) by method:")
for m, v in sep_by_method.items():
    S.append(f"   {m:<9} {_fmt(v,3)}")
best_sep = sep_by_method.idxmax()
S.append(f"INTERPRETATION: '{best_sep}' produces the strongest leader separation "
         f"(mean top/median={_fmt(sep_by_method.max(),3)}).")
S.append("")

# --- D) rank sensitivity ---
S.append(BAR); S.append(f"D) RANK SENSITIVITY TO METHOD (shift >= {SENSITIVE_RANK_SHIFT})"); S.append(BAR)
if len(sens):
    S.append(f"Total drug x trait combos with rank shift >= {SENSITIVE_RANK_SHIFT}: {len(sens)}")
    S.append(f"{'drug':<24}{'trait':<8}" +
             "".join(f"{m[:7]:>8}" for m in present_methods) + f"{'shift':>7}")
    for r in sens.head(25).itertuples(index=False):
        d = getattr(r, "drug"); t = getattr(r, "trait")
        ranks = "".join(f"{int(getattr(r, m)):>8}" for m in present_methods)
        S.append(f"{str(d)[:23]:<24}{str(t):<8}{ranks}{int(r.rank_shift):>7}")
    S.append("")
    S.append("Most method-STABLE drugs (max shift across traits):")
    for d, v in stab.items():
        S.append(f"   {d:<24} max_shift={int(v)}")
else:
    S.append("No drugs exceeded the sensitivity threshold (methods largely agree on ranks).")
S.append("")

# --- E) leader gap ---
S.append(BAR); S.append(f"E) {LEADER_DRUG.upper()} SCORE BY METHOD (ref {REFERENCE_DRUG}=100)"); S.append(BAR)
S.append(gap_piv.round(1).to_string())
S.append("")
if "Original" in gap_piv.columns:
    for m in [x for x in METHOD_NAMES if x != "Original" and x in gap_piv.columns]:
        diff = (gap_piv[m] - gap_piv["Original"]).dropna()
        if len(diff):
            S.append(f"{LEADER_DRUG.capitalize()} lift {m} - Original: "
                     f"mean={_fmt(diff.mean(),1)}, max={_fmt(diff.max(),1)} "
                     f"(trait={diff.idxmax().upper()})")
S.append("")

# --- F) receptor drivers ---
S.append(BAR); S.append("F) RECEPTOR-CONTRIBUTION DRIVERS (recomputed per method)"); S.append(BAR)
if recep_consensus:
    for m, s in recep_consensus.items():
        S.append(f"[{m:<9}] top-8 (mean rank): " +
                 ", ".join(f"{g}({_fmt(v,1)})" for g, v in s.head(8).items()))
    S.append("")
    S.append(f"Receptors in the top-3 of EVERY method: "
             f"{', '.join(sorted(common_top3)) if common_top3 else '(none)'}")
    S.append("INTERPRETATION: overlap here means the choice of TWAS scaling changes "
             "score magnitude but NOT the dominant metabolic receptors.")
else:
    S.append("No long-format results available for receptor recomputation.")
S.append("")

# --- G) amplification ---
S.append(BAR); S.append("G) PER-DRUG AMPLIFICATION FACTORS (vs Original)"); S.append(BAR)
if not amp.empty and amp_cols:
    S.append("Summary statistics (mean amplification factor across drug x trait):")
    for c in amp_cols:
        col = amp[c].dropna()
        if len(col):
            S.append(f"   {c:<22} mean={_fmt(col.mean(),3)} "
                     f"median={_fmt(col.median(),3)} max={_fmt(col.max(),3)}")
    S.append("")
    pref_method = next((c for c in ["Logp_vs_Original", "Expo_z_vs_Original",
                                    "Linear_vs_Original"] if c in amp_cols), None)
    if pref_method:
        base_m = pref_method.replace("_vs_Original", "")
        S.append(f"Top 10 rows most amplified by {base_m} vs Original:")
        cols = [c for c in ["drug", "trait", "Original", base_m, pref_method]
                if c in amp.columns]
        top_amp = amp.dropna(subset=[pref_method]).nlargest(10, pref_method)[cols]
        S.append(top_amp.round(2).to_string(index=False))
else:
    S.append("Amplification not computed (Original method absent).")
S.append("")

# --- H) consensus drugs ---
S.append(BAR); S.append("H) CONSENSUS HIGH-RISK DRUGS (across all methods & traits)"); S.append(BAR)
S.append(f"{'#':>3}  {'drug':<26}{'mean_risk':>10}{'mean_rank':>10}{'n_obs':>7}")
for i, r in enumerate(consensus.head(20).itertuples(), 1):
    S.append(f"{i:>3}  {str(r.drug)[:25]:<26}{_fmt(r.mean_risk,1):>10}"
             f"{_fmt(r.mean_rank,2):>10}{int(r.n_obs):>7}")
S.append("")

# --- I) statistical tests ---
S.append(BAR); S.append("I) STATISTICAL TESTS ACROSS METHODS (Friedman, per trait)"); S.append(BAR)
if _HAS_SCIPY and fried_rows:
    for r in fried_rows:
        sig = "SIGNIFICANT" if r["p"] < 0.05 else "n.s."
        S.append(f"[{r['trait'].upper()}] chi2={_fmt(r['chi2'],3)} p={_fmt(r['p'],4)} "
                 f"({sig}; n_drugs={r['n_drugs']}, methods={r['n_methods']})")
    S.append("")
    S.append("NOTE: a significant Friedman test means at least one method yields "
             "systematically different score magnitudes; it does NOT by itself imply "
             "the drug RANKING changed (see section A for ranking agreement).")
elif not _HAS_SCIPY:
    S.append("scipy not available -> tests skipped.")
else:
    S.append("Not enough overlap for Friedman tests.")
S.append("")

# --- J) overall verdict ---
S.append(BAR); S.append("J) OVERALL VERDICT"); S.append(BAR)
if not corr_df.empty:
    overall_rho = corr_df["spearman"].mean()
    S.append(f"Mean cross-method Spearman rho (all pairs, all traits): {_fmt(overall_rho,3)}")
    if overall_rho >= 0.95:
        S.append(" -> Methods are HIGHLY concordant on drug ranking; the TWAS scaling "
                 "choice mainly rescales magnitude rather than reordering drugs.")
    elif overall_rho >= 0.85:
        S.append(" -> Methods are broadly concordant with some meaningful reordering "
                 "for mid-rank drugs.")
    else:
        S.append(" -> Methods diverge substantially; method choice materially affects "
                 "conclusions and should be reported explicitly.")
S.append(f"Strongest leader amplification : {best_sep}")
if consensus is not None and len(consensus):
    S.append(f"Most robust high-risk drug     : {consensus.iloc[0]['drug']} "
             f"(mean_rank={_fmt(consensus.iloc[0]['mean_rank'],2)})")
S.append("")

# --- K) output files ---
S.append(BAR); S.append("K) OUTPUT FILES"); S.append(BAR)
for p in sorted(x for x in OUT_DIR.rglob("*") if x.is_file()):
    S.append(f"   {p}")
S.append("")
S.append(BAR); S.append("END OF DETAILED METHOD-COMPARISON SUMMARY"); S.append(BAR)

summary_path = OUT_DIR / "method_comparison_detailed_summary.txt"
with open(summary_path, "w", encoding="utf-8") as fh:
    fh.write("\n".join(S))
with open(OUT_DIR / "method_comparison_run_log.txt", "w", encoding="utf-8") as fh:
    fh.write("\n".join(LOG))

print("\n" + BAR)
print(f"DETAILED SUMMARY -> {summary_path}")
print(f"Run log          -> {OUT_DIR / 'method_comparison_run_log.txt'}")
print(f"Figures          -> {FIG_DIR}")
print(BAR)

# Optional: copy back to Drive (uncomment if mounted)
# !cp -r "/content/pipeline_output/method_comparison_v4" "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS/"

DOWNSTREAM METHOD COMPARISON — Original vs Linear vs Expo_z vs Logp
Generated : 2026-07-15 04:19:05
Base dir  : /content/pipeline_output

[ok]   Original : 5 trait(s) -> hdl, ldl, logtg, nonhdl, tc
[ok]   Linear   : 5 trait(s) -> hdl, ldl, logtg, nonhdl, tc
[ok]   Expo_z   : 5 trait(s) -> hdl, ldl, logtg, nonhdl, tc
[ok]   Logp     : 5 trait(s) -> hdl, ldl, logtg, nonhdl, tc

Methods compared : Original, Linear, Expo_z, Logp
Traits in ALL    : hdl, ldl, logtg, nonhdl, tc
Traits in ANY    : hdl, ldl, logtg, nonhdl, tc

1) LOADING RISK SCORES
[ok] loaded 1040 rows across 4 method(s) x 5 trait(s)
[ok] wrote risk_score_wide.csv & risk_rank_wide.csv

2) BETWEEN-METHOD RISK-SCORE CORRELATION (per trait)
[HDL] n_drugs=52
     Original  vs Linear     rho=0.992  pearson r=0.998
     Original  vs Expo_z     rho=0.990  pearson r=0.997
     Original  vs Logp       rho=0.991  pearson r=0.998
     Linear    vs Expo_z     rho=0.998  pearson r=0.999
     Linear    vs Logp       rho=0.997  pearson r=0.

In [11]:
!ls /content/pipeline_output/method_comparison_v4

between_method_correlation.csv		per_drug_amplification_factors.csv
consensus_ranking.csv			rank_shift_all.csv
figures					receptor_contrib_hdl.csv
friedman_per_trait.csv			risk_rank_wide.csv
leader_gap.csv				risk_score_wide.csv
method_comparison_detailed_summary.txt	score_magnitude.csv
method_comparison_run_log.txt		topn_overlap.csv


In [ ]:
!cp "/content/pipeline_output" "/content/drive/MyDrive/Dr Uccello/00_Studies/080_GHS_CVS" -r

# The End